# Menos odio a la vista (v5): detección multitarea de vulgaridad, insulto y odio grupal con contexto, en español
**Proyecto de Aplicación 2: Hugging Face** · Técnicas de Aprendizaje de Máquina · Pontificia Universidad Javeriana

*Evolución del filtro previo para cuidar a quien lee —comunidades, chicas, niños y adolescentes— sin apagar la conversación: de una sola etiqueta de odio a tres señales independientes que consideran el contexto.*

**Integrantes:** Esteban Salazar, Samuel Pico, David Vargas y Tatiana Vivas.

**Entregable:** este cuaderno (informe técnico reproducible, versión 5) + aplicación de prueba en Streamlit (y exportación lista para Gradio / Hugging Face Spaces). Ejecutar las celdas **en orden**. Semilla de referencia: **42**. Modelo evaluado: **RoBERTuito multitarea con contexto** (cabeza de odio grupal: F1 ≈ 0,706 en DEV_EVAL; sin diferencia detectable frente al modelo original A3, F1 ≈ 0,689). El **TEST permanece sin evaluar** (ver 5.7 y 10).

## Mapa del cuaderno
| Sección | Contenido |
|---|---|
| 0 – 2 | Pitch, definición del problema (Fase 0), corpus |
| **3** | Entorno, carga de datos y **EDA**: 3.1–3.3 entorno · 3.4 módulos (partidos en celdas documentadas) · 3.5–3.7 carga y validación · 3.8–3.13 EDA sobre la partición por artículo · **3.14–3.16 EDA de las particiones oficiales y reconstrucción de la partición por artículo** |
| **4** | Preparación de etiquetas y representación: 4.1–4.4 multitarea (etiquetas con máscara, aumentación, tokenización con contexto) · **4.5 preparación para los modelos de referencia A1/A2** |
| 5 | Modelado y evaluación (arquitectura, entrenamiento, calibración, métricas, robustez, contexto, comparación con A3, errores, HateCheck-es, cómputo) |
| 6 | Aplicación web (inferencia, exportación, Streamlit, verificaciones) |
| 7 – 10 | Limitaciones, equipo, registro de revisión con IA y conclusiones |

# 0. Elevator pitch

El modelo original (A3) responde una sola pregunta: ¿este comentario contiene odio contra un grupo? Quien modera una comunidad encuentra situaciones más variadas: una grosería dicha entre amigos (“usted sí es mucho hp, parce”), un insulto personal en medio de una discusión, un estereotipo contra un grupo regional, o una cita que denuncia el odio sin ejercerlo. Reducir todo a una etiqueta obliga a elegir entre ocultar de más o dejar pasar de menos.

**Pregunta que guía esta versión:** ¿puede un único modelo en español separar tres señales distintas —**vulgaridad**, **insulto o ataque personal** y **odio grupal**—, resistir la escritura alterada con la que se evade la detección (`HI J U E P U T A`, `H1JUEPUTA`, `hijuep*ta`) y aprovechar el contexto cuando existe, sin dejar de funcionar cuando no existe?

**Por qué importa (para quien podría financiar esto):**
1. **Cuidado de las personas:** distinguir una grosería amistosa de una agresión reduce tanto la exposición a ataques como el castigo injusto a quien conversa en confianza. El proyecto no afirma que un modelo “cure” la salud mental; afirma que ofrecer más información a quien decide (tres probabilidades en lugar de una) permite intervenir con más cuidado.
2. **Comunidades más humanas:** las tres señales se pueden combinar con reglas distintas por comunidad (por ejemplo, tolerar vulgaridad pero no insulto) sin reentrenar.
3. **Moderación al alcance de equipos pequeños:** el sistema entrega probabilidades calibradas y umbrales explícitos; la decisión final sigue en manos de una persona.

**Cómo se entrega:** (1) un modelo multitarea que se guarda y se carga como un solo objeto (`Predictor`), (2) una app de prueba en Streamlit para escribir un comentario, añadir contexto opcional y ver las tres señales, (3) un paquete de exportación para Gradio / Hugging Face Spaces.

**Justificación del enfoque y síntesis de resultados:** la cabeza de odio grupal rinde igual que A3 en las mismas filas (ΔF1 = +0,016; IC 95 % por artículo [−0,012; 0,045], que incluye 0), de modo que el valor de la versión no está en “superar a A3”, sino en añadir dos señales que A3 no tiene (vulgaridad e insulto), en tolerar mejor las variantes ortográficas y en aceptar contexto. Los resultados de vulgaridad e insulto se midieron con muy pocas filas de validación (37 y 48) y con una batería pequeña; por eso **no se declaran mejoras** y quedan pendientes la evaluación en TEST y datos anotados en español colombiano (sección 7).

# 1. Fase 0: definición del problema

## 1.1 Problema (técnico y social)
**Social:** una misma palabra puede ser cariño, desahogo o agresión según quién la dice y en qué conversación. Un filtro que solo mira la palabra castiga a quien conversa en confianza y deja pasar el ataque que no usa groserías. El proyecto sigue sin prometer “seguridad total”: busca **interrumpir a tiempo** los mensajes que agreden y **no castigar** los que solo son vulgares, informales o que denuncian el odio.

**Técnico:** clasificación **multietiqueta** de un comentario en español (texto + contexto opcional) con tres salidas **no excluyentes**: `vulgaridad`, `insulto` (ataque personal) y `odio_grupal`. Un comentario puede activar varias a la vez (por ejemplo, vulgaridad e insulto). Cada salida se calibra y tiene su propio umbral.

## 1.2 Entrada y salida
- **Unidad de análisis:** un comentario y, si existe, su contexto (título de la noticia o mensaje previo).
- **Entrada:** `comentario` (obligatorio) y `contexto` (opcional). Se codifican como un par de segmentos, con truncación solo del contexto, de modo que el comentario conserva prioridad.
- **Salida:** tres probabilidades calibradas en [0, 1] y tres señales (sí/no) según umbrales elegidos en DEV_EVAL. Si una cabeza no tiene datos suficientes, el modelo la declara **“sin entrenar”** y devuelve `NaN` en lugar de inventar un resultado.

## 1.3 Datos y referencias de evaluación
- **Corpus de odio grupal:** `piuba-bigdata/contextualized_hate_speech` (tuits argentinos en respuesta a noticias). Su etiqueta `HATEFUL` define **odio contra grupos**; solo supervisa la cabeza `odio_grupal`.
- **Corpus de vulgaridad e insulto:** `SINAI/OffendES` (comentarios de influencers en español; etiquetas `NO`, `NOE`, `OFP`, `OFG`; dataset protegido, con licencia declarada de forma contradictoria en la ficha). Es la única fuente disponible con anotaciones compatibles con esas dos cabezas.
- **Batería contrastiva (v4):** 77 casos de desarrollo y 76 de prueba, disjuntos por familia, con ofuscaciones, estereotipos frente a contra-discurso, y usos amistosos frente a ofensivos de expresiones colombianas. Sus etiquetas son **propuestas de los autores** (`validado=False`): sirven para ver patrones, no como verdad de referencia.
- **Limitación a declarar:** `piuba` es español rioplatense y tema pandémico; `OffendES` viene de comentarios de redes de influencers; la batería contiene expresiones colombianas. Ningún conjunto de entrenamiento es colombiano.

## 1.4 Propuesta de valor
**Para quién:** administradores de blogs, foros y medios en español que quieren cuidar a su audiencia sin un equipo de moderación permanente y que necesitan **distinguir** vulgaridad, insulto y odio grupal para aplicar reglas diferentes.

**Cómo se usa:** filtro previo con humano en el loop. La app muestra las tres probabilidades, el umbral de cada una y cuánto cambian al añadir contexto. **No sustituye** el juicio humano.

## 1.5 Justificación del enfoque
- **¿Por qué no una lista de groserías?** En la v4 una regla léxica alcanzó F1 ≈ 0,20 en DEV: el odio viaja sin groserías y hay groserías sin agresión. Aquí el normalizador de ofuscación **no decide nada**: solo genera una vista alternativa del texto, y la decisión la toma el modelo.
- **¿Por qué no inventar etiquetas para las cabezas nuevas?** Porque `HATEFUL = 0` en piuba **no** significa “sin insulto” (los insultos personales aparecen como no-hateful). Por eso la pérdida usa una **máscara**: cada fuente solo aporta las etiquetas que su definición respalda.
- **¿Por qué una sola arquitectura?** Un encoder compartido (RoBERTuito, el mismo de A3) con tres cabezas independientes permite comparar con A3 en la tarea común y evita mantener tres modelos.
- **Hipótesis a comprobar** (estado al final de la sección 5):
  - **H1 (etiquetas compatibles):** cada cabeza se entrena solo con fuentes que la respaldan.
  - **H2 (sin pérdida en la tarea original):** la cabeza `odio_grupal` no empeora frente a A3 en las mismas filas.
  - **H3 (ortografía alterada):** el modelo conserva la mayor parte de su desempeño con texto ofuscado.
  - **H4 (contexto):** el contexto modifica la salida en el sentido esperado (por ejemplo, un `hp` amistoso frente a uno ofensivo).

## 1.6 Revisión crítica con IA de la Fase 0
**Prompt usado (revisión de etiquetas):** *¿Puede usarse `HATEFUL = 0` de piuba como “no hay insulto ni vulgaridad” para entrenar las otras dos cabezas? ¿Qué fuente respalda cada etiqueta y qué pasa donde no hay etiquetas?*

**Prompt usado (revisión de evaluación):** *¿Qué evidencia haría falta para afirmar que la arquitectura nueva mejora a A3, y cómo evitar que la validación y el TEST se contaminen?*

**Crítica recibida (síntesis):** no usar `HATEFUL = 0` como negativo de las otras cabezas; enmascarar etiquetas ausentes; no comparar métricas de tareas distintas; separar la partición que elige el checkpoint de la que calibra y compara; declarar cuando una cabeza se evalúa con pocas filas.

**Decisión del equipo (verificada después con datos):**
1. Pérdida con máscara y auditoría de etiquetas por fuente (sección 4).
2. DEV partido por artículo en `DEV_CKPT` (elige checkpoint) y `DEV_EVAL` (calibra y compara).
3. Comparación con A3 solo en la tarea común (`odio_grupal`), con IC 95 % por artículo.
4. TEST reservado hasta congelar el modelo. Detalle completo en la **sección 9**.

# 2. Descripción y justificación de los corpus

**Por qué estos corpus (y no otros):** cada cabeza necesita etiquetas cuya definición coincida con lo que debe aprender. Ninguno de los dos conjuntos disponibles etiqueta las tres categorías a la vez, por eso se combinan con **etiquetas parciales** (matriz `Y` y máscara `M`) en lugar de forzar una fuente a describir lo que no anota.

**piuba-bigdata/contextualized_hate_speech**

| Campo | Función en v5 |
|---|---|
| `text` | Comentario (entrada principal) |
| `title` | Contexto opcional (título de la noticia) y grupo para particionar por artículo |
| `context_tweet`, `body` | Contexto alterno; `body` identifica el artículo al reconstruir la partición |
| `HATEFUL` | Etiqueta de `odio_grupal` (1 = odio, 0 = no odio **grupal**) |
| `CALLS`, `WOMEN`, `LGBTI`, `RACISM`, `CLASS`, `POLITICS`, `DISABLED`, `APPEARANCE`, `CRIMINAL` | No se usan como etiquetas de las nuevas cabezas |

**SINAI/OffendES**

| Campo | Función en v5 |
|---|---|
| `comment` | Comentario (sin contexto disponible) |
| `label` | `NO` (no ofensivo), `NOE` (no ofensivo con lenguaje vulgar), `OFP` (ofensivo contra una persona), `OFG` (ofensivo contra un grupo) |
| `influencer`, `influencer_gender`, `media`, `comment_id` | No se usan |

Particiones oficiales: train 16.710 · validation 100 · test 13.606 filas.

**Batería contrastiva v4 (`R.construir_bateria`)**: familias de casos con la misma estructura (por ejemplo, la misma frase con y sin ofuscación, o el mismo término en uso amistoso y ofensivo). Solo se usa para **evaluar**.

_Fuente: esquemas consultados en el Hub (nov 2026). Se verifican en las celdas de la sección 3._

# 3. Entorno, carga de datos y análisis exploratorio (EDA)
Esta sección cubre el **entorno**, la **carga**, la **validación** y el **análisis exploratorio** de los conjuntos, con doble mirada: **técnica** (formato, vacíos, duplicados, balance, longitudes, fugas) y **humana** (¿a quién apunta el odio? ¿hay agresiones sin groserías y groserías sin agresión? ¿cómo se escriben las variantes alteradas?). Las conclusiones cerradas están al final de la sección (tras 3.16); las celdas imprimen la evidencia.

**Datos de referencia de la corrida completa** (Colab con GPU, 2026-11-08):

| Partición | Filas | % Hateful | Artículos únicos (`title`) |
|---|---:|---:|---:|
| train | 36.587 | 15,4 % | 789 |
| dev | 8.971 | 14,0 % | 198 |
| test | 11.311 | 16,2 % | 248 |

Partición por **artículo** (`GroupShuffleSplit`, seed 42, la misma lógica de la v4): ningún artículo cruza particiones.

## 3.1 Dependencias
Colab ya incluye torch, transformers y datasets, por lo que **no se reinstalan** (evita choques de versión). `pysentimiento` se instala sin dependencias porque solo se necesita su `preprocess_tweet`, el preprocesamiento que espera RoBERTuito.

In [ ]:
# 1.1 Dependencias. Colab ya trae torch/transformers/datasets: NO se reinstalan (evita choques de versión).
# pysentimiento se instala sin dependencias: solo se necesita su `preprocess_tweet` (el que espera RoBERTuito).
import subprocess, sys
def pip(*a): return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
for m in ["transformers", "datasets", "scikit-learn", "scipy", "pandas", "pyarrow", "tokenizers"]:
    try: __import__("sklearn" if m == "scikit-learn" else m)
    except ImportError: pip(m)
r = pip("--no-deps", "pysentimiento", "emoji")
import torch, transformers, numpy as np, pandas as pd
try: from IPython.display import display
except ImportError: display = print
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")

## 3.2 Acceso al Hub
`OffendES` es un dataset protegido: requiere aceptar sus condiciones en el Hub y un token en los *Secrets* de Colab (`HF_TOKEN`). Sin token, el cuaderno continúa, pero `vulgaridad` e `insulto` quedan **sin datos y sin entrenar** (no se simulan).

In [ ]:
# 3.2 Autenticación en el Hub (opcional pero necesaria para OffendES)
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login, whoami
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Autenticado como:", whoami()["name"])
else:
    print("⚠ Sin HF_TOKEN: OffendES (protegido) no podrá descargarse y vulgaridad/insulto quedarán sin entrenar.")

## 3.3 Configuración global
Es lo único que normalmente se modifica. `MODO='prueba'` usa una submuestra y pocos pasos (verifica que todo corre); `MODO='completo'` entrena con todos los datos. `SIN_RED=True` solo verifica el código sin Hub (datos sintéticos; sus cifras no significan nada).

In [ ]:
# 1.2 Configuración global (lo único que normalmente se toca)
import time, json, random, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)

MODO = "completo"             # "prueba" (submuestra, pocos pasos) | "completo" (entrenamiento completo, ver parámetros abajo)
SIN_RED = False             # True = verificación técnica SIN Hub: encoder aleatorio diminuto + datos SINTÉTICOS (métricas sin valor)
EVALUAR_TEST_FINAL = False  # True SOLO al final, con todo congelado; el TEST nunca se usa para elegir nada
USAR_CONTEXTO = True        # el modelo acepta contexto; si False se entrena/infiere solo con el comentario
COL_CTX = "title"           # contexto de piuba: "title" (título de la noticia) o "context_tweet"
ENCODER = "pysentimiento/robertuito-base-uncased"
A3_HUB = "tatisvivass/menos-odio-a3-robertuito"      # modelo original; solo inferencia, jamás se modifica
SEED = 42

# Fuentes opcionales de etiquetas para vulgaridad / insulto (ver sección 3; ninguna se simula)
USAR_OFFENDES = True        # True tras aceptar las condiciones del dataset en el Hub (SINAI/OffendES) y fijar las columnas
COL_TEXTO_OFF, COL_ETIQ_OFF = "comment", "label"   # ← fijar tras inspeccionar el dataset (no se asumen nombres)
OFG_COMO_GRUPO = False      # True SOLO si se verificó que OFG de OffendES equivale al HATEFUL de piuba
RUTA_ANOTADOS = None        # CSV anotado por humanos (columna `particion` ∈ train/dev/test + y_vulgaridad/y_insulto/y_odio_grupal o y_final)

PARAMS = {
    "prueba":   dict(n_train=3000, n_dev=1200, epochs=1, max_pasos=60, bs=16, acum=1, lr=3e-5, max_len=96, paciencia=2, eval_cada=0.5),
    "completo": dict(n_train=None, n_dev=None, epochs=4, max_pasos=None, bs=16, acum=2, lr=2e-5, max_len=128, paciencia=4, eval_cada=0.5),
}[MODO]
if SIN_RED:
    PARAMS = dict(n_train=1500, n_dev=600, epochs=2, max_pasos=None, bs=16, acum=1, lr=5e-4, max_len=48, paciencia=3, eval_cada=0.5)

RUN_ID = time.strftime("%Y%m%d-%H%M%S") + f"_{MODO}" + ("_SINRED" if SIN_RED else "")
OUT = Path("artifacts_v5"); RUN = OUT / RUN_ID; RUN.mkdir(parents=True, exist_ok=True)
CHECKS = {}          # lista de verificaciones reales de ESTA ejecución (se imprime al final)
print("run:", RUN, "|", MODO, "| SIN_RED =", SIN_RED, "| params:", PARAMS)
if SIN_RED: print("⚠ SIN_RED: datos sintéticos y encoder aleatorio. Solo comprueba que el código corre; ninguna cifra tiene valor.")

## 3.4 Módulos
Se escriben a disco para que el cuaderno sea autosuficiente (no depende de variables ni archivos de la v4):
- `robustez_es.py` — **reutilizado de la v4 sin cambios**: normalizador de ofuscación, perturbaciones, batería contrastiva, métricas, calibración y *bootstrap*.
- `multitarea_es.py` — **nuevo**: etiquetas con máscara, *dataset* con aumentación en línea, tokenización de pares, arquitectura, pérdida enmascarada, bucle de entrenamiento, `Predictor` (guardar / cargar / inferir), evaluación y exportación. Incluye la tokenización de pares ya corregida (acorta el comentario con margen creciente para que `only_second` siempre sea viable).

Cada módulo se presenta **dividido en celdas cortas** (`%writefile` para la primera y `%writefile -a` para las siguientes), cada una precedida por su explicación (subsecciones 3.4.1 a 3.4.32).

> **Cómo leer esta sección.** Cada módulo se escribe a disco en varias celdas: la primera usa `%%writefile` (crea el archivo) y las siguientes `%%writefile -a` (agregan al final). **Ejecuta las celdas en orden y de una sola vez**; si vuelves a ejecutar solo una celda intermedia, el archivo quedará con código duplicado (reinicia desde la primera celda del módulo).

### 3.4.1 `robustez_es.py` (1/17): encabezado, principios de diseño e importaciones

Docstring del módulo y dependencias. Principio de diseño: **sin `torch` ni `transformers`** (solo `numpy`, `pandas`, `scikit-learn`), de modo que este módulo se puede probar fuera de la GPU. El normalizador **no clasifica** ni expande abreviaturas ambiguas (`hp` se queda como `hp`): solo genera una *vista alternativa* del texto y decide el modelo.

In [ ]:
%%writefile robustez_es.py
# -*- coding: utf-8 -*-
"""
robustez_es.py — utilidades para detección de lenguaje ofensivo en español:
robustez a ofuscación ortográfica, batería contrastiva, métricas, calibración e incertidumbre.

Diseño:
  * Sin dependencia de torch/transformers (solo numpy, pandas, scikit-learn), para poder
    probarse con pytest fuera de la GPU.
  * El normalizador NO clasifica ni expande abreviaturas ambiguas: ("hp", "HP") se dejan intactas.
    Produce una *vista alternativa* del texto; el modelo decide con el contexto.
  * Las etiquetas de la batería son anotaciones PROPUESTAS por los autores del código
    (columna `validado=False`) y deben validarse con el protocolo humano antes de usarse
    como verdad de referencia.
"""
from __future__ import annotations

import re
import unicodedata
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

LETRA = r"[^\W\d_]"

### 3.4.2 `robustez_es.py` (2/17): normalización: constantes y expresiones regulares

Tablas y patrones que usa el normalizador de ofuscación: caracteres de ancho cero, homoglifos cirílicos/griegos, mapa *leet* (`1→i`, `3→e`, `@→a`…), patrones para letras espaciadas (`H I J U E P U T A`), máscaras con `*` y alargamientos (`hijueputaaaa`).

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 1. NORMALIZACIÓN CONTROLADA (vista alternativa, nunca reemplazo obligatorio)
# --------------------------------------------------------------------------------------
_ZW = dict.fromkeys(map(ord, "​‌‍⁠﻿­"), None)
_HOMOGLIFOS = str.maketrans({
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ѕ": "s", "ј": "j",
    "ο": "o", "α": "a", "ε": "e", "ι": "i", "τ": "t",
})
_LEET = {"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s", "!": "i", "|": "i"}
_RE_LEET = re.compile(rf"(?<={LETRA})[013457@$!|](?={LETRA})")
_STOP2 = {"de", "la", "el", "en", "un", "se", "me", "te", "lo", "no", "si", "es", "ya", "su", "mi", "tu",
          "al", "le", "ni", "yo", "tú", "él", "ha", "he", "ve", "va", "da", "di", "ir", "ay", "oh", "eh",
          "y", "a", "o", "u", "e"}
_RE_ESPACIADO = re.compile(
    rf"(?<!{LETRA})(?:({LETRA}{{2}}) )?((?:{LETRA}[ .\-_]){{3,}}{LETRA})(?!{LETRA})")
_RE_MASCARA = re.compile(r"[^\W\d_][\w*#•·]*[*#•·][\w*#•·]*")
_RE_ALARGA = re.compile(rf"({LETRA})\1{{2,}}")
_RE_EMAIL = re.compile(r"\S+@\S+\.\S+")

# Formas canónicas usadas SOLO para reconstruir palabras enmascaradas ("hijuep*ta").
# No se usa para clasificar ni para expandir abreviaturas ("hp" no se expande).

### 3.4.3 `robustez_es.py` (3/17): normalización: léxico de recuperación y funciones auxiliares

`LEXICO_RECUPERACION` se usa **solo** para reconstruir palabras enmascaradas de forma única (`hijuep*ta → hijueputa`); no sirve para clasificar. Las auxiliares revierten *leet* (protegiendo números, `COVID19`, `5G`, `@usuarios`), colapsan el espaciado (resolviendo el caso ambiguo `T A y lo sabes`) y resuelven máscaras con `*`.

In [ ]:
%%writefile -a robustez_es.py
LEXICO_RECUPERACION = [
    "hijueputa", "hijueputas", "hijoputa", "gonorrea", "malparido", "malparida", "puta", "putas", "puto",
    "putos", "prostituta", "prostitutas", "marica", "maricas", "mierda", "pendejo", "pendeja", "idiota",
    "imbecil", "imbécil", "cabron", "cabrón", "verga", "carajo", "huevon", "huevón", "zorra", "gonorreas",
]


def _leet_token(tok: str) -> str:
    if tok[:1] in "@#" or tok.lower().startswith("http") or _RE_EMAIL.fullmatch(tok):
        return tok
    if len(re.findall(LETRA, tok)) < 2:      # protege números, "COVID19", "5G", etc.
        return tok

    def sub(m):
        c = m.group(0)
        if c in "!|":                         # "Hola!Gracias": minúscula+!+Mayúscula = límite de oración
            prev, nxt = m.string[m.start() - 1], m.string[m.end()]
            if prev.islower() and nxt.isupper():
                return c
        return _LEET[c]

    for _ in range(3):
        nuevo = _RE_LEET.sub(sub, tok)
        if nuevo == tok:
            break
        tok = nuevo
    return tok


def _colapsar_espaciado(t: str) -> str:
    def rep(m):
        pre, run = m.group(1), re.sub(r"[ .\-_]", "", m.group(2))
        pegar_pre = bool(pre) and pre.lower() not in _STOP2     # "HI" se pega; "un", "de"... no
        unido = pre + run if pegar_pre else run
        cola = ""
        # Caso ambiguo: "T A y lo sabes" → la "y" final es una palabra real, no parte de la ofuscada.
        # Se separa solo si quitarla deja una forma conocida y conservarla no.
        if unido[-1].lower() in "yaoeu" and unido.lower() not in LEXICO_RECUPERACION \
                and unido[:-1].lower() in LEXICO_RECUPERACION:
            unido, cola = unido[:-1], " " + unido[-1]
        return ((pre + " ") if (pre and not pegar_pre) else "") + unido + cola
    return _RE_ESPACIADO.sub(rep, t)


def _resolver_mascara(core: str) -> Optional[str]:
    low = core.lower()
    cands = set()
    patron = "".join("[a-zñáéíóúü]" if c in "*#•·" else re.escape(c) for c in low)
    cands |= {f for f in LEXICO_RECUPERACION if re.fullmatch(patron, f)}
    m = re.fullmatch(r"([^*#•·]{4,})[*#•·]{2,}", low)
    if m:
        cands |= {f for f in LEXICO_RECUPERACION if f.startswith(m.group(1)) and len(f) > len(m.group(1))}
    if not cands:
        return None
    corto = min(cands, key=len)
    return corto if all(c.startswith(corto) for c in cands) else None


def _mascaras(t: str) -> str:
    def rep(m):
        r = _resolver_mascara(m.group(0))
        return r if r else m.group(0)
    return _RE_MASCARA.sub(rep, t)

### 3.4.4 `robustez_es.py` (4/17): normalización: función pública y vistas del texto

`normalizar_ofuscacion` encadena todas las reversiones y `vistas_texto` devuelve `[original, normalizada]` si la normalización cambia algo. Esa segunda vista es la que usa la *test-time augmentation* (TTA) del `Predictor`.

In [ ]:
%%writefile -a robustez_es.py
def normalizar_ofuscacion(texto: str, minusculas: bool = True) -> str:
    """Revierte ofuscaciones comunes sin alterar el significado léxico de abreviaturas ambiguas.

    Revierte: separación de letras, sustitución interna por números/símbolos, homoglifos,
    caracteres de ancho cero, alargamiento (3+ repeticiones), máscaras con * resolubles de forma única.
    No revierte: eliminación de caracteres. No expande "hp"/"HP".
    """
    t = unicodedata.normalize("NFKC", texto).translate(_ZW).translate(_HOMOGLIFOS)
    t = _colapsar_espaciado(t)
    t = " ".join(_leet_token(tok) for tok in t.split(" "))
    t = _mascaras(t)
    t = _RE_ALARGA.sub(r"\1", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t.lower() if minusculas else t


def vistas_texto(texto: str) -> List[str]:
    """[original, normalizada] si la normalización cambia algo; si no, solo [original]."""
    n = normalizar_ofuscacion(texto, minusculas=False)
    return [texto] if n == texto else [texto, n]

### 3.4.5 `robustez_es.py` (5/17): perturbaciones sintéticas (aumentación): familias básicas

Operaciones que **ofuscan** una palabra (espaciado, *leet*, máscara `*`, alargamiento, mayúsculas alternas, borrado de una letra, homoglifos). Se usan en entrenamiento (aumentación en línea) y para medir robustez ortográfica.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 2. PERTURBACIONES SINTÉTICAS (aumentación; misma tasa en ambas clases)
# --------------------------------------------------------------------------------------
_A_LEET = {"a": ["4", "@"], "e": ["3"], "i": ["1", "!"], "o": ["0"], "s": ["5", "$"], "t": ["7"]}
_A_HOMO = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "y": "у", "x": "х"}


def p_espaciado(w, rng):
    return rng.choice([" ", ".", "-", "_"]).join(w)


def p_leet(w, rng):
    idx = [i for i in range(1, len(w) - 1) if w[i].lower() in _A_LEET]
    if not idx:
        return w
    i = int(rng.choice(idx))
    return w[:i] + str(rng.choice(_A_LEET[w[i].lower()])) + w[i + 1:]


def p_mascara(w, rng):
    idx = [i for i in range(1, len(w) - 1) if w[i].lower() in "aeiou"] or list(range(1, len(w) - 1))
    if not idx:
        return w
    i = int(rng.choice(idx))
    return w[:i] + "*" + w[i + 1:]


def p_alargar(w, rng):
    idx = [i for i, c in enumerate(w) if c.lower() in "aeiou"]
    if not idx:
        return w
    i = idx[-1] if rng.random() < 0.6 else int(rng.choice(idx))
    return w[:i + 1] + w[i] * int(rng.integers(2, 6)) + w[i + 1:]


def p_caso(w, rng):
    modo = int(rng.integers(0, 3))
    if modo == 0:
        return w.upper()
    if modo == 1:
        return "".join(c.upper() if i % 2 else c.lower() for i, c in enumerate(w))
    return "".join(c.upper() if rng.random() < 0.5 else c.lower() for c in w)


def p_borrar(w, rng):
    if len(w) < 4:
        return w
    i = int(rng.integers(1, len(w) - 1))
    return w[:i] + w[i + 1:]


def p_homoglifo(w, rng):
    idx = [i for i, c in enumerate(w) if c.lower() in _A_HOMO]
    if not idx:
        return w
    i = int(rng.choice(idx))
    return w[:i] + _A_HOMO[w[i].lower()] + w[i + 1:]


PERTURBACIONES = {"espaciado": p_espaciado, "leet": p_leet, "mascara": p_mascara, "alargar": p_alargar,
                  "caso": p_caso, "borrar": p_borrar, "homoglifo": p_homoglifo}

### 3.4.6 `robustez_es.py` (6/17): perturbaciones añadidas en v3

Sustitución de letras parecidas, errores de teclado QWERTY, eliminación de vocales, la abreviatura `hijueputa → hp` (solo desde la forma canónica, **nunca** al revés) y `p_mezcla`, que encadena 2–3 perturbaciones sobre la misma palabra.

In [ ]:
%%writefile -a robustez_es.py
# --- Perturbaciones añadidas en v3 -----------------------------------------------------
_TECLADO = {"q": "wa", "w": "qes", "e": "wrd", "r": "etf", "t": "ryg", "y": "tuh", "u": "yij", "i": "uok",
            "o": "ipl", "p": "ol", "a": "qsz", "s": "adw", "d": "sfe", "f": "dgr", "g": "fht", "h": "gjy",
            "j": "hku", "k": "jli", "l": "kop", "z": "ax", "x": "zc", "c": "xv", "v": "cb", "b": "vn",
            "n": "bm", "m": "n", "ñ": "nl"}
_SUSTITUTOS = {"a": "ou", "e": "ai", "i": "eyl", "o": "au", "u": "ox", "s": "zc", "c": "sk", "b": "v", "v": "b",
               "t": "d", "d": "t", "g": "j", "j": "g", "k": "c", "x": "s", "p": "b", "m": "n", "n": "m"}
# Abreviaturas usadas SOLO como aumentación de entrenamiento (forma canónica -> abreviatura).
# El normalizador jamás las expande: "hp" sigue siendo ambiguo y lo resuelve el contexto.
_ABREV = {"hijueputa": "hp", "hijueputas": "hps", "hijoputa": "hp", "hijodeputa": "hdp", "hijodeputas": "hdps"}


def p_sustituir(w, rng):
    """Cambia UNA letra interna por otra parecida o frecuente en errores (hijueputa -> hijuxputa/hijuepvta)."""
    idx = [i for i in range(1, len(w) - 1) if w[i].lower() in _SUSTITUTOS]
    pool = idx or list(range(1, len(w) - 1))
    if not pool:
        return w
    i = int(rng.choice(pool))
    c = w[i].lower()
    nuevo = str(rng.choice(list(_SUSTITUTOS.get(c, "x"))))
    return w[:i] + (nuevo.upper() if w[i].isupper() else nuevo) + w[i + 1:]


def p_teclado(w, rng):
    """Reemplaza una letra interna por una vecina del teclado QWERTY."""
    idx = [i for i in range(1, len(w) - 1) if w[i].lower() in _TECLADO]
    if not idx:
        return w
    i = int(rng.choice(idx))
    nuevo = str(rng.choice(list(_TECLADO[w[i].lower()])))
    return w[:i] + (nuevo.upper() if w[i].isupper() else nuevo) + w[i + 1:]


def p_sin_vocales(w, rng):
    """Omite vocales internas (prostituta -> prstt, hijueputa -> hjpt). Conserva primera y última letra."""
    if len(w) < 5:
        return w
    nucleo = "".join(c for c in w[1:-1] if c.lower() not in "aeiouáéíóú")
    return w[0] + nucleo + w[-1] if nucleo != w[1:-1] and len(nucleo) >= 1 else w


def p_abrev(w, rng):
    """hijueputa -> hp, etc. Solo si la palabra es una forma canónica conocida; si no, no hace nada."""
    r = _ABREV.get(w.lower())
    if r is None:
        return w
    return r.upper() if w.isupper() else r


_BASE_MEZCLA = ["espaciado", "leet", "mascara", "alargar", "caso", "borrar", "homoglifo", "sustituir", "teclado", "sin_vocales"]


def p_mezcla(w, rng):
    """Aplica 2 o 3 perturbaciones distintas en cadena a la misma palabra (p. ej. teclado + leet + caso)."""
    k = int(rng.integers(2, 4))
    for nombre in rng.choice(_BASE_MEZCLA, size=k, replace=False):
        w = PERTURBACIONES[str(nombre)](w, rng)
    return w

### 3.4.7 `robustez_es.py` (7/17): registro de perturbaciones, `perturbar_texto` y `aumentar_df`

`perturbar_texto` perturba palabras alfabéticas (≥ 4 letras), omite `@usuarios`, `#hashtags` y URLs, y garantiza al menos una palabra alterada. `aumentar_df` genera copias perturbadas **con la misma tasa en ambas clases**, para que el modelo no aprenda «texto perturbado = odio».

In [ ]:
%%writefile -a robustez_es.py
PERTURBACIONES.update({"sustituir": p_sustituir, "teclado": p_teclado, "sin_vocales": p_sin_vocales,
                       "abrev": p_abrev, "mezcla": p_mezcla})

_RE_PALABRA = re.compile(rf"^(\W*)({LETRA}+)(\W*)$")


def perturbar_texto(texto: str, rng: np.random.Generator, p_palabra: float = 0.35,
                    tipos: Optional[Sequence[str]] = None, min_len: int = 4) -> Tuple[str, List[str]]:
    """Perturba palabras alfabéticas (>= min_len letras); omite @usuarios, #hashtags y URLs.
    Garantiza al menos una palabra perturbada si existe alguna elegible."""
    tipos = list(tipos or PERTURBACIONES)
    toks = texto.split(" ")
    elegibles = []
    for i, tk in enumerate(toks):
        m = _RE_PALABRA.match(tk)
        if m and len(m.group(2)) >= min_len and tk[:1] not in "@#" and not tk.lower().startswith("http"):
            elegibles.append(i)
    if not elegibles:
        return texto, []
    elegidos = [i for i in elegibles if rng.random() < p_palabra] or [int(rng.choice(elegibles))]
    ops = []
    for i in elegidos:
        pre, w, post = _RE_PALABRA.match(toks[i]).groups()
        # "abrev" solo aplica a formas canónicas conocidas (hijueputa -> hp): se prefiere cuando es posible y se
        # excluye en el resto. Si una operación no cambia la palabra, se reintenta con otra (sin copias idénticas).
        abreviable = w.lower() in _ABREV and "abrev" in tipos
        pool = [t for t in tipos if t != "abrev"] or ["leet"]
        nombre, nueva = None, w
        for intento in range(6):
            nombre = "abrev" if (abreviable and (intento == 0 and rng.random() < 0.5)) else str(rng.choice(pool))
            nueva = PERTURBACIONES[nombre](w, rng)
            if nueva != w:
                break
        toks[i] = pre + nueva + post
        ops.append(nombre)
    return " ".join(toks), ops


def aumentar_df(df: pd.DataFrame, frac: float, seed: int = 42, col_texto: str = "text",
                col_y: str = "HATEFUL", p_palabra: float = 0.35, tipos=None) -> pd.DataFrame:
    """Copias perturbadas de una fracción ESTRATIFICADA por clase (misma tasa en cada clase), para que
    el modelo no aprenda 'texto perturbado = odio'. La etiqueta se conserva."""
    rng = np.random.default_rng(seed)
    sub = df.groupby(col_y, group_keys=False).sample(frac=frac, random_state=seed).copy()
    res = [perturbar_texto(t, rng, p_palabra, tipos) for t in sub[col_texto]]
    sub["texto_original"] = sub[col_texto].values
    sub[col_texto] = [r[0] for r in res]
    sub["perturbado"] = 1
    sub["ops"] = [",".join(r[1]) for r in res]
    return sub

### 3.4.8 `robustez_es.py` (8/17): batería contrastiva: etiquetas y generadores de casos

Mapas de categoría → etiqueta de ofensividad / grupo, variantes ofuscadas de *hijueputa* y *prostituta*, y la clase auxiliar `_B` que construye los casos con identificador, familia, fenómeno y nota. Las etiquetas son **propuestas de los autores** (`validado=False`).

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 3. BATERÍA CONTRASTIVA (DEV para elegir transformaciones/umbrales; TEST para el reporte final)
# --------------------------------------------------------------------------------------
_Y_OFENSIVO = {"neutral": 0, "vulgar_sin_agresion": 2, "insulto": 1, "estereotipo": 1, "odio_grupo": 1, "ambiguo": 2}
_Y_GRUPO = {"neutral": 0, "vulgar_sin_agresion": 0, "insulto": 0, "estereotipo": 1, "odio_grupo": 1, "ambiguo": 2}
CATEGORIAS = list(_Y_OFENSIVO)

_OFUSC_HIJUEPUTA = {"espaciado": "HI J U E P U T A", "leet_1": "H1JUEPUTA", "leet_excl": "H!JUEPUTA",
                    "mascara": "hijuep*ta", "puntos": "h.i.j.u.e.p.u.t.a", "alargado": "hijueputaaaa",
                    "caso_alterno": "hIjUePuTa", "homoglifo": "hijuеputa"}
_OFUSC_PROSTITUTA = {"caso": "prostituTa", "leet": "pr0stitu7a", "mascara": "prostitu*a",
                     "espaciado": "p r o s t i t u t a", "alargado": "prostituuuta", "homoglifo": "prоstituta"}


class _B:
    def __init__(self, prefijo):
        self.p, self.rows, self.n = prefijo, [], 0

    def add(self, familia, fenomeno, categoria, texto, nota="", regional="CO", base_id=None, perturbacion=None):
        self.n += 1
        rid = f"{self.p}-{self.n:03d}"
        self.rows.append(dict(id=rid, familia=f"{self.p}-{familia}", fenomeno=fenomeno, categoria=categoria,
                              y_ofensivo=_Y_OFENSIVO[categoria], y_grupo=_Y_GRUPO[categoria], texto=texto,
                              regional=regional, base_id=base_id, perturbacion=perturbacion, nota=nota,
                              validado=False))
        return rid

    def df(self):
        return pd.DataFrame(self.rows)


def _grupos(b, pares, frames):
    for g, a in pares:
        for fen, cat, plantilla in frames:
            b.add(f"GRP-{g}", fen, cat, plantilla.format(g=g, a=a))


def _ofuscar(b, familia, base, token, variantes, cat):
    rid = b.add(familia, "ofuscacion_base", cat, base)
    for nombre, forma in variantes.items():
        b.add(familia, "ofuscacion", cat, base.replace(token, forma), base_id=rid, perturbacion=nombre)


def _benignos_perturbados(b, familia_pref, bases, semillas):
    for k, base in enumerate(bases):
        rid = b.add(f"{familia_pref}{k}", "ofuscacion_base", "neutral", base)
        for s in semillas:
            rng = np.random.default_rng(1000 * k + s)
            t, ops = perturbar_texto(base, rng, p_palabra=0.5, tipos=["leet", "espaciado", "alargar", "caso", "mascara"])
            if t != base:
                b.add(f"{familia_pref}{k}", "ofuscacion_benigna", "neutral", t, base_id=rid, perturbacion=",".join(ops))

### 3.4.9 `robustez_es.py` (9/17): batería contrastiva: `construir_bateria` (DEV y TEST)

Construye 77 casos de desarrollo y 76 de prueba, **disjuntos por familia y por texto** (se verifica con un `assert`): estereotipo frente a contra-discurso, `hp` marca frente a insulto, usos amistosos frente a ofensivos de expresiones colombianas/argentinas/mexicanas, ironía, palabras inventadas, anglicismos y ofuscaciones.

In [ ]:
%%writefile -a robustez_es.py
def construir_bateria() -> Dict[str, pd.DataFrame]:
    """Devuelve {'dev': df, 'test': df}. Conjuntos disjuntos por familia y por texto."""
    # ---------------- DEV ----------------
    d = _B("DEV")
    _grupos(d, [("costeños", "perezosos"), ("paisas", "tacaños"), ("venezolanos", "ladrones")], [
        ("geografico", "neutral", "Estoy escribiendo un blog sobre la cultura de los {g} y su gastronomía."),
        ("estereotipo", "estereotipo", "Todos los {g} son {a}."),
        ("estereotipo", "estereotipo", "Los {g} son unos {a}, no se puede confiar en ellos."),
        ("contradiscurso", "neutral", "No está bien decir que todos los {g} son {a}."),
        ("cita_denuncia", "neutral", "Hoy me dijeron que los {g} son {a}; eso es discriminación y lo denuncio."),
        ("negacion", "neutral", "Es falso que los {g} sean {a}; trabajan como cualquier otra persona."),
    ])
    d.add("CHAMO", "coloquial_regional", "ambiguo", "Ese videojuego es cosa de chamos, yo prefiero algo más serio.",
          "'cosa de chamos' ≈ cosa de muchachos; solo despectivo según el contexto", "VE/CO")
    d.add("CHAMO", "geografico", "neutral", "Mis amigos chamos organizaron una fiesta y nos divertimos mucho.", regional="VE/CO")
    d.add("CHAMO", "estereotipo", "odio_grupo", "Esos chamos que llegaron al barrio son puros delincuentes, que se vayan de aquí.", regional="VE/CO")
    d.add("CHAMO", "contradiscurso", "neutral", "Dejen de decir que todo lo que hacen los chamos es cosa de delincuentes.", regional="VE/CO")
    d.add("HP", "marca_homonimo", "neutral", "Compré un portátil HP y la batería dura poco.")
    d.add("HP", "marca_homonimo", "neutral", "Mi impresora HP ya no imprime en color.")
    d.add("HP", "insulto", "insulto", "Ese tipo es un hp, me quitó el puesto en la fila.")
    d.add("HP", "coloquial_amistoso", "ambiguo", "Parce, usted sí es mucho hp jugando fútbol jajaja")
    d.add("HP", "periodistico", "neutral", "El mandatario llamó 'hp' a varios periodistas, según informó el diario.")
    d.add("HP", "homonimo_politico", "ambiguo", "En su discurso dijo 'HP' para referirse a los honorables parlamentarios.",
          "uso documentado en prensa colombiana (El Universal, 2025-04-26)")
    d.add("PROS", "insulto", "insulto", "Eres una prostituta y todo el barrio lo sabe.")
    d.add("PROS", "insulto", "insulto", "Esa prosti me robó el novio y ni vergüenza tiene.")
    d.add("PROS", "educativo", "neutral", "La ONG estudia la prostitución desde la salud pública y los derechos laborales.")
    d.add("PROS", "contradiscurso", "neutral", "No le digas prostituta a nadie; usar esa palabra como insulto es violencia verbal.")
    d.add("PROS", "neutral", "neutral", "El documental muestra con respeto la vida de una trabajadora sexual.")
    d.add("HIJ", "insulto", "insulto", "Usted es un hijueputa, no vuelva por aquí.")
    d.add("HIJ", "coloquial_exclamacion", "vulgar_sin_agresion", "¡Hijueputa, se me cayó el celular!")
    d.add("HIJ", "coloquial_amistoso", "vulgar_sin_agresion", "Qué hijueputa susto me dio, parce, casi me da algo jajaja")
    d.add("HIJ", "educativo", "neutral", "En Colombia 'hijueputa' se usa tanto para insultar como para expresar sorpresa.")
    d.add("HIJ", "negacion", "neutral", "No creo que seas un hijueputa; solo estabas de mal humor.")
    d.add("HIJ", "negacion", "neutral", "Nunca le diría hijueputa a mi mamá ni a nadie.")
    d.add("AR", "coloquial_amistoso", "vulgar_sin_agresion", "Che, boludo, no te olvides de traer el mate.", regional="AR")
    d.add("AR", "insulto", "insulto", "Sos un pelotudo, no entendés nada.", regional="AR")
    d.add("MARICA", "coloquial_amistoso", "ambiguo", "Marica, no me deje plantada hoy, mire que la fiesta es importante.")
    d.add("GAY", "odio", "odio_grupo", "Los gays son una plaga y deberían desaparecer.", regional="ES-LATAM")
    d.add("GAY", "contradiscurso", "neutral", "Defiendo los derechos de los gays y de todas las personas.", regional="ES-LATAM")
    d.add("IRON", "ironia", "ambiguo", "Claro, los paisas son tan generosos que ni el aire comparten, jajaja")
    d.add("INV", "inventada", "ambiguo", "Ese tetranutra no sabe lo que dice.",
          "sin significado documentado; la búsqueda web solo halló la marca TetraNatura (alimento para peces)")
    d.add("INV", "marca_homonimo", "neutral", "El alimento TetraNatura es bueno para los peces del acuario.")
    d.add("ANG", "prestamo", "neutral", "Ese cringe de video me dio pena ajena, qué fail.")
    d.add("ANG", "prestamo", "neutral", "Mi crush me dejó en visto y ahora estoy en modo drama jajaja")
    _ofuscar(d, "OFU-HIJ", "Eres un hijueputa y todos lo saben.", "hijueputa", _OFUSC_HIJUEPUTA, "insulto")
    _ofuscar(d, "OFU-PROS", "Eres una prostituta y no sirves para nada.", "prostituta", _OFUSC_PROSTITUTA, "insulto")
    _benignos_perturbados(d, "OFU-BEN", [
        "La reunión es mañana en la oficina del centro.",
        "Gracias por la receta, quedó deliciosa y a mi familia le encantó.",
        "El partido empieza a las ocho y vamos todos juntos al estadio."], semillas=[1, 2, 3])

    # ---------------- TEST ----------------
    t = _B("TST")
    _grupos(t, [("cachacos", "creídos"), ("pastusos", "bobos"), ("porteños", "arrogantes")], [
        ("geografico", "neutral", "Voy a visitar a mis amigos {g} este fin de semana; dicen que su comida es deliciosa."),
        ("estereotipo", "estereotipo", "Los {g} son siempre {a}, ni vale la pena hablarles."),
        ("estereotipo", "estereotipo", "Qué se puede esperar de unos {g}, todos son {a}."),
        ("contradiscurso", "neutral", "Me molesta que digan que los {g} son {a}; es un estereotipo injusto."),
        ("cita_denuncia", "neutral", "Escuché a alguien decir que los {g} son {a} y le respondí que no se generaliza."),
        ("negacion", "neutral", "Los {g} no son {a} por naturaleza; nadie lo es por su origen."),
    ])
    t.add("PELAO", "coloquial_regional", "ambiguo", "Eso del karaoke es cosa de pelaos, a mí no me llama la atención.")
    t.add("CHAMO", "estereotipo", "odio_grupo", "Los chamos que cruzan la frontera son todos unos vagos y ladrones, que los devuelvan.", regional="VE/CO")
    t.add("CHAMO", "neutral", "neutral", "Conocí unos chamos muy trabajadores en el taller de mi tío.", regional="VE/CO")
    t.add("HP", "marca_homonimo", "neutral", "El cargador HP se dañó y no consigo repuesto en Bogotá.")
    t.add("HP", "marca_homonimo", "neutral", "Mi HP Pavilion trae 16 GB de RAM y funciona perfecto.")
    t.add("HP", "insulto", "insulto", "Nos tocó ver ese partido horrible, el árbitro es un hp.")
    t.add("HP", "insulto", "insulto", "Esa HP de vecina nunca saluda y siempre está criticando a todo el mundo.")
    t.add("HP", "coloquial_amistoso", "ambiguo", "Qué hp de gol el de anoche, parce, no lo puedo creer.")
    t.add("HP", "metalinguistico", "neutral", "Cuando alguien dice 'hp' en una entrevista, unos lo toman como insulto y otros como chiste.")
    t.add("PROS", "insulto", "insulto", "Eres una prostituta y una fresca.")
    t.add("PROS", "insulto", "insulto", "Esa prosti ni se baña y todavía se cree la reina.")
    t.add("PROS", "contradiscurso", "neutral", "Me parece mal que usen 'prostituta' como insulto contra las mujeres.")
    t.add("PROS", "neutral", "neutral", "La película cuenta la historia de una prostituta que lucha por criar a su hija.")
    t.add("PROS", "negacion", "neutral", "Jamás diría que eres una prostituta; respeto mucho a las mujeres.")
    t.add("HIJ", "insulto", "insulto", "Eres un hijueputa y ojalá te pase lo peor.")
    t.add("HIJ", "coloquial_exclamacion", "vulgar_sin_agresion", "¡Hijueputa, otra vez llueve y sin paraguas!")
    t.add("HIJ", "coloquial_amistoso", "vulgar_sin_agresion", "Ese hijueputa partido nos dejó sin aire, parce, jajaja")
    t.add("HIJ", "educativo", "neutral", "El profesor explicó que 'hijueputa' es una contracción de 'hijo de puta'.")
    t.add("HIJ", "negacion", "neutral", "No puedo creer que alguien le diga hijueputa a un niño.")
    t.add("MX", "coloquial_amistoso", "vulgar_sin_agresion", "No manches, güey, ¡qué buen concierto estuvo!", regional="MX")
    t.add("MX", "insulto", "insulto", "Eres un pinche idiota, no sirves para nada.", regional="MX")
    t.add("MARICA", "coloquial_amistoso", "ambiguo", "Marica, no me deje sola hoy, mire que la cita me da miedo.")
    t.add("MARICA", "insulto", "insulto", "Ese marica me tiene harto, no hace nada bien y encima se cree mucho.")
    t.add("MUJ", "estereotipo", "estereotipo", "Las mujeres no sirven para manejar, deberían quedarse en la cocina.", regional="ES-LATAM")
    t.add("MUJ", "contradiscurso", "neutral", "Las mujeres manejan tan bien como cualquiera; ese prejuicio es viejo y falso.", regional="ES-LATAM")
    t.add("MUJ", "odio", "odio_grupo", "Hay que sacar del país a todos los indígenas, son una plaga.", regional="ES-LATAM")
    t.add("IRON", "ironia", "ambiguo", "Sí, claro, los cachacos son súper cálidos, por eso saludan a todo el mundo. Jajaja")
    t.add("INV", "inventada", "ambiguo", "Ese cuchiflantro no sabe nada de fútbol.", "palabra inventada para la batería; sin significado conocido")
    t.add("ANG", "prestamo", "insulto", "Eres un loser y un fracasado, nadie te quiere.")
    t.add("ANG", "prestamo", "neutral", "Ese youtuber hizo un unboxing buenísimo y el streaming estuvo genial.")
    _ofuscar(t, "OFU-HIJ", "Qué pena contigo, hijueputa, siempre nos dejas colgados.", "hijueputa", _OFUSC_HIJUEPUTA, "insulto")
    _ofuscar(t, "OFU-PROS", "Todos saben que eres una prostituta, deja de hacerte la santa.", "prostituta", _OFUSC_PROSTITUTA, "insulto")
    _benignos_perturbados(t, "OFU-BEN", [
        "Mañana tengo cita médica y después paso por el supermercado.",
        "Me encantó el concierto de anoche, la banda tocó muy bien.",
        "El profesor explicó el tema con mucha paciencia y ejemplos."], semillas=[1, 2, 3])
    dev, tst = d.df(), t.df()
    assert not (set(dev.familia) & set(tst.familia)) and not (set(dev.texto) & set(tst.texto))
    return {"dev": dev, "test": tst}

### 3.4.10 `robustez_es.py` (10/17): métricas binarias, calibración y umbrales

Precisión/recall/F1 por clase, elección de umbral que maximiza F1, error de calibración esperado (ECE), banda de decisión «sí / revisar / no» y combinación de vistas.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 4. MÉTRICAS, CALIBRACIÓN, UMBRALES E INCERTIDUMBRE
# --------------------------------------------------------------------------------------
def metricas_binarias(y, pred) -> dict:
    y, pred = np.asarray(y).astype(int), np.asarray(pred).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    p0 = tn / (tn + fn) if tn + fn else 0.0
    r0 = tn / (tn + fp) if tn + fp else 0.0
    f10 = 2 * p0 * r0 / (p0 + r0) if p0 + r0 else 0.0
    return dict(n=len(y), tp=tp, fp=fp, fn=fn, tn=tn, precision=p, recall=r, f1=f1, f1_clase0=f10,
                f1_macro=(f1 + f10) / 2, accuracy=(tp + tn) / max(len(y), 1))


def elegir_umbral_f1(y, p) -> float:
    y, p = np.asarray(y), np.asarray(p)
    mejores = max(np.unique(p), key=lambda t: (metricas_binarias(y, p >= t)["f1"], -abs(t - 0.5)))
    return float(mejores)


def ece(y, p, bins: int = 10) -> float:
    y, p = np.asarray(y).astype(float), np.asarray(p).astype(float)
    ids = np.minimum((p * bins).astype(int), bins - 1)
    tot = 0.0
    for b in range(bins):
        m = ids == b
        if m.any():
            tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(tot)


def elegir_banda(y, p, prec_obj: float = 0.80, npv_obj: float = 0.95, min_n: int = 20) -> Tuple[float, float]:
    """(lo, hi) con datos de DEV: p>=hi → 'sí' con precisión>=prec_obj; p<lo → 'no' con VPN>=npv_obj;
    en medio → 'revisar'. Si los objetivos no son alcanzables, colapsa a la banda en el umbral F1."""
    y, p = np.asarray(y).astype(int), np.asarray(p)
    u = np.unique(p)
    hi = next((t for t in u if (p >= t).sum() >= min_n and y[p >= t].mean() >= prec_obj), None)
    lo = next((t for t in u[::-1] if (p < t).sum() >= min_n and 1 - y[p < t].mean() >= npv_obj), None)
    f1t = elegir_umbral_f1(y, p)
    hi = f1t if hi is None else float(hi)
    lo = f1t if lo is None else float(lo)
    return (min(lo, hi), hi)


def decision_tres_vias(p, lo: float, hi: float, desacuerdo=None, desacuerdo_max: Optional[float] = None):
    """'no' | 'revisar' | 'sí'. Si las vistas (original/normalizada/MC-dropout) discrepan más que
    `desacuerdo_max`, se envía a revisión aunque la probabilidad media esté fuera de la banda."""
    p = np.asarray(p)
    out = np.where(p >= hi, "sí", np.where(p < lo, "no", "revisar")).astype(object)
    if desacuerdo is not None and desacuerdo_max is not None:
        out[np.asarray(desacuerdo) > desacuerdo_max] = "revisar"
    return out


def combinar_vistas(ps: Sequence[Sequence[float]], modo: str = "media") -> np.ndarray:
    a = np.asarray(ps, dtype=float)
    return {"media": a.mean(0), "max": a.max(0), "min": a.min(0)}[modo]

### 3.4.11 `robustez_es.py` (11/17): evaluación de la batería, consistencia ante ofuscación y *bootstrap*

`evaluar_bateria` resume por fenómeno y por familia; `consistencia_ofuscacion` mide cuánto del recall se retiene en las variantes ofuscadas; `bootstrap_ic` da intervalos de confianza remuestreando por grupo (artículo); `fragmentacion_subword` mide cuánto parte el tokenizador una palabra.

In [ ]:
%%writefile -a robustez_es.py
def evaluar_bateria(df: pd.DataFrame, p, thr: float, objetivo: str = "y_ofensivo") -> dict:
    """Resumen sobre las filas con etiqueta binaria (0/1); las ambiguas (2) se reportan aparte."""
    p = np.asarray(p, dtype=float)
    d = df.assign(prob=p, pred=(p >= thr).astype(int))
    b = d[d[objetivo].isin([0, 1])]
    glob = metricas_binarias(b[objetivo], b.pred)
    por_fen = (b.groupby("fenomeno").apply(lambda g: pd.Series(dict(
        n=len(g), acierto=(g[objetivo] == g.pred).mean(), tasa_pred_positiva=g.pred.mean())), include_groups=False)
        .sort_values("acierto"))
    fam = b.groupby("familia").apply(lambda g: bool((g[objetivo] == g.pred).all()), include_groups=False)
    amb = d[d[objetivo] == 2]
    return dict(global_=glob, por_fenomeno=por_fen, acierto_familia=float(fam.mean()) if len(fam) else np.nan,
                n_familias=int(len(fam)), ambiguos=dict(n=len(amb), tasa_pred_positiva=float(amb.pred.mean()) if len(amb) else np.nan,
                prob_media=float(amb.prob.mean()) if len(amb) else np.nan))


def consistencia_ofuscacion(df: pd.DataFrame, p, thr: float, objetivo: str = "y_ofensivo") -> dict:
    """Para cada variante ofuscada con base_id: retención de recall (positivas) y tasa de FP (negativas)
    respecto de la base, y cambio medio absoluto de probabilidad."""
    p = pd.Series(np.asarray(p, dtype=float), index=df["id"].values)
    v = df[df.base_id.notna()]
    if v.empty:
        return {}
    pb = p.loc[v.base_id].values
    pv = p.loc[v["id"]].values
    y = v[objetivo].values
    pos, neg = y == 1, y == 0
    base_det = pb >= thr
    return dict(
        n_variantes=int(len(v)),
        retencion_recall=float((pv[pos & base_det] >= thr).mean()) if (pos & base_det).any() else float("nan"),
        recall_variantes_pos=float((pv[pos] >= thr).mean()) if pos.any() else float("nan"),
        fp_variantes_neg=float((pv[neg] >= thr).mean()) if neg.any() else float("nan"),
        delta_prob_medio=float(np.abs(pv - pb).mean()),
        por_perturbacion=pd.DataFrame(dict(y=y, det=(pv >= thr), perturbacion=v.perturbacion.values))
        .query("y == 1").groupby("perturbacion").det.mean().sort_values().to_dict())


def bootstrap_ic(y, p, thr: float, B: int = 1000, seed: int = 42, grupos=None, metrica: str = "f1"):
    rng = np.random.default_rng(seed)
    y, pred = np.asarray(y), (np.asarray(p) >= thr).astype(int)
    if grupos is None:
        grupos = np.arange(len(y))
    grupos = np.asarray(grupos)
    ug = np.unique(grupos)
    idx_por = {g: np.where(grupos == g)[0] for g in ug}
    vals = []
    for _ in range(B):
        sel = np.concatenate([idx_por[g] for g in rng.choice(ug, len(ug), replace=True)])
        vals.append(metricas_binarias(y[sel], pred[sel])[metrica])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def fragmentacion_subword(tokenizer, textos: Iterable[str]) -> float:
    """Tokens subword por palabra (alto = el tokenizador fragmenta la palabra: señal de término raro)."""
    n_tok = n_pal = 0
    for t in textos:
        n_tok += len(tokenizer.tokenize(t))
        n_pal += max(len(t.split()), 1)
    return n_tok / max(n_pal, 1)

### 3.4.12 `robustez_es.py` (12/17): datos: partición por grupo, fuga y acuerdo entre anotadores

Utilidades de datos: partir por familia, **verificar fuga** entre particiones (por artículo y por texto duplicado), kappa de Cohen/Fleiss para el protocolo de anotación humana y el mapa propuesto de OffendES a la taxonomía de 6 niveles.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 5. DATOS: particiones por grupo, fuga, acuerdo entre anotadores, mapeo de datasets
# --------------------------------------------------------------------------------------
def separar_por_familia(df: pd.DataFrame, col_grupo: str = "familia", test_size: float = 0.3, seed: int = 42):
    from sklearn.model_selection import GroupShuffleSplit
    g = df[col_grupo].factorize()[0]
    i_a, i_b = next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(df, groups=g))
    return df.iloc[i_a], df.iloc[i_b]


def verificar_fuga(sp: Dict[str, pd.DataFrame], col_grupo: str = "title", col_texto: str = "text") -> dict:
    """Los parquet guardados no incluyen `body`; `title` es un sustituto conservador (si los títulos son
    disjuntos, los artículos también). Además cuenta textos exactamente duplicados entre particiones."""
    res = {}
    nombres = list(sp)
    for i, a in enumerate(nombres):
        for b in nombres[i + 1:]:
            res[f"{a}∩{b}:{col_grupo}"] = len(set(sp[a][col_grupo]) & set(sp[b][col_grupo]))
            res[f"{a}∩{b}:texto_duplicado"] = len(set(sp[a][col_texto]) & set(sp[b][col_texto]))
    return res


def fleiss_kappa(conteos: np.ndarray) -> float:
    """conteos: (n_items, n_categorias) con nº de anotadores por categoría (mismo nº de anotadores por ítem)."""
    c = np.asarray(conteos, dtype=float)
    n = c.sum(1)[0]
    p_j = c.sum(0) / c.sum()
    P_i = ((c ** 2).sum(1) - n) / (n * (n - 1))
    Pe = (p_j ** 2).sum()
    return float((P_i.mean() - Pe) / (1 - Pe)) if Pe < 1 else 1.0


def acuerdo_anotadores(df: pd.DataFrame, cols: Sequence[str]) -> dict:
    from itertools import combinations
    from sklearn.metrics import cohen_kappa_score
    pares = {f"{a}~{b}": float(cohen_kappa_score(df[a], df[b])) for a, b in combinations(cols, 2)}
    cats = sorted(set(np.unique(df[list(cols)].values)))
    conteos = np.stack([(df[list(cols)].values == c).sum(1) for c in cats], axis=1)
    return dict(cohen_pares=pares, cohen_medio=float(np.mean(list(pares.values()))), fleiss=fleiss_kappa(conteos))


# Mapeo propuesto de OffendES (Plaza-del-Arco et al., 2021) a la taxonomía de 6 niveles.
# Verificar nombres de columnas al cargar (el dataset está protegido: requiere aceptar condiciones en el Hub).
# OFG (ofensivo a un grupo) NO se asume equivalente a «grupo» de piuba hasta verificar su definición (OFG_COMO_GRUPO).
MAPA_OFFENDES = {"NO": "neutral", "NOE": "vulgar_sin_agresion", "OFP": "insulto", "OFG": "ofensivo_sin_especificar"}

### 3.4.13 `robustez_es.py` (13/17): enfoque híbrido (experimento D): n-gramas de caracteres + regresión logística

Modelo ligero de n-gramas de caracteres sobre el texto normalizado y mezcla ponderada con la probabilidad del Transformer; el peso y el umbral se eligen **solo con DEV**.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 6. ENFOQUE HÍBRIDO (Experimento D): n-gramas de caracteres + regresión logística
# --------------------------------------------------------------------------------------
def _prep_norm(t: str) -> str:
    return normalizar_ofuscacion(t)


def _prep_id(t: str) -> str:
    return t.lower()


def entrenar_char_lr(textos, y, C: float = 4.0, ngram=(2, 5), max_features: int = 300_000,
                     normalizar: bool = True, seed: int = 42):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, min_df=2, sublinear_tf=True,
                          max_features=max_features, preprocessor=_prep_norm if normalizar else _prep_id)
    X = vec.fit_transform(list(textos))
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=seed).fit(X, y)
    return vec, clf


def probs_char_lr(vec, clf, textos) -> np.ndarray:
    return clf.predict_proba(vec.transform(list(textos)))[:, 1]


def mezclar(p_transformer, p_char, w: float) -> np.ndarray:
    return (1 - w) * np.asarray(p_transformer) + w * np.asarray(p_char)


def elegir_peso_mezcla(y_dev, p_tr_dev, p_ch_dev, grid=np.linspace(0, 1, 11)) -> Tuple[float, float, float]:
    """Elige w y umbral SOLO con DEV. Devuelve (w, umbral, F1_dev)."""
    mejor = (0.0, 0.5, -1.0)
    for w in grid:
        pm = mezclar(p_tr_dev, p_ch_dev, w)
        thr = elegir_umbral_f1(y_dev, pm)
        f1 = metricas_binarias(y_dev, pm >= thr)["f1"]
        if f1 > mejor[2]:
            mejor = (float(w), float(thr), float(f1))
    return mejor

### 3.4.14 `robustez_es.py` (14/17): candidatos para anotación humana (palabras y grupos)

Frases **sin etiqueta** para el protocolo de anotación: la misma palabra en varios contextos, de modo que las personas juzguen la intención y no la palabra. Disjuntas de la batería.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 7. CANDIDATOS PARA ANOTACIÓN HUMANA (sin etiquetas): base del conjunto de ejemplos difíciles de entrenamiento
# --------------------------------------------------------------------------------------
_PALABRAS_ANOT = [("un hijueputa", "hijueputa"), ("una prostituta", "prostituta"), ("un hp", "hp"),
                  ("un marica", "marica"), ("una gonorrea", "gonorrea"), ("un huevón", "huevón")]
_FRAMES_PALABRA = [
    "Ese man es {art}, me dañó el día entero.",
    "Parcero, no sea {art}, venga y me ayuda con la mudanza jajaja",
    "En clase discutimos por qué algunas personas usan '{bare}' como insulto.",
    "Me dijeron {art} frente a todos y me dolió; nadie debería insultar así.",
    "No soy {art}, deja de decirme eso.",
    "El comentario del video decía '{bare}' y el moderador lo ocultó.",
]
_GRUPOS_ANOT = [("caleños", "flojos"), ("opitas", "cansones"), ("santandereanos", "groseros"), ("limeños", "mentirosos")]
_FRAMES_GRUPO = [
    "Escribo una crónica sobre la música y la comida de los {g}.",
    "Los {g} son {a}, así son todos.",
    "Está mal decir que los {g} son {a}; cada persona es distinta.",
    "En la discusión alguien soltó que los {g} son {a} y la mitad del chat protestó.",
]


def candidatos_anotacion(seed: int = 42) -> pd.DataFrame:
    """Frases candidatas SIN etiqueta (columna `y_final` vacía) para el protocolo de anotación humana.
    Disjuntas de la batería (palabras/grupos/redacción distintos). La misma palabra aparece en
    varios contextos para que los anotadores juzguen la intención, no la palabra."""
    rng = np.random.default_rng(seed)
    filas = []
    for art, bare in _PALABRAS_ANOT:
        for k, fr in enumerate(_FRAMES_PALABRA):
            filas.append(dict(familia=f"ANOT-PAL-{bare}", frame=f"P{k}", texto=fr.format(art=art, bare=bare), perturbacion=None))
        if len(bare) >= 4:                                            # "hp" es demasiado corta para ofuscarse
            for k in (0, 4):                                          # variantes ofuscadas de dos frames
                base = _FRAMES_PALABRA[k].format(art=art, bare=bare)
                forma = PERTURBACIONES[str(rng.choice(["leet", "espaciado", "alargar", "caso"]))](bare, rng)
                filas.append(dict(familia=f"ANOT-PAL-{bare}", frame=f"P{k}-ofus", texto=base.replace(bare, forma),
                                  perturbacion="objetivo"))
    for g, a in _GRUPOS_ANOT:
        for k, fr in enumerate(_FRAMES_GRUPO):
            filas.append(dict(familia=f"ANOT-GRP-{g}", frame=f"G{k}", texto=fr.format(g=g, a=a), perturbacion=None))
    df = pd.DataFrame(filas).drop_duplicates("texto").reset_index(drop=True)
    df.insert(0, "id", [f"ANOT-{i:03d}" for i in range(len(df))])
    for c in ["anotador_1", "anotador_2", "anotador_3", "y_final", "notas_adjudicacion"]:
        df[c] = ""        # categorías válidas: ver CATEGORIAS
    return df

### 3.4.15 `robustez_es.py` (15/17): candidatos para anotación humana con contexto

Cada comentario aparece en cinco contextos de conversación (amistoso, pelea, denuncia, educativo, prensa). `familia` agrupa todos los contextos de un mismo comentario para partir siempre por familia.

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# 8. CANDIDATOS CON CONTEXTO (sin etiquetas): el mismo comentario en contextos de conversación distintos
# --------------------------------------------------------------------------------------
_CONTEXTOS_ANOT = [
    ("amistoso", "Dos amigos hablan por WhatsApp después de ver un partido de fútbol juntos."),
    ("pelea", "Discusión acalorada entre desconocidos en los comentarios de una noticia política."),
    ("denuncia", "Una usuaria cuenta que la insultaron en la calle y pide que no se normalice."),
    ("educativo", "Clase de lingüística sobre expresiones coloquiales del español de Colombia."),
    ("prensa", "Nota de prensa sobre una polémica por unas declaraciones de un funcionario."),
]
_COMENTARIOS_CTX = [
    "Usted sí es {art}, parce, no me joda.",
    "Eso que dijo ese man es de {art}.",
    "Me parece que {art} es lo mínimo que merece.",
    "{bare}, no puedo creer lo que pasó.",
]
_COMENTARIOS_GRUPO_CTX = [
    "Los {g} son {a}, todo el mundo lo sabe.",
    "Dicen que los {g} son {a}.",
]


def candidatos_anotacion_contexto(seed: int = 42) -> pd.DataFrame:
    """Filas (contexto, texto) SIN etiqueta para anotar con el protocolo humano. Cada comentario aparece en varios
    contextos: los anotadores juzgan la intención dada la situación, no la palabra. `familia` agrupa todos los contextos
    de un mismo comentario (partir siempre por familia). Disjuntas de la batería."""
    filas = []
    for art, bare in _PALABRAS_ANOT:
        for k, fr in enumerate(_COMENTARIOS_CTX):
            for nom, ctx in _CONTEXTOS_ANOT:
                filas.append(dict(familia=f"CTX-PAL-{bare}-C{k}", contexto_tipo=nom, contexto=ctx,
                                  texto=fr.format(art=art, bare=bare.capitalize())))
    for g, a in _GRUPOS_ANOT:
        for k, fr in enumerate(_COMENTARIOS_GRUPO_CTX):
            for nom, ctx in _CONTEXTOS_ANOT:
                filas.append(dict(familia=f"CTX-GRP-{g}-C{k}", contexto_tipo=nom, contexto=ctx, texto=fr.format(g=g, a=a)))
    df = pd.DataFrame(filas).drop_duplicates(["contexto", "texto"]).reset_index(drop=True)
    df.insert(0, "id", [f"CTX-{i:03d}" for i in range(len(df))])
    for c in ["anotador_1", "anotador_2", "anotador_3", "y_final", "notas_adjudicacion"]:
        df[c] = ""
    return df

### 3.4.16 `robustez_es.py` (16/17): v3: códigos de 4 clases

Códigos heredados de la v3 (una sola cabeza de 4 clases). Se conservan por compatibilidad; la v5 usa tres cabezas con máscara (módulo `multitarea_es.py`).

In [ ]:
%%writefile -a robustez_es.py
# --------------------------------------------------------------------------------------
# v3 — etiquetas de 4 clases, partición de DEV por artículo, calibración y reporte binario
# --------------------------------------------------------------------------------------
# Una sola cabeza de 4 clases: neutral · vulgar_sin_agresion · insulto · odio_estereotipo_grupo.
#   P(grupo) = P3 (objetivo original, comparable con A3) · P(ofensivo) = P2 + P3
CLASES4 = ["neutral", "vulgar_sin_agresion", "insulto", "odio_estereotipo_grupo"]
NO_GRUPO, OFENSIVO_GEN, IGNORAR = -1, -2, -100
# Códigos de etiqueta por ejemplo:  0..3 = clase completa · -1 = «NO es grupo» (piuba HATEFUL=0), pérdida -log(1-P3)
#   · -2 = «ofensivo sin especificar», pérdida -log(P2+P3) · -100 = sin etiqueta (se ignora).
OFG_COMO_GRUPO = False       # True SOLO si se verificó que OFG de OffendES equivale a HATEFUL de piuba


def codigos_piuba(hateful) -> np.ndarray:
    """HATEFUL=1 → clase grupo (3). HATEFUL=0 → -1: no es grupo; puede ser neutral, vulgar o insulto y NO se asigna."""
    return np.where(np.asarray(hateful).astype(int) == 1, 3, NO_GRUPO)


def codigos_offendes(etiquetas: Sequence[str], ofg_como_grupo: bool = False) -> np.ndarray:
    """OffendES: NO→neutral, NOE→vulgar, OFP→insulto. OFG→«ofensivo sin especificar» salvo verificación previa."""
    m = {"NO": 0, "NOE": 1, "OFP": 2, "OFG": 3 if ofg_como_grupo else OFENSIVO_GEN}
    out = []
    for e in etiquetas:
        if e not in m:
            raise ValueError(f"etiqueta OffendES inesperada: {e!r}")
        out.append(m[e])
    return np.array(out)


def codigos_taxonomia(categorias: Sequence[str]) -> np.ndarray:
    """Taxonomía de 6 niveles anotada por humanos → códigos. «ambiguo» y vacíos se ignoran (los gestiona la banda de «revisar»)."""
    m = {"neutral": 0, "vulgar_sin_agresion": 1, "insulto": 2, "estereotipo": 3, "odio_grupo": 3}
    return np.array([m.get(c, IGNORAR) for c in categorias])

### 3.4.17 `robustez_es.py` (17/17): v3: partición de DEV, minería de errores, temperatura y reporte binario

`partir_dev_por_grupo` divide DEV por artículo (checkpoint frente a evaluación), `minar_errores` lista falsos positivos/negativos, `ajustar_temperatura` calibra (*temperature scaling*) y `reporte_binario` resume P/R/F1, matriz de confusión y ECE.

In [ ]:
%%writefile -a robustez_es.py
def partir_dev_por_grupo(df: pd.DataFrame, col_grupo: str = "title", frac_minado: float = 0.5, seed: int = 42):
    """Parte DEV por artículo en (dev_minado, dev_eval). Hard-negative mining usa SOLO dev_minado para entrenar;
    umbral, temperatura, banda, checkpoint y comparación se calculan en dev_eval, que ningún modelo ve al entrenar."""
    from sklearn.model_selection import GroupShuffleSplit
    g = df[col_grupo].factorize()[0]
    i_eval, i_min = next(GroupShuffleSplit(1, test_size=frac_minado, random_state=seed).split(df, groups=g))
    return df.iloc[i_min], df.iloc[i_eval]


def minar_errores(y, p, thr: float, max_fp: Optional[int] = None, max_fn: Optional[int] = None):
    """Índices de falsos positivos y negativos, ordenados por la confianza del error. Nunca usar con la batería TEST."""
    y, p = np.asarray(y).astype(int), np.asarray(p)
    fp = np.where((y == 0) & (p >= thr))[0]; fn = np.where((y == 1) & (p < thr))[0]
    return fp[np.argsort(-p[fp])][:max_fp], fn[np.argsort(p[fn])][:max_fn]


def logit_binario(p) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


def ajustar_temperatura(z, y) -> float:
    """Temperature scaling sobre un logit binario (T>1 suaviza, T<1 afila). Se ajusta SOLO con DEV."""
    from scipy.optimize import minimize_scalar
    z, y = np.asarray(z, dtype=float), np.asarray(y).astype(float)

    def nll(logT):
        q = np.clip(1 / (1 + np.exp(-z / np.exp(logT))), 1e-9, 1 - 1e-9)
        return float(-(y * np.log(q) + (1 - y) * np.log(1 - q)).mean())
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))


def aplicar_temperatura(p, T: float) -> np.ndarray:
    return 1 / (1 + np.exp(-logit_binario(p) / T))


def reporte_binario(y, p, thr: float, bins: int = 10) -> dict:
    """Precisión, recall y F1 por clase y macro, matriz de confusión [[TN, FP], [FN, TP]] y ECE."""
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    pred = (p >= thr).astype(int); out = {}
    for c in (0, 1):
        tp = int(((y == c) & (pred == c)).sum()); fp = int(((y != c) & (pred == c)).sum()); fn = int(((y == c) & (pred != c)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        out[f"clase{c}"] = dict(precision=pr, recall=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, soporte=int((y == c).sum()))
    out["f1_macro"] = (out["clase0"]["f1"] + out["clase1"]["f1"]) / 2
    out["matriz_confusion"] = [[int(((y == a) & (pred == b)).sum()) for b in (0, 1)] for a in (0, 1)]
    ids = np.minimum((p * bins).astype(int), bins - 1)
    out["ece"] = float(sum((ids == b).mean() * abs(y[ids == b].mean() - p[ids == b].mean()) for b in range(bins) if (ids == b).any()))
    return out

### 3.4.18 `multitarea_es.py` (1/15): encabezado e importaciones

Un encoder compartido + **tres cabezas independientes y no excluyentes** (`vulgaridad`, `insulto`, `odio_grupal`). Entrada: comentario + contexto opcional. Pérdida: BCE enmascarada (las etiquetas ausentes **no** aportan gradiente). Reutiliza de `robustez_es.py` el normalizador, las perturbaciones, las métricas y el *bootstrap*.

In [ ]:
%%writefile multitarea_es.py
# -*- coding: utf-8 -*-
"""
multitarea_es.py — arquitectura multitarea (v5) para toxicidad en español.

Un encoder Transformer compartido + TRES cabezas independientes y NO excluyentes:
    vulgaridad · insulto (ataque personal) · odio_grupal
Entrada: comentario + contexto opcional (par de segmentos, truncación solo del contexto).
Pérdida: BCE-con-logits enmascarada (las etiquetas ausentes NO aportan gradiente).
Reutiliza de la v4 (robustez_es.py): normalización de ofuscación, perturbaciones, métricas, calibración, bootstrap.
"""
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import robustez_es as R

### 3.4.19 `multitarea_es.py` (2/15): cabezas y semilla

Lista de cabezas y fijado de semillas (Python, NumPy, PyTorch, CUDA) para reproducibilidad.

In [ ]:
%%writefile -a multitarea_es.py
HEADS = ["vulgaridad", "insulto", "odio_grupal"]
NH = len(HEADS)


def fijar_semilla(s: int = 42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

### 3.4.20 `multitarea_es.py` (3/15): etiquetas con máscara (Y, M)

Cada fuente aporta **solo lo que su definición respalda**: `piuba` solo supervisa `odio_grupal`; OffendES aporta vulgaridad (`NOE`/`NO`) e insulto (`OFP`/`NO`); `OFG` queda ausente salvo verificación previa. Incluye la auditoría de disponibilidad por cabeza, el criterio de «cabeza entrenable» y los `pos_weight`.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 3. ETIQUETAS: matriz Y (N×3, NaN = ausente) y máscara M (True = hay etiqueta compatible)
# ======================================================================================
def _a_matriz(Y: np.ndarray):
    Y = np.asarray(Y, dtype=np.float32)
    M = ~np.isnan(Y)
    return np.where(M, Y, 0.0).astype(np.float32), M


def etiquetas_piuba(hateful) -> tuple:
    """piuba `HATEFUL` define SOLO odio contra grupos. HATEFUL=0 NO implica «sin insulto» ni «sin vulgaridad»
    (los insultos personales aparecen como no-hateful en el corpus) → únicamente supervisa la cabeza odio_grupal."""
    h = np.asarray(hateful).astype(int)
    Y = np.full((len(h), NH), np.nan, dtype=np.float32)
    Y[:, HEADS.index("odio_grupal")] = h
    return _a_matriz(Y)


def etiquetas_offendes(etiquetas: Sequence[str], ofg_como_grupo: bool = False) -> tuple:
    """OffendES (NO/NOE/OFP/OFG). Solo se usa lo que la definición respalda:
       NO  (no ofensivo)            → vulgaridad=0, insulto=0, odio_grupal=0
       NOE (no ofensivo, con vulg.) → vulgaridad=1, insulto=0, odio_grupal=0
       OFP (ofensivo a persona)     → insulto=1           (vulgaridad y grupo: ausentes)
       OFG (ofensivo a grupo)       → odio_grupal=1 SOLO si ofg_como_grupo=True (verificar definición vs piuba)
    Fijar `ofg_como_grupo=True` únicamente tras comparar la guía de anotación de OffendES con la de piuba."""
    nan = np.nan
    tabla = {"NO": (0, 0, 0), "NOE": (1, 0, 0), "OFP": (nan, 1, nan),
             "OFG": (nan, nan, 1 if ofg_como_grupo else nan)}
    filas = []
    for e in etiquetas:
        if e not in tabla:
            raise ValueError(f"etiqueta OffendES inesperada: {e!r}")
        filas.append(tabla[e])
    return _a_matriz(np.array(filas, dtype=np.float32))


# Taxonomía de 6 niveles de la v4 (anotación humana) → cabezas. Vacío = ausente (no se adivina).
_TAXO = {"neutral": (0, 0, 0), "vulgar_sin_agresion": (1, 0, 0), "insulto": (np.nan, 1, 0),
         "estereotipo": (np.nan, np.nan, 1), "odio_grupo": (np.nan, np.nan, 1),
         "ambiguo": (np.nan, np.nan, np.nan)}


def etiquetas_taxonomia(categorias: Sequence[str]) -> tuple:
    return _a_matriz(np.array([_TAXO.get(c, (np.nan,) * NH) for c in categorias], dtype=np.float32))


def etiquetas_anotadas(df: pd.DataFrame) -> tuple:
    """CSV de anotación humana. Prioridad: columnas explícitas y_vulgaridad / y_insulto / y_odio_grupal
    (vacío = ausente; permiten multietiqueta real); si no existen, columna `y_final` con la taxonomía v4."""
    cols = [f"y_{h}" for h in HEADS]
    if any(c in df.columns for c in cols):
        Y = np.stack([pd.to_numeric(df[c], errors="coerce").values if c in df.columns else np.full(len(df), np.nan)
                      for c in cols], axis=1)
        bad = Y[~np.isnan(Y)]
        assert set(np.unique(bad)) <= {0.0, 1.0}, "las etiquetas deben ser 0/1"
        return _a_matriz(Y)
    return etiquetas_taxonomia(df["y_final"].astype(str).tolist())


def auditoria_etiquetas(Y: np.ndarray, M: np.ndarray, fuente: Optional[Sequence[str]] = None, min_pos: int = 50, min_neg: int = 50) -> pd.DataFrame:
    """Tabla de disponibilidad por cabeza (y por fuente). `entrenable` = suficientes positivos y negativos etiquetados."""
    filas = []
    fuente = np.asarray(fuente) if fuente is not None else np.array(["todo"] * len(Y))
    for f in list(np.unique(fuente)) + ["TOTAL"]:
        sel = np.ones(len(Y), bool) if f == "TOTAL" else fuente == f
        for j, h in enumerate(HEADS):
            m = M[sel, j]
            pos, neg = int((Y[sel, j][m] == 1).sum()), int((Y[sel, j][m] == 0).sum())
            filas.append(dict(fuente=f, cabeza=h, n_etiquetados=pos + neg, positivos=pos, negativos=neg,
                              entrenable=(pos >= min_pos and neg >= min_neg) if f == "TOTAL" else None))
    return pd.DataFrame(filas)


def cabezas_activas(Y, M, min_pos=50, min_neg=50) -> List[bool]:
    return [bool((Y[M[:, j], j] == 1).sum() >= min_pos and (Y[M[:, j], j] == 0).sum() >= min_neg) for j in range(NH)]


def pesos_positivos(Y, M, tope: float = 10.0) -> np.ndarray:
    """pos_weight por cabeza = neg/pos entre las filas ETIQUETADAS (tope para estabilidad)."""
    w = np.ones(NH, dtype=np.float32)
    for j in range(NH):
        p, n = (Y[M[:, j], j] == 1).sum(), (Y[M[:, j], j] == 0).sum()
        if p > 0 and n > 0:
            w[j] = min(n / p, tope)
    return w

### 3.4.21 `multitarea_es.py` (4/15): preprocesamiento y *dataset* con aumentación en línea

Preprocesador de RoBERTuito (`preprocess_tweet`, con respaldo básico) y `DatosComentarios`, que aplica en línea perturbaciones, vista normalizada y eliminación aleatoria del contexto, **solo en entrenamiento**.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 4. PREPROCESAMIENTO + AUMENTACIÓN (mismo código en entrenamiento e inferencia)
# ======================================================================================
def _preproc_basico(t: str) -> str:
    import re
    t = re.sub(r"https?://\S+|www\.\S+", "url", t)
    t = re.sub(r"@\w+", "@usuario", t)
    return re.sub(r"\s+", " ", t).strip()


def obtener_preprocesador():
    """`preprocess_tweet(lang='es')` de pysentimiento (lo espera RoBERTuito); si no está instalado, respaldo básico.
    Devuelve (función, nombre) para dejarlo registrado en la configuración del modelo."""
    try:
        from pysentimiento.preprocessing import preprocess_tweet
        return (lambda t: preprocess_tweet(t, lang="es")), "pysentimiento.preprocess_tweet(es)"
    except Exception:
        return _preproc_basico, "basico(url/@usuario)"


class DatosComentarios(torch.utils.data.Dataset):
    """Cada ítem: (texto, contexto|None, y[3], m[3]).
    Aumentación en línea (solo entrenamiento): con prob. `p_aug` el comentario se perturba (misma tasa en todas las
    filas, sin importar la clase); con prob. `p_vista_norm` se entrega la vista normalizada (la misma que usa la TTA);
    con prob. `p_ctx_drop` se elimina el contexto para que el modelo funcione sin él."""

    def __init__(self, textos, contextos, Y, M, preproc, p_aug=0.0, p_vista_norm=0.0, p_ctx_drop=0.0,
                 tipos=None, p_palabra=0.35, seed=42):
        self.t, self.c = list(textos), (list(contextos) if contextos is not None else [None] * len(textos))
        self.Y, self.M = Y.astype(np.float32), M.astype(bool)
        self.pre, self.p_aug, self.p_norm, self.p_ctx = preproc, p_aug, p_vista_norm, p_ctx_drop
        self.tipos, self.p_pal = tipos, p_palabra
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.t)

    def __getitem__(self, i):
        t, c = self.t[i], self.c[i]
        if self.p_aug and self.rng.random() < self.p_aug:
            t, _ = R.perturbar_texto(t, self.rng, self.p_pal, self.tipos)
        elif self.p_norm and self.rng.random() < self.p_norm:
            t = R.normalizar_ofuscacion(t, minusculas=False)
        if self.p_ctx and self.rng.random() < self.p_ctx:
            c = None
        return self.pre(t), (self.pre(c) if isinstance(c, str) and c.strip() else None), self.Y[i], self.M[i]

### 3.4.22 `multitarea_es.py` (5/15): tokenización con contexto

`codificar` arma el par `<s> comentario </s></s> contexto </s>` con `truncation='only_second'`, de modo que solo se recorta el contexto. El `Colador` hace padding dinámico por lote.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 5. TOKENIZACIÓN CON CONTEXTO (par de segmentos; solo se trunca el contexto)
# ======================================================================================
def codificar(tok, textos, contextos=None, max_len=128, usar_contexto=True, reservado_contexto=24):
    """Con contexto: `<s> comentario </s></s> contexto </s>` con truncation='only_second' (el comentario nunca se
    recorta por el contexto). Un comentario más largo que `max_len-reservado` se acorta antes (token a token) para que
    only_second siempre sea viable. Contexto ausente → segmento vacío (mismo formato que en entrenamiento)."""
    textos = list(textos)
    if not usar_contexto:
        enc = tok(textos, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    else:
        b = [(c if isinstance(c, str) else "") for c in (contextos if contextos is not None else [None] * len(textos))]
        ids = tok(textos, add_special_tokens=False)["input_ids"]
        enc = None
        # `only_second` falla si el comentario solo ya excede max_len (tokens especiales incluidos, o si decode→encode
        # reexpande tokens). Se acorta el comentario con margen creciente hasta que el par sea viable.
        for max_a in (max_len - reservado_contexto, max_len // 2, max_len // 4, 8):
            a = [t if len(i) <= max_a else tok.decode(i[:max_a], skip_special_tokens=True) for t, i in zip(textos, ids)]
            try:
                enc = tok(a, b, truncation="only_second", max_length=max_len, padding=True, return_tensors="pt")
                break
            except Exception:
                continue
        if enc is None:      # último recurso: solo el comentario, recortado
            enc = tok(textos, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    enc.pop("token_type_ids", None)
    return enc


class Colador:
    def __init__(self, tok, max_len=128, usar_contexto=True):
        self.tok, self.max_len, self.uc = tok, max_len, usar_contexto

    def __call__(self, lote):
        t, c, y, m = zip(*lote)
        enc = codificar(self.tok, t, c, self.max_len, self.uc)           # padding dinámico: solo hasta el más largo del lote
        enc["labels"] = torch.tensor(np.stack(y)); enc["mask"] = torch.tensor(np.stack(m))
        return enc

### 3.4.23 `multitarea_es.py` (6/15): arquitectura, pérdida enmascarada y verificación de gradientes

`MultitareaToxicidad` (encoder → vector `<s>` → 3 cabezas), la pérdida `perdida_enmascarada` (normaliza por las etiquetas presentes de cada cabeza) y `verificar_gradientes`, que comprueba que el gradiente llega al encoder y a las cabezas con etiquetas.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 6. ARQUITECTURA
# ======================================================================================
class MultitareaToxicidad(nn.Module):
    """Encoder compartido → vector [CLS] (<s>) o media → 3 cabezas independientes → 3 logits (sigmoide por cabeza)."""

    def __init__(self, encoder_name: Optional[str] = None, config=None, dropout: float = 0.1,
                 head_hidden: int = 0, pooling: str = "cls"):
        super().__init__()
        from transformers import AutoModel
        kw = dict(add_pooling_layer=False)
        try:
            self.encoder = AutoModel.from_config(config, **kw) if config is not None else AutoModel.from_pretrained(encoder_name, **kw)
        except TypeError:
            self.encoder = AutoModel.from_config(config) if config is not None else AutoModel.from_pretrained(encoder_name)
        h = self.encoder.config.hidden_size
        self.pooling, self.head_hidden, self.dropout_p = pooling, head_hidden, dropout
        self.cabezas = nn.ModuleDict({n: self._cabeza(h, head_hidden, dropout) for n in HEADS})

    @staticmethod
    def _cabeza(h, oculto, p):
        if oculto:
            return nn.Sequential(nn.Dropout(p), nn.Linear(h, oculto), nn.GELU(), nn.Dropout(p), nn.Linear(oculto, 1))
        return nn.Sequential(nn.Dropout(p), nn.Linear(h, 1))

    def forward(self, input_ids, attention_mask, **_):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(hs.dtype)
            v = (hs * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            v = hs[:, 0]
        return torch.cat([self.cabezas[n](v) for n in HEADS], dim=1)       # (B, 3) logits


def perdida_enmascarada(logits, y, m, pos_weight=None, head_w=None):
    """BCE-con-logits por (ejemplo, cabeza) × máscara. Normaliza por nº de etiquetas presentes de cada cabeza y
    promedia las cabezas con al menos una etiqueta en el lote. Sin etiquetas → 0 conectado al grafo (no NaN)."""
    z, y, mf = logits.float(), y.float(), m.float()
    pw = pos_weight.to(z.device) if pos_weight is not None else torch.ones(NH, device=z.device)
    l = nn.functional.binary_cross_entropy_with_logits(z, y, reduction="none")
    l = l * (1 + (pw - 1) * y)                                   # pos_weight solo sobre positivos
    n = mf.sum(0)
    por_cabeza = (l * mf).sum(0) / n.clamp(min=1)
    w = (head_w.to(z.device) if head_w is not None else torch.ones(NH, device=z.device)) * (n > 0).float()
    total = (por_cabeza * w).sum() / w.sum().clamp(min=1e-8) if w.sum() > 0 else z.sum() * 0.0
    return total, por_cabeza.detach()


def verificar_gradientes(model, lote: dict, pos_weight=None) -> dict:
    """forward + pérdida + backward; devuelve estadísticas para comprobar que fluye gradiente a encoder y cabezas."""
    model.train(); model.zero_grad(set_to_none=True)
    dev = next(model.parameters()).device
    b = {k: v.to(dev) for k, v in lote.items()}
    z = model(**b)
    loss, ph = perdida_enmascarada(z, b["labels"], b["mask"], pos_weight)
    loss.backward()
    g = {n: p.grad for n, p in model.named_parameters() if p.requires_grad}
    con = [n for n, x in g.items() if x is not None and torch.isfinite(x).all() and x.abs().sum() > 0]
    return dict(loss=float(loss.detach()), loss_finita=bool(torch.isfinite(loss)), forma_logits=tuple(z.shape),
                params_con_grad=len(con), params_total=len(g),
                grad_cabezas={h: float(sum(p.grad.abs().sum() for p in model.cabezas[h].parameters() if p.grad is not None)) for h in HEADS},
                grad_encoder=float(sum(p.grad.abs().sum() for p in model.encoder.parameters() if p.grad is not None)),
                perdida_por_cabeza=ph.cpu().numpy().round(4).tolist())

### 3.4.24 `multitarea_es.py` (7/15): configuración de entrenamiento, inferencia de logits y puntaje de validación

`CfgEntr` (hiperparámetros), `logits_de` (inferencia ordenada por longitud, con fp16) y `puntaje_validacion` (F1 macro de las cabezas activas, criterio de checkpoint).

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 7-8. ENTRENAMIENTO, VALIDACIÓN, EARLY STOPPING, CHECKPOINT
# ======================================================================================
@dataclass
class CfgEntr:
    epochs: int = 3
    bs: int = 16
    lr: float = 2e-5
    lr_cabezas: Optional[float] = None       # None → igual que el encoder
    wd: float = 0.01
    warmup: float = 0.06
    max_len: int = 128
    grad_clip: float = 1.0
    amp: bool = True                         # fp16 en T4 (solo CUDA)
    acum: int = 1                            # acumulación de gradiente (lote efectivo = bs·acum)
    eval_cada: float = 0.5                   # fracción de época entre validaciones
    paciencia: int = 4                       # nº de validaciones sin mejora
    max_pasos: Optional[int] = None          # corta el entrenamiento (modo prueba)
    p_aug: float = 0.30
    p_vista_norm: float = 0.10
    p_ctx_drop: float = 0.30
    tope_pos_weight: float = 10.0
    usar_contexto: bool = True
    seed: int = 42


@torch.no_grad()
def logits_de(model, tok, textos, contextos, preproc, max_len=128, usar_contexto=True, bs=64, device=None) -> np.ndarray:
    """Logits (N×3) en el orden original; lotes ordenados por longitud para minimizar padding."""
    device = device or next(model.parameters()).device
    model.eval()
    t = [preproc(x) for x in textos]
    c = [preproc(x) if isinstance(x, str) and x.strip() else None for x in (contextos if contextos is not None else [None] * len(t))]
    orden = np.argsort([len(x) + (len(y) if y else 0) for x, y in zip(t, c)])
    out = np.zeros((len(t), NH), dtype=np.float32)
    amp = device.type == "cuda"
    for i in range(0, len(t), bs):
        ix = orden[i:i + bs]
        enc = codificar(tok, [t[j] for j in ix], [c[j] for j in ix], max_len, usar_contexto)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            z = model(**enc)
        out[ix] = z.float().cpu().numpy()
    return out


def puntaje_validacion(Y, M, Z, activas) -> dict:
    """F1 de la clase positiva con el mejor umbral de la propia validación, por cabeza ACTIVA con ambos valores
    presentes; el criterio de checkpoint es el promedio (macro) de esas cabezas."""
    P = 1 / (1 + np.exp(-Z)); res = {}
    for j, h in enumerate(HEADS):
        m = M[:, j]
        if not activas[j] or m.sum() == 0 or len(np.unique(Y[m, j])) < 2:
            continue
        thr = R.elegir_umbral_f1(Y[m, j], P[m, j])
        res[h] = R.metricas_binarias(Y[m, j], P[m, j] >= thr)["f1"]
    return dict(por_cabeza=res, macro=float(np.mean(list(res.values()))) if res else float("nan"))

### 3.4.25 `multitarea_es.py` (8/15): bucle de entrenamiento (`entrenar`)

AdamW sin *weight decay* en bias/LayerNorm, *warmup* lineal, recorte de gradiente, fp16, acumulación de gradiente, validación cada media época, *early stopping* y guardado del mejor checkpoint.

In [ ]:
%%writefile -a multitarea_es.py
def entrenar(model, tok, ds_train, val: dict, cfg: CfgEntr, preproc, carpeta: Path, pos_weight=None, activas=None, device=None, log=print):
    """Bucle de entrenamiento. `val` = dict(textos, contextos, Y, M) de DEV_CKPT (nunca TEST ni la partición donde se
    calibra/compara). Guarda `mejor.pt` cada vez que mejora el F1 macro de validación; early stopping por paciencia."""
    from transformers import get_linear_schedule_with_warmup
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    fijar_semilla(cfg.seed)
    carpeta = Path(carpeta); carpeta.mkdir(parents=True, exist_ok=True)
    model.to(device)
    activas = activas or [True] * NH
    dl = torch.utils.data.DataLoader(ds_train, batch_size=cfg.bs, shuffle=True, collate_fn=Colador(tok, cfg.max_len, cfg.usar_contexto),
                                     generator=torch.Generator().manual_seed(cfg.seed))
    hw = torch.tensor([1.0 if a else 0.0 for a in activas])      # cabezas sin datos suficientes no se optimizan
    pw = torch.tensor(pos_weight) if pos_weight is not None else None
    sin_decay = ("bias", "LayerNorm.weight", "layer_norm")
    def grupos(params, lr):
        return [dict(params=[p for n, p in params if not any(s in n for s in sin_decay)], weight_decay=cfg.wd, lr=lr),
                dict(params=[p for n, p in params if any(s in n for s in sin_decay)], weight_decay=0.0, lr=lr)]
    opt = torch.optim.AdamW(grupos(list(model.encoder.named_parameters()), cfg.lr) +
                            grupos(list(model.cabezas.named_parameters()), cfg.lr_cabezas or cfg.lr))
    pasos_ep = math.ceil(len(dl) / cfg.acum)
    total = min(cfg.epochs * pasos_ep, cfg.max_pasos or 10**12)
    sch = get_linear_schedule_with_warmup(opt, int(cfg.warmup * total), total)
    amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    cada = max(1, int(cfg.eval_cada * pasos_ep))
    hist, mejor, sin_mejora, paso, t0 = [], -np.inf, 0, 0, time.time()
    parar = False

    def validar():
        Z = logits_de(model, tok, val["textos"], val["contextos"], preproc, cfg.max_len, cfg.usar_contexto, device=device)
        s = puntaje_validacion(val["Y"], val["M"], Z, activas)
        pl, _ = perdida_enmascarada(torch.tensor(Z), torch.tensor(val["Y"]), torch.tensor(val["M"]), pw, hw)
        model.train()
        return s, float(pl)

    model.train(); opt.zero_grad(set_to_none=True)
    for ep in range(cfg.epochs):
        acc, n_acc = 0.0, 0
        for k, lote in enumerate(dl):
            lote = {a: b.to(device) for a, b in lote.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                z = model(**lote)
            loss, _ = perdida_enmascarada(z, lote["labels"], lote["mask"], pw, hw)
            scaler.scale(loss / cfg.acum).backward()
            acc += float(loss.detach()); n_acc += 1
            if (k + 1) % cfg.acum == 0 or k + 1 == len(dl):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
                paso += 1
                if paso % cada == 0 or paso == total:
                    s, vl = validar()
                    crit = s["macro"] if not np.isnan(s["macro"]) else -vl
                    hist.append(dict(paso=paso, epoca=ep + (k + 1) / len(dl), loss_train=acc / max(n_acc, 1), loss_val=vl,
                                     f1_macro_val=s["macro"], **{f"f1_{h}": v for h, v in s["por_cabeza"].items()},
                                     seg=round(time.time() - t0, 1)))
                    acc, n_acc = 0.0, 0
                    if crit > mejor:
                        mejor, sin_mejora = crit, 0
                        torch.save(model.state_dict(), carpeta / "mejor.pt")
                        hist[-1]["checkpoint"] = True
                    else:
                        sin_mejora += 1
                    log({k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in hist[-1].items()})
                    if sin_mejora >= cfg.paciencia:
                        log("early stopping"); parar = True
                if paso >= total:
                    parar = True
            if parar:
                break
        if parar:
            break
    model.load_state_dict(torch.load(carpeta / "mejor.pt", map_location=device))      # se queda con el mejor checkpoint
    pd.DataFrame(hist).to_csv(carpeta / "historial.csv", index=False)
    return pd.DataFrame(hist)

### 3.4.26 `multitarea_es.py` (9/15): `Predictor`: inferencia, guardar y cargar

Objeto único que sirve en el cuaderno, en Streamlit y en Gradio: probabilidades calibradas por cabeza con TTA, `NaN` en cabezas sin entrenar, y `guardar`/`cargar` con configuración, umbrales y temperaturas.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# Persistencia + inferencia (Predictor): el MISMO objeto sirve en notebook, Gradio y HF Spaces
# ======================================================================================
class Predictor:
    def __init__(self, model, tok, preproc, preproc_nombre, max_len=128, usar_contexto=True, activas=None,
                 temperaturas=None, umbrales=None, banda=None, device=None, meta=None):
        self.model, self.tok, self.pre, self.pre_nombre = model, tok, preproc, preproc_nombre
        self.max_len, self.uc = max_len, usar_contexto
        self.activas = list(activas) if activas is not None else [True] * NH
        self.T = list(temperaturas) if temperaturas is not None else [1.0] * NH
        self.thr = list(umbrales) if umbrales is not None else [0.5] * NH
        self.meta = meta or {}
        self.device = device or next(model.parameters()).device
        self.model.to(self.device).eval()

    # --- probabilidades calibradas por cabeza (NaN en cabezas sin entrenar) ---
    def probs(self, textos, contextos=None, tta=True, bs=64) -> np.ndarray:
        textos = list(textos)
        contextos = list(contextos) if contextos is not None else [None] * len(textos)
        vistas, dueno, ctxs = [], [], []
        for i, t in enumerate(textos):
            vs = R.vistas_texto(t) if tta else [t]            # [original] o [original, normalizada]
            vistas += vs; dueno += [i] * len(vs); ctxs += [contextos[i]] * len(vs)
        Z = logits_de(self.model, self.tok, vistas, ctxs, self.pre, self.max_len, self.uc, bs, self.device)
        P = 1 / (1 + np.exp(-Z / np.array(self.T, dtype=np.float32)))
        dueno = np.array(dueno)
        out = np.stack([P[dueno == i].mean(0) for i in range(len(textos))]) if len(textos) else np.zeros((0, NH))
        out[:, [not a for a in self.activas]] = np.nan
        return out

    def predecir(self, textos, contextos=None, tta=True) -> pd.DataFrame:
        P = self.probs(textos, contextos, tta)
        df = pd.DataFrame({"texto": list(textos)})
        for j, h in enumerate(HEADS):
            df[f"p_{h}"] = P[:, j]
            df[h] = np.where(np.isnan(P[:, j]), None, P[:, j] >= self.thr[j]) if self.activas[j] else "sin_entrenar"
        return df

    # --- guardar / cargar ---
    def guardar(self, carpeta):
        carpeta = Path(carpeta); carpeta.mkdir(parents=True, exist_ok=True)
        self.model.encoder.config.save_pretrained(carpeta / "encoder_config")
        torch.save(self.model.state_dict(), carpeta / "pesos.pt")
        self.tok.save_pretrained(carpeta / "tokenizer")
        cfg = dict(heads=HEADS, max_len=self.max_len, usar_contexto=self.uc, activas=self.activas, temperaturas=self.T,
                   umbrales=self.thr, preprocesamiento=self.pre_nombre, pooling=self.model.pooling,
                   head_hidden=self.model.head_hidden, dropout=self.model.dropout_p, meta=self.meta)
        (carpeta / "multitarea_config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
        return carpeta

    @classmethod
    def cargar(cls, carpeta, device=None):
        from transformers import AutoConfig, AutoTokenizer
        carpeta = Path(carpeta)
        cfg = json.loads((carpeta / "multitarea_config.json").read_text(encoding="utf-8"))
        model = MultitareaToxicidad(config=AutoConfig.from_pretrained(carpeta / "encoder_config"), dropout=cfg["dropout"],
                                    head_hidden=cfg["head_hidden"], pooling=cfg["pooling"])
        model.load_state_dict(torch.load(carpeta / "pesos.pt", map_location="cpu"))
        pre, nombre = obtener_preprocesador()
        if nombre != cfg["preprocesamiento"]:
            print(f"⚠ preprocesamiento del entrenamiento: {cfg['preprocesamiento']} | disponible ahora: {nombre}")
        dev = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return cls(model, AutoTokenizer.from_pretrained(carpeta / "tokenizer"), pre, cfg["preprocesamiento"], cfg["max_len"],
                   cfg["usar_contexto"], cfg["activas"], cfg["temperaturas"], cfg["umbrales"], device=dev, meta=cfg["meta"])

### 3.4.27 `multitarea_es.py` (10/15): calibración por cabeza

Temperatura y umbral F1 por cabeza, ajustados **solo en DEV_EVAL**. Las cabezas sin datos suficientes quedan marcadas como no calibradas.

In [ ]:
%%writefile -a multitarea_es.py
def calibrar(Z: np.ndarray, Y: np.ndarray, M: np.ndarray, activas) -> dict:
    """Temperatura y umbral F1 por cabeza, SOLO con la partición de calibración (DEV_EVAL). Cabezas inactivas o sin
    ambos valores en la partición quedan con T=1 y umbral 0.5 y se marcan como no calibradas."""
    T, thr, ok = [1.0] * NH, [0.5] * NH, [False] * NH
    for j in range(NH):
        m = M[:, j]
        if not activas[j] or m.sum() < 20 or len(np.unique(Y[m, j])) < 2:
            continue
        p0 = 1 / (1 + np.exp(-Z[m, j]))
        T[j] = R.ajustar_temperatura(R.logit_binario(p0), Y[m, j])
        thr[j] = R.elegir_umbral_f1(Y[m, j], 1 / (1 + np.exp(-Z[m, j] / T[j])))
        ok[j] = True
    return dict(temperaturas=T, umbrales=thr, calibradas=ok)

### 3.4.28 `multitarea_es.py` (11/15): métricas multietiqueta, robustez ortográfica y etiquetas de la batería

`metricas_multietiqueta` (por cabeza y solo sobre filas etiquetadas), `evaluar_ortografia` (retención de F1 por tipo de perturbación) y las funciones que convierten la batería contrastiva en etiquetas por cabeza.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 9-10. EVALUACIÓN (multietiqueta enmascarada, robustez, contexto, A3, errores)
# ======================================================================================
def metricas_multietiqueta(Y, M, P, thr, nombres=HEADS) -> pd.DataFrame:
    """Por cabeza (solo filas etiquetadas): P/R/F1 de la clase positiva, F1 de la negativa, F1 macro de la cabeza,
    AUC y matriz de confusión. Última fila: macro entre cabezas evaluables. NO mezcla tareas en una sola cifra."""
    from sklearn.metrics import roc_auc_score
    filas = []
    for j, h in enumerate(nombres):
        m = M[:, j] & ~np.isnan(P[:, j])
        if m.sum() == 0 or len(np.unique(Y[m, j])) < 2:
            filas.append(dict(cabeza=h, n=int(m.sum()), nota="sin etiquetas suficientes en esta partición")); continue
        r = R.metricas_binarias(Y[m, j], P[m, j] >= thr[j])
        filas.append(dict(cabeza=h, n=r["n"], positivos=int(Y[m, j].sum()), precision=r["precision"], recall=r["recall"], f1=r["f1"],
                          f1_macro_cabeza=r["f1_macro"], auc=float(roc_auc_score(Y[m, j], P[m, j])), umbral=thr[j],
                          TN=r["tn"], FP=r["fp"], FN=r["fn"], TP=r["tp"]))
    d = pd.DataFrame(filas)
    ev = d[d.get("f1").notna()] if "f1" in d else d.iloc[0:0]
    if len(ev):
        d = pd.concat([d, pd.DataFrame([dict(cabeza="MACRO (cabezas evaluables)", n=int(ev.n.sum()), f1=ev.f1.mean(),
                                             precision=ev.precision.mean(), recall=ev.recall.mean(), auc=ev.auc.mean())])], ignore_index=True)
    return d


def evaluar_ortografia(pred: Predictor, df: pd.DataFrame, Y, M, tipos=None, seed=42, tta=True, col_ctx=None) -> pd.DataFrame:
    """Perturba el comentario (una familia de ofuscación por vez; las mismas filas en cada fila de la tabla) y mide F1
    por cabeza evaluable frente al texto original. Retención = F1_perturbado / F1_original."""
    tipos = tipos or [k for k in R.PERTURBACIONES if k not in ("abrev",)]
    ctx = df[col_ctx].tolist() if col_ctx else None
    base = metricas_multietiqueta(Y, M, pred.probs(df.text.tolist(), ctx, tta), pred.thr)
    f0 = base.set_index("cabeza").f1
    filas = []
    for tp in tipos:
        rng = np.random.default_rng(seed)
        tx = [R.perturbar_texto(t, rng, 0.5, [tp])[0] for t in df.text]
        m = metricas_multietiqueta(Y, M, pred.probs(tx, ctx, tta), pred.thr).set_index("cabeza").f1
        for h in HEADS:
            if h in m.index and not np.isnan(m[h]) and f0.get(h, np.nan) == f0.get(h, np.nan):
                filas.append(dict(perturbacion=tp, cabeza=h, f1_original=f0[h], f1_perturbado=m[h], retencion=m[h] / f0[h] if f0[h] else np.nan))
    return pd.DataFrame(filas)


def etiquetas_bateria(bat: pd.DataFrame):
    """Etiquetas de la batería v4 por cabeza, derivadas de `categoria` (PROPUESTAS por los autores, validado=False)."""
    return etiquetas_taxonomia(bat.categoria.tolist())


def evaluar_bateria_cabezas(pred: Predictor, bat: pd.DataFrame, tta=True) -> dict:
    Y, M = etiquetas_bateria(bat)
    P = pred.probs(bat.texto.tolist(), None, tta)
    return dict(metricas=metricas_multietiqueta(Y, M, P, pred.thr), P=P)

### 3.4.29 `multitarea_es.py` (12/15): pruebas contextuales y análisis de errores

Casos funcionales **sin y con contexto** (por ejemplo, `hp` amistoso frente a `hp` en discusión) y listado de falsos positivos/negativos más seguros.

In [ ]:
%%writefile -a multitarea_es.py
PRUEBAS_CONTEXTO = [
    # (caso, comentario, contexto_amistoso/neutral, contexto_hostil) — pruebas FUNCIONALES, no etiquetas de entrenamiento
    ("cultura", "Me encanta la cultura costeña", None),
    ("estereotipo", "Todos los costeños son perezosos", None),
    ("contradiscurso", "No está bien decir que todos los costeños son perezosos", None),
    ("hp_amistoso", "Usted sí es mucho hp, parce", "Conversación entre amigos: planeando el asado del domingo, todo en buena onda."),
    ("hp_discusion", "Usted es un hp", "Discusión acalorada: le reclama a otro usuario por una deuda y lo amenaza."),
    ("ofuscado_espaciado", "HI J U E P U T A", None),
    ("ofuscado_leet", "H1JUEPUTA", None),
    ("ofuscado_mascara", "hijuep*ta", None),
    ("ofuscado_caso", "prostituTa", None),
    ("abreviado", "prosti", None),
]


def evaluar_contexto(pred: Predictor, casos=PRUEBAS_CONTEXTO, tta=True) -> pd.DataFrame:
    """Probabilidades por cabeza SIN y CON contexto para cada caso; no hay etiqueta, se inspeccionan y se comparan."""
    t = [c[1] for c in casos]; ctx = [c[2] for c in casos]
    sin, con = pred.probs(t, None, tta), pred.probs(t, ctx, tta)
    filas = []
    for i, c in enumerate(casos):
        f = dict(caso=c[0], comentario=c[1], contexto=c[2] or "—")
        for j, h in enumerate(HEADS):
            f[f"{h}|sin_ctx"] = sin[i, j]; f[f"{h}|con_ctx"] = con[i, j]
        filas.append(f)
    return pd.DataFrame(filas)


def errores_fp_fn(df: pd.DataFrame, y, p, thr: float, k: int = 10, col="text"):
    y, p = np.asarray(y).astype(int), np.asarray(p)
    fp, fn = R.minar_errores(y, p, thr, k, k)
    return df.iloc[fp][[col]].assign(prob=p[fp]), df.iloc[fn][[col]].assign(prob=p[fn])

### 3.4.30 `multitarea_es.py` (13/15): comparación con A3 (`ProbA3` y *bootstrap* de ΔF1)

`ProbA3` carga el modelo original **solo para inferencia** (nunca se modifica) y `bootstrap_delta_f1` calcula la diferencia pareada de F1 con remuestreo por artículo.

In [ ]:
%%writefile -a multitarea_es.py
class ProbA3:
    """Modelo A3 ORIGINAL (clasificación binaria de odio). Solo inferencia; jamás se modifica ni se guarda."""

    def __init__(self, ruta_o_hub: str, device=None, max_len=128):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(ruta_o_hub)
        self.m = AutoModelForSequenceClassification.from_pretrained(ruta_o_hub).eval()
        self.dev = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.m.to(self.dev); self.max_len = max_len
        self.pre, self.pre_nombre = obtener_preprocesador()

    @torch.no_grad()
    def probs(self, textos, bs=64) -> np.ndarray:
        t = [self.pre(x) for x in textos]; orden = np.argsort([len(x) for x in t]); out = np.zeros(len(t), dtype=np.float32)
        for i in range(0, len(t), bs):
            ix = orden[i:i + bs]
            enc = self.tok([t[j] for j in ix], truncation=True, max_length=self.max_len, padding=True, return_tensors="pt")
            enc = {k: v.to(self.dev) for k, v in enc.items()}
            out[ix] = torch.softmax(self.m(**enc).logits.float(), -1)[:, 1].cpu().numpy()
        return out


def bootstrap_delta_f1(y, p_a, thr_a, p_b, thr_b, grupos, B=1000, seed=42) -> dict:
    """Diferencia pareada F1(b) − F1(a) con remuestreo por artículo (mismo remuestreo para ambos modelos)."""
    rng = np.random.default_rng(seed)
    y, pa, pb = np.asarray(y).astype(int), np.asarray(p_a) >= thr_a, np.asarray(p_b) >= thr_b
    g = np.asarray(grupos); ug = np.unique(g); idx = {k: np.where(g == k)[0] for k in ug}
    d = []
    for _ in range(B):
        sel = np.concatenate([idx[k] for k in rng.choice(ug, len(ug), replace=True)])
        d.append(R.metricas_binarias(y[sel], pb[sel])["f1"] - R.metricas_binarias(y[sel], pa[sel])["f1"])
    return dict(delta_medio=float(np.mean(d)), ic95=(float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))),
                prob_mejora=float(np.mean(np.array(d) > 0)))

### 3.4.31 `multitarea_es.py` (14/15): exportación para Gradio / Hugging Face Spaces

Plantilla de `app.py` y `exportar_app`, que guarda el modelo, copia los módulos y escribe `requirements.txt`.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# 12. EXPORTACIÓN (Gradio / HF Spaces)
# ======================================================================================
APP_GRADIO = '''# -*- coding: utf-8 -*-
import gradio as gr, pandas as pd
from multitarea_es import Predictor, HEADS
P = Predictor.cargar("modelo_v5")
def analizar(comentario, contexto):
    df = P.predecir([comentario], [contexto or None])
    filas = []
    for j, h in enumerate(HEADS):
        v = df.loc[0, f"p_{h}"]
        filas.append([h, "sin entrenar (faltan datos)" if P.activas[j] is False else round(float(v), 3),
                      "—" if not P.activas[j] else ("sí" if v >= P.thr[j] else "no"), round(P.thr[j], 3)])
    return pd.DataFrame(filas, columns=["categoría", "probabilidad", "señal", "umbral"])
demo = gr.Interface(analizar, [gr.Textbox(label="Comentario"), gr.Textbox(label="Contexto (opcional)")],
                    gr.Dataframe(label="Resultado"), title="Detector de toxicidad (v5)",
                    description="Las señales son probabilísticas y requieren revisión humana; no son un veredicto.")
if __name__ == "__main__":
    demo.launch()
'''


def exportar_app(pred: Predictor, destino="export_v5") -> Path:
    import shutil
    d = Path(destino); (d).mkdir(parents=True, exist_ok=True)
    pred.guardar(d / "modelo_v5")
    for f in ("multitarea_es.py", "robustez_es.py"):
        if Path(f).exists():
            shutil.copy(f, d / f)
    (d / "app.py").write_text(APP_GRADIO, encoding="utf-8")
    (d / "requirements.txt").write_text("torch\ntransformers\ngradio\nnumpy\npandas\nscikit-learn\nscipy\nemoji\npysentimiento\n", encoding="utf-8")
    return d

### 3.4.32 `multitarea_es.py` (15/15): utilidades de prueba técnica sin red

Datos **sintéticos** y un encoder diminuto sin preentrenar para verificar que el código corre con `SIN_RED=True`. Sus métricas no tienen valor científico.

In [ ]:
%%writefile -a multitarea_es.py
# ======================================================================================
# Utilidades de PRUEBA TÉCNICA sin red (encoder diminuto aleatorio + datos SINTÉTICOS)
# Sirven solo para verificar que el código corre; sus métricas no tienen valor científico.
# ======================================================================================
_PAL_NEUTRAS = "hoy mañana noticia gobierno ciudad precio partido concierto profesor clase tema gente país trabajo casa familia calle bus metro".split()
_PAL_GRUPO = "costeños paisas venezolanos cachacos migrantes indígenas".split()
_PAL_ODIO = "plaga basura ladrones perezosos asquerosos".split()
_PAL_INS = "idiota imbécil estúpido inútil hijueputa gonorrea".split()
_PAL_VULG = "mierda carajo hijueputa gonorrea".split()


def datos_sinteticos(n=600, seed=42, n_titulos=60) -> pd.DataFrame:
    """DataFrame con el esquema de piuba (id, text, title, context_tweet, HATEFUL) y texto generado al azar:
    HATEFUL=1 si la plantilla mezcla un grupo con un término despectivo. Solo para pruebas técnicas."""
    rng = np.random.default_rng(seed); filas = []
    for i in range(n):
        base = list(rng.choice(_PAL_NEUTRAS, rng.integers(4, 9)))
        odio = rng.random() < 0.18
        if odio:
            base += [str(rng.choice(_PAL_GRUPO)), "son", str(rng.choice(_PAL_ODIO))]
        elif rng.random() < 0.2:
            base += ["usted", "es", "un", str(rng.choice(_PAL_INS))]
        elif rng.random() < 0.2:
            base += [str(rng.choice(_PAL_VULG)), "qué", "día"]
        rng.shuffle(base)
        t = int(rng.integers(0, n_titulos))
        filas.append(dict(id=i, text=" ".join(base), title=f"titular {t} " + " ".join(rng.choice(_PAL_NEUTRAS, 5)),
                          context_tweet="", HATEFUL=int(odio)))
    return pd.DataFrame(filas)


def datos_offendes_sinteticos(n=300, seed=7) -> pd.DataFrame:
    rng = np.random.default_rng(seed); etq = rng.choice(["NO", "NOE", "OFP", "OFG"], n, p=[.4, .2, .25, .15])
    tx = [" ".join(rng.choice(_PAL_NEUTRAS + (_PAL_VULG if e == "NOE" else _PAL_INS if e == "OFP" else _PAL_GRUPO if e == "OFG" else []), 6)) for e in etq]
    return pd.DataFrame(dict(text=tx, label=etq))


def construir_encoder_minimo(textos: Sequence[str], carpeta, hidden=64, capas=2, max_pos=130):
    """BPE a nivel de byte entrenado localmente + RoBERTa diminuto SIN preentrenar. Devuelve (tokenizer, config)."""
    from tokenizers import ByteLevelBPETokenizer
    from transformers import RobertaConfig, RobertaTokenizerFast
    carpeta = Path(carpeta); carpeta.mkdir(parents=True, exist_ok=True)
    bpe = ByteLevelBPETokenizer(); bpe.train_from_iterator(list(textos), vocab_size=600, min_frequency=1,
                                                          special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"])
    bpe.save_model(str(carpeta))
    tok = RobertaTokenizerFast.from_pretrained(str(carpeta))      # (en transformers 5 `vocab_file=` crea un tokenizador vacío)
    assert len(tok) > 50, 'tokenizador vacío'
    cfg = RobertaConfig(vocab_size=len(tok), hidden_size=hidden, num_hidden_layers=capas, num_attention_heads=2,
                        intermediate_size=hidden * 2, max_position_embeddings=max_pos + 2, pad_token_id=tok.pad_token_id,
                        bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id, type_vocab_size=1)
    return tok, cfg

In [ ]:
import robustez_es as R, multitarea_es as MT
importlib.reload(R); importlib.reload(MT)
MT.fijar_semilla(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dispositivo:", DEVICE, "| cabezas:", MT.HEADS)

## 3.5 Carga de la partición por artículo
Si existen los `split_articulo_*.parquet` de la v4 se **reutilizan tal cual** (mismas filas, comparable con la v4); si no, se reconstruyen con el mismo procedimiento desde el Hub (como en la corrida de referencia).

In [ ]:
# 2.1 Cargar la partición por artículo
from sklearn.model_selection import GroupShuffleSplit

def particionar_por_articulo(full, key, seed=42):
    """Misma lógica que v4 §3.3: 20 % test por artículo; del resto, 20 % dev por artículo."""
    g = full[key].factorize()[0]
    i_rest, i_te = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(full, groups=g))
    rest = full.iloc[i_rest]
    i_tr, i_dv = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(rest, groups=g[i_rest]))
    return {"train": rest.iloc[i_tr], "dev": rest.iloc[i_dv], "test": full.iloc[i_te]}

SPLIT_DIR = Path(".")        # carpeta con split_articulo_{train,dev,test}.parquet de la v4 (opcional)
CATS = ["CALLS", "WOMEN", "LGBTI", "RACISM", "CLASS", "POLITICS", "DISABLED", "APPEARANCE", "CRIMINAL"]
if SIN_RED:
    SP = particionar_por_articulo(MT.datos_sinteticos(2500, n_titulos=250), "title"); ORIGEN = "SINTÉTICO"
elif all((SPLIT_DIR / f"split_articulo_{n}.parquet").exists() for n in ["train", "dev", "test"]):
    SP = {n: pd.read_parquet(SPLIT_DIR / f"split_articulo_{n}.parquet") for n in ["train", "dev", "test"]}; ORIGEN = "parquet v4"
else:
    from datasets import load_dataset
    ds = load_dataset("piuba-bigdata/contextualized_hate_speech")
    full = pd.concat([ds[k].to_pandas() for k in ["train", "dev", "test"]], ignore_index=True)
    SP = particionar_por_articulo(full, "body"); ORIGEN = "reconstruido desde el Hub (body)"
    for n, d in SP.items():                       # se guardan en la carpeta de ESTA corrida; no se toca nada de la v4
        d[[c for c in ["id", "text", "title", "context_tweet", "HATEFUL"] + CATS if c in d]].to_parquet(RUN / f"split_articulo_{n}.parquet", index=False)
print("origen de la partición:", ORIGEN)

## 3.6 Validación: formato, vacíos, duplicados y fuga

In [ ]:
# 2.2 Validación: columnas, vacíos, duplicados, balance y FUGA entre particiones
req = {"text", "HATEFUL", "title"} | ({COL_CTX} if USAR_CONTEXTO else set())
for n, d in SP.items():
    faltan = req - set(d.columns); assert not faltan, f"{n}: faltan columnas {faltan}"
    vac = int(d.text.isna().sum() + (d.text.astype(str).str.strip() == "").sum())
    print(f"{n:5s} filas={len(d):6d} | artículos(title)={d.title.nunique():5d} | vacíos={vac} | dup. texto={int(d.text.duplicated().sum())} | % HATEFUL={d.HATEFUL.mean()*100:.1f}")
    SP[n] = d[d.text.notna() & (d.text.astype(str).str.strip() != "")].reset_index(drop=True)
    assert set(SP[n].HATEFUL.unique()) <= {0, 1}
col_g = "body" if all("body" in d for d in SP.values()) else "title"
fuga = R.verificar_fuga(SP, col_grupo=col_g)
print("fuga por", col_g, "y texto duplicado entre particiones:", fuga)
assert all(v == 0 for k, v in fuga.items() if k.endswith(col_g)), "¡un artículo aparece en dos particiones!"
CHECKS["datasets cargados y sin fuga por artículo"] = True

## 3.7 Fuentes de etiquetas adicionales y batería
OffendES se carga con **sus** particiones oficiales, se deduplica (también contra piuba y la batería, para evitar fugas) y sus filas de `test` quedan reservadas como TEST de las cabezas de vulgaridad e insulto.

In [ ]:
# 2.3 Fuentes opcionales (apagadas por defecto) + batería
import re, unicodedata
BAT = R.construir_bateria()
OFF = None     # dict(train, dev, test) con columnas texto/etiqueta cuando está activa

def cargar_offendes():
    """OffendES (SINAI): etiquetas NO/NOE/OFP/OFG. Usa SUS particiones oficiales; deduplica contra piuba y la batería."""
    if SIN_RED:
        d = MT.datos_offendes_sinteticos(900); ct, ce = "text", "label"
        partes = {"train": d.iloc[:600], "dev": d.iloc[600:750], "test": d.iloc[750:]}
    else:
        from datasets import load_dataset
        of = load_dataset("SINAI/OffendES", token=HF_TOKEN); print(of)       # falla si no se aceptaron las condiciones (dataset protegido)
        print("licencia declarada:", getattr(of[list(of.keys())[0]].info, "license", None), "→ la ficha se contradice: verifícala antes de redistribuir.")
        if COL_TEXTO_OFF is None or COL_ETIQ_OFF is None:
            raise RuntimeError("Fija COL_TEXTO_OFF y COL_ETIQ_OFF según las columnas impresas arriba.")
        ct, ce = COL_TEXTO_OFF, COL_ETIQ_OFF
        mapa = {"train": "train", "dev": "validation" if "validation" in of else "dev", "test": "test"}
        assert all(v in of for v in mapa.values()), f"splits de OffendES no esperados: {list(of.keys())}"
        partes = {k: of[v].to_pandas()[[ct, ce]].dropna() for k, v in mapa.items()}
    norm = lambda t: R.normalizar_ofuscacion(t)
    prohibidos = set(map(norm, pd.concat([d.text for d in SP.values()]))) | set(map(norm, pd.concat(BAT.values()).texto))
    out = {}
    for k, d in partes.items():
        d = d.rename(columns={ct: "text", ce: "label"}).assign(_n=lambda x: x.text.map(norm)).drop_duplicates("_n")
        out[k] = d[~d._n.isin(prohibidos)].drop(columns="_n").reset_index(drop=True)
        prohibidos |= set(map(norm, out[k].text)) if k == "train" else set()    # sin repetidos entre train y el resto
    print({k: (len(v), v.label.value_counts().to_dict()) for k, v in out.items()})
    return out

if USAR_OFFENDES: OFF = cargar_offendes()

ANOT = None
if RUTA_ANOTADOS:
    ANOT = pd.read_csv(RUTA_ANOTADOS)
    assert "particion" in ANOT and set(ANOT.particion) <= {"train", "dev", "test"}, "el CSV anotado debe traer `particion` ∈ train/dev/test (separada por familia, sin fuga)"
    print("anotados:", ANOT.particion.value_counts().to_dict())
print("batería:", {k: len(v) for k, v in BAT.items()}, "| OffendES:", "activo" if OFF else "no", "| anotados:", "sí" if ANOT is not None else "no")

## 3.8 Balance de clases y tipos de odio
`HATEFUL` de piuba y las etiquetas de OffendES (`NO`, `NOE`, `OFP`, `OFG`) tienen balances muy distintos; esto condiciona los `pos_weight` por cabeza (sección 4.2).

In [ ]:
# 3.8 Balance por partición y tipos de odio (piuba) + distribución de etiquetas de OffendES
for k, d in SP.items():
    print(f"{k:5s} % HATEFUL = {d.HATEFUL.mean()*100:.1f} | desbalance neg/pos ≈ 1 a {(d.HATEFUL==0).sum()/max((d.HATEFUL==1).sum(),1):.1f}")
if all(c in SP["train"] for c in CATS):
    print("\nTipos de odio en train (proporción dentro de HATEFUL=1; multietiqueta):")
    print(SP["train"].loc[SP["train"].HATEFUL == 1, CATS].mean().sort_values(ascending=False).round(3))
else:
    print("\n(la partición cargada no trae las categorías de odio)")
if OFF:
    print("\nOffendES — filas por etiqueta:"); print(pd.DataFrame({k: v.label.value_counts() for k, v in OFF.items()}).fillna(0).astype(int))
    print("\nOffendES — % por etiqueta:"); print((pd.DataFrame({k: v.label.value_counts(normalize=True) for k, v in OFF.items()}) * 100).round(1))

## 3.9 Longitud en palabras y tokens
El límite es 128 tokens. Se mide con el preprocesamiento y el tokenizador reales del modelo, para el comentario solo y para comentario + título (el par que recibe el modelo).

In [ ]:
# 3.9 Longitud en palabras y tokens (piuba y OffendES)
from transformers import AutoTokenizer
_pre, _ = MT.obtener_preprocesador()
tr = SP["train"].assign(n_words=lambda d: d.text.str.split().str.len())
print("piuba (train) — palabras por comentario:"); print(tr.n_words.describe().round(1)); print("media por clase:", tr.groupby("HATEFUL").n_words.mean().round(2).to_dict())
if OFF:
    print("\nOffendES (train) — palabras por etiqueta:"); print(OFF["train"].assign(n_words=lambda d: d.text.str.split().str.len()).groupby("label").n_words.describe().round(1))
if not SIN_RED:
    _tok = AutoTokenizer.from_pretrained(ENCODER)
    m = tr.sample(min(2000, len(tr)), random_state=0); a = [_pre(t) for t in m.text]; b = [_pre(t) for t in m.title]
    l_txt = np.array([len(x) for x in _tok(a)["input_ids"]]); l_par = np.array([len(x) for x in _tok(a, b)["input_ids"]])
    print("\npiuba — solo comentario:   tokens p50/p95/max =", np.percentile(l_txt, [50, 95, 100]), f"| >128: {(l_txt > 128).mean()*100:.2f} %")
    print("piuba — comentario+título: tokens p50/p95/max =", np.percentile(l_par, [50, 95, 100]), f"| >128: {(l_par > 128).mean()*100:.2f} %")
    if OFF:
        l_of = np.array([len(x) for x in _tok([_pre(t) for t in OFF["train"].text.sample(min(2000, len(OFF["train"])), random_state=0)])["input_ids"]])
        print("OffendES — solo comentario: tokens p50/p95/max =", np.percentile(l_of, [50, 95, 100]), f"| >128: {(l_of > 128).mean()*100:.2f} %")
else:
    print("\n(SIN_RED: sin tokenizador real; se omite la longitud en tokens)")

## 3.10 Muestras representativas, ambiguas y difíciles
Se leen ejemplos de cada clase (no solo se cuentan). La búsqueda de “groserías” es una **heurística de exploración**, no una etiqueta: sirve para ver cuánto odio hay sin groserías y cuánta grosería hay sin odio.

In [ ]:
# 3.10 Muestras (leerlas, no solo mostrarlas)
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 250)
tr = SP["train"]
print("piuba HATEFUL=1"); display(tr[tr.HATEFUL == 1].sample(8, random_state=1)[["text"]])
print("piuba HATEFUL=0"); display(tr[tr.HATEFUL == 0].sample(8, random_state=1)[["text"]])
neg = tr.text.str.contains(r"\b(?:no|nunca|ni|jamás)\b", case=False)
insulto = tr.text.str.contains(r"\b(?:puto|puta|idiota|imb[eé]cil|pelotudo|boludo|mierda)\b", case=False)
print("En train: con negación", int(neg.sum()), "| con grosería heurística", int(insulto.sum()))
print("odio SIN grosería:", int(((tr.HATEFUL == 1) & ~insulto).sum()), "| grosería SIN odio:", int(((tr.HATEFUL == 0) & insulto).sum()))
print("Ejemplos de odio sin grosería:"); display(tr[(tr.HATEFUL == 1) & ~insulto].sample(4, random_state=2)[["text"]])
print("Ejemplos de grosería sin odio:"); display(tr[(tr.HATEFUL == 0) & insulto].sample(4, random_state=2)[["text"]])
if OFF:
    for et in ["NO", "NOE", "OFP", "OFG"]:
        d_ = OFF["train"][OFF["train"].label == et]
        if len(d_): print(f"OffendES {et} ({len(d_)} filas)"); display(d_.sample(min(4, len(d_)), random_state=0)[["text"]])

## 3.11 ¿Basta una lista de palabras?
Regla ingenua evaluada en DEV (evidencia para la Fase 0): se marca como odio todo comentario con alguna palabra de una lista ad hoc de insultos frecuentes. Además se mide, en OffendES, qué fracción de cada etiqueta contiene alguna palabra de una lista ampliada (vulgar y colombiana / mexicana): si las etiquetas `OFP` (insulto) u `OFG` no contienen esas palabras, una lista no puede ser la solución; si `NO` las contiene, tampoco.

In [ ]:
# 3.11 Regla léxica (la lista es solo evidencia exploratoria; el modelo NO usa listas para decidir)
import re
from sklearn.metrics import precision_score, recall_score, f1_score
GROSERIAS = ["mierda", "puto", "puta", "boludo", "pelotudo", "idiota", "imbecil", "imbécil", "estupido", "estúpido", "basura",
             "forro", "negro de mierda", "kuka", "zurdo", "gorila", "chorro", "vago", "asco", "mamerto"]
patron = re.compile(r"\b(?:" + "|".join(map(re.escape, GROSERIAS)) + r")\b", re.IGNORECASE)
dv = SP["dev"]; pred = dv.text.str.contains(patron).astype(int)
print("regla (piuba, DEV)  P=%.3f R=%.3f F1=%.3f" % (precision_score(dv.HATEFUL, pred), recall_score(dv.HATEFUL, pred), f1_score(dv.HATEFUL, pred)))
print("odio SIN palabra de la lista:", int(((dv.HATEFUL == 1) & (pred == 0)).sum()), "| palabra de la lista SIN odio:", int(((dv.HATEFUL == 0) & (pred == 1)).sum()))
display(dv[(dv.HATEFUL == 1) & (pred == 0)].sample(5, random_state=0)[["text"]]); display(dv[(dv.HATEFUL == 0) & (pred == 1)].sample(5, random_state=0)[["text"]])
if OFF:
    EXT = GROSERIAS + ["hijueputa", "hijo de puta", "hijoputa", "hdp", "carajo", "joder", "coño", "gonorrea", "malparido", "marica", "perra",
                       "cabrón", "cabron", "pendejo", "verga", "culero", "chingar", "mrd", "hp"]
    pat2 = re.compile(r"\b(?:" + "|".join(map(re.escape, EXT)) + r")\b", re.IGNORECASE)
    o = OFF["train"].assign(contiene_lista=lambda d: d.text.str.contains(pat2))
    print("\nOffendES — % de comentarios con alguna palabra de la lista ampliada, por etiqueta:")
    print((o.groupby("label").contiene_lista.mean() * 100).round(1))

## 3.12 Mayúsculas, emojis y formas alteradas
Se mide antes de decidir si el preprocesamiento debe borrar o conservar estas señales, y **cuántos textos de entrenamiento ya contienen formas alteradas** (espaciado, *leet*, máscaras, alargamientos) que el normalizador de la v4 modifica. Esto dimensiona cuánto pesa la aumentación sintética de la sección 4.3.

In [ ]:
# 3.12 Mayúsculas / emojis por clase + prevalencia de formas alteradas en los datos de entrenamiento
EMOJI = re.compile("[\U0001F300-\U0001FAFF☀-➿]")
def prop_mayus(t):
    w = re.findall(r"[^\W\d_]{3,}", t)
    return np.mean([x.isupper() for x in w]) if w else 0.0
d = SP["train"]
print("piuba — prop. de palabras en MAYÚSCULAS por clase:", d.text.map(prop_mayus).groupby(d.HATEFUL).mean().round(3).to_dict())
print("piuba — % con emoji por clase:", d.text.str.contains(EMOJI).groupby(d.HATEFUL).mean().round(3).to_dict())
print("piuba — % con URL:", round(d.text.str.contains(r"https?://").mean() * 100, 1), "| % con @usuario:", round(d.text.str.contains("@usuario").mean() * 100, 1))
if OFF:
    o = OFF["train"]
    print("OffendES — prop. de MAYÚSCULAS por etiqueta:", o.text.map(prop_mayus).groupby(o.label).mean().round(3).to_dict())
    print("OffendES — % con emoji por etiqueta:", o.text.str.contains(EMOJI).groupby(o.label).mean().round(3).to_dict())
cambia = lambda t: R.normalizar_ofuscacion(t, minusculas=False) != t
RE_HP = re.compile(r"\b(?:hp|hdp|hijueputa\w*|hijoputa\w*|hijo de puta)\b", re.IGNORECASE)
for nombre, ser in [("piuba", d.text), *([("OffendES", OFF["train"].text)] if OFF else [])]:
    mu = ser.sample(min(5000, len(ser)), random_state=0)
    print(f"{nombre}: % de textos que el normalizador modifica = {mu.map(cambia).mean()*100:.1f} | % con hp / hijueputa / hdp = {ser.str.contains(RE_HP).mean()*100:.2f}")
print("\nEjemplos que el normalizador modifica (piuba):")
_m = d.text.sample(min(5000, len(d)), random_state=1); _m = _m[_m.map(cambia)].head(6)
display(pd.DataFrame({"original": _m.values, "normalizado": [R.normalizar_ofuscacion(t, minusculas=False) for t in _m]}))

## 3.13 Duplicados y ruido de etiquetas
Un mismo texto con etiquetas contradictorias indica ruido de anotación (techo de desempeño) y obliga a vigilar fugas por copia entre particiones.

In [ ]:
# 3.13 Duplicados dentro de cada partición y textos repetidos con etiquetas contradictorias
for k, d_ in SP.items():
    print(f"{k:5s} textos duplicados (filas repetidas) = {int(d_.text.duplicated(keep=False).sum())} de {len(d_)}")
g = SP["train"].groupby("text").HATEFUL.agg(["nunique", "size"])
print("\npiuba (train): textos repetidos =", int((g["size"] > 1).sum()), "| de ellos con etiquetas contradictorias =", int(((g["size"] > 1) & (g["nunique"] > 1)).sum()))
if OFF:
    g2 = OFF["train"].groupby("text").label.agg(["nunique", "size"])
    print("OffendES (train, ya deduplicado por texto normalizado): repetidos =", int((g2["size"] > 1).sum()), "| contradictorios =", int(((g2["size"] > 1) & (g2["nunique"] > 1)).sum()))

## 3.14 Particiones oficiales de piuba: calidad, balance y tipos de odio
Antes de partir por artículo conviene leer las **particiones oficiales** del corpus. Esta celda carga las tres (descarga desde el Hub; se omite con `SIN_RED=True`) y mide, para cada una: forma, textos vacíos, duplicados exactos y % de `HATEFUL`. Además cuenta qué **tipos de odio** (quién es agredido) son más frecuentes entre los comentarios con odio.

In [ ]:
# 3.14 Particiones OFICIALES de piuba: inventario, calidad, balance y tipos de odio
if SIN_RED:
    OF = None
    print("SIN_RED: no hay acceso al Hub; se omite el análisis de las particiones oficiales.")
else:
    from datasets import load_dataset
    _ds = load_dataset("piuba-bigdata/contextualized_hate_speech")
    OF = {k: _ds[k].to_pandas() for k in ["train", "dev", "test"]}
    for k, d_ in OF.items():
        vac = int(d_.text.isna().sum() + (d_.text.str.strip() == "").sum())
        print(f"{k:5s} {d_.shape} | vacíos en text: {vac} | duplicados exactos de text: {int(d_.text.duplicated().sum())} "
              f"| % HATEFUL: {d_.HATEFUL.mean()*100:.1f}")
    print("\nTipos de odio en train oficial (proporción dentro de HATEFUL=1; multietiqueta):")
    print(OF["train"].loc[OF["train"].HATEFUL == 1, CATS].mean().sort_values(ascending=False).round(3))

**Resultado de la corrida de referencia (cuaderno original):**

| Partición oficial | Filas | Duplicados exactos de `text` | % `HATEFUL` |
|---|---:|---:|---:|
| train | 36.420 | 1.206 | 15,2 % |
| dev | 9.106 | 109 | 15,2 % |
| test | 11.343 | 291 | 15,8 % |

Sin textos vacíos. Entre los comentarios con odio, las categorías más frecuentes son **racismo (28,2 %)**, **apariencia (20,9 %)** y **criminales (18,6 %)**; siguen llamados a la acción (16,5 %), política (16,5 %), mujeres (15,2 %), clase (9,8 %), LGBTI (8,9 %) y discapacidad (6,6 %). Es decir, el odio apunta a grupos concretos, no solo a «palabras feas».

## 3.15 Fuga por artículo en las particiones oficiales
Un mismo artículo (`body`) genera decenas de comentarios. Si las mismas noticias aparecen en train y en la partición de evaluación, el modelo puede aprender **el tema** en lugar del comentario y la evaluación sale inflada. La celda mide qué fracción de dev y test ya vio su artículo en train y cuánto varía la tasa de odio entre artículos.

In [ ]:
# 3.15 ¿Las mismas noticias aparecen en train y en dev/test oficiales?
if OF is None:
    print("SIN_RED: se omite.")
else:
    key = "body"      # identifica el artículo; title tiene menos valores únicos que body
    arts = {k: set(d_[key]) for k, d_ in OF.items()}
    print("artículos únicos train/dev/test:", len(arts["train"]), len(arts["dev"]), len(arts["test"]))
    print("test con artículo visto en train:", round(OF["test"][key].isin(arts["train"]).mean(), 3))
    print("dev con artículo visto en train:", round(OF["dev"][key].isin(arts["train"]).mean(), 3))
    # ¿El odio depende del artículo? Si sí, el modelo puede aprender el tema en vez del comentario.
    tasa = OF["train"].groupby(key).HATEFUL.mean()
    print("tasa de odio por artículo (train):", tasa.describe().round(3).to_dict())

**Resultado (corrida de referencia):** 990 / 989 / 248 artículos únicos en train / dev / test. El **test oficial no comparte artículos con train (0 %)**, pero el **dev oficial sí (100 %)**: no mide generalización a noticias nuevas. Además la tasa de odio varía mucho entre artículos (media 0,155, desviación 0,139, máximo 0,914), así que un modelo podría aprender el tema.

## 3.16 Reconstrucción de la partición por artículo
Decisión: la evaluación principal usa una partición **por artículo** (`GroupShuffleSplit`, semilla 42, 20 % test y 20 % del resto como dev), de modo que ningún artículo cruza particiones. La celda vuelve a construirla desde las particiones oficiales y comprueba (a) que no hay fuga y (b) que coincide en tamaño con la partición `SP` que usa todo el cuaderno.

In [ ]:
# 3.16 Reconstruir la partición por artículo desde las particiones oficiales y compararla con SP
if OF is None:
    print("SIN_RED: se omite.")
else:
    full_of = pd.concat([OF[k].assign(split_oficial=k) for k in OF], ignore_index=True)
    REC = particionar_por_articulo(full_of, "body")
    for n, d_ in REC.items():
        print(f"{n:5s} {len(d_):6d} filas | artículos: {d_.body.nunique():4d} | % hateful: {d_.HATEFUL.mean()*100:.1f}")
    assert not (set(REC["train"].body) & set(REC["test"].body)) and not (set(REC["train"].body) & set(REC["dev"].body)) \
        and not (set(REC["dev"].body) & set(REC["test"].body)), "¡un artículo aparece en dos particiones!"
    print("coincide en tamaño con SP:", {n: len(REC[n]) == len(SP[n]) for n in REC})
    CHECKS["partición por artículo reconstruida sin fuga"] = True

**Resultado (corrida de referencia):** train 36.587 filas (792 artículos, 15,4 % hateful) · dev 8.971 (198 artículos, 14,0 %) · test 11.311 (248 artículos, 16,2 %). Con esta partición **los modelos de referencia (A1, A2, A3, L) y el modelo multitarea se miden sobre exactamente las mismas filas**.

**Conclusiones del EDA y de la validación** (las marcadas con † provienen del EDA del cuaderno original sobre el mismo corpus piuba y se **recalculan** en las celdas 3.8–3.13; las demás se verificaron al ejecutar la corrida completa de la v5):
- Sin textos vacíos en ninguna partición; hay duplicados exactos dentro de cada una (train 1.247, dev 279, test 421).
- **Sin fuga por artículo:** train∩dev, train∩test y dev∩test = 0 artículos (`body`).
- **Hallazgo nuevo:** aun con artículos disjuntos hay textos exactamente repetidos entre particiones (train∩dev 134, train∩test 167, dev∩test 67). Son comentarios cortos y genéricos que se repiten entre noticias; se declaran como limitación y no se eliminaron para conservar las mismas filas que la v4.
- Balance de `HATEFUL` ≈ 14–16 % en las tres particiones (desbalance ≈ 1 a 5,5). † Los tipos de odio más frecuentes son racismo, apariencia y criminales.
- † **Longitud corta:** mediana ≈ 11 palabras, sin diferencia de longitud entre clases (14,7 palabras de media en ambas); en tokens p95 ≈ 54 y máximo ≈ 106, con comentario + título p95 ≈ 77 y solo ≈ 0,08 % de truncados a 128.
- † **Una lista de groserías no basta:** en DEV la regla obtiene P ≈ 0,285, R ≈ 0,151 y F1 ≈ 0,197; hay ≈ 1.178 comentarios con odio sin ninguna palabra de la lista y ≈ 524 con una palabra de la lista y sin odio. Es la evidencia de que el modelo no puede apoyarse en léxico.
- † **Mayúsculas y emojis:** diferencia pequeña entre clases (mayúsculas 0,033 frente a 0,037; emojis 9,1 % frente a 7,3 %); no justifican un preprocesamiento agresivo. `@usuario` aparece en ≈ 100 % del texto (el corpus ya está anonimizado).
- **OffendES (tras deduplicar):** train 16.697 (`NO` 13.211 · `OFP` 2.041 · `NOE` 1.233 · `OFG` 212), validation 99, test 13.563. La partición `validation` oficial es **diminuta** (99 filas): condiciona la validación de las cabezas nuevas (ver 4.2 y 7). Las celdas 3.8–3.13 y 3.14–3.16 imprimen además sus longitudes, ejemplos por etiqueta, la cobertura de la lista ampliada de groserías por etiqueta, la prevalencia de formas alteradas y los textos contradictorios; **esas cifras deben leerse en la salida** antes de citarlas.
- La licencia declarada de OffendES se contradice en su ficha: debe verificarse antes de redistribuir el modelo o los datos.

# 4. Preparación de etiquetas y representación de los datos
Todas las decisiones se toman con **train** (lo que se aprende) y **DEV** (lo que se elige). El TEST no se mira. DEV se parte por artículo en dos mitades disjuntas: **DEV_CKPT** (elige el checkpoint) y **DEV_EVAL** (calibra temperaturas y umbrales, evalúa y compara). Así la partición que se usa para decidir no es la misma que se usa para reportar.

## 4.1 Etiquetas con máscara
La salida es una matriz **Y (N×3)** y una máscara **M (N×3)**: `M = 1` solo cuando la fuente *define* esa etiqueta.

| Fuente | vulgaridad | insulto | odio grupal | Justificación |
|---|:-:|:-:|:-:|---|
| piuba `HATEFUL = 1 / 0` | — | — | 1 / 0 | `HATEFUL` es odio contra grupos. `HATEFUL = 0` **no** dice “sin insulto”: los insultos personales aparecen como no-hateful (hallazgo D1 de la v4). |
| OffendES `NO` | 0 | 0 | 0 | No ofensivo |
| OffendES `NOE` | 1 | 0 | 0 | No ofensivo, pero con lenguaje vulgar |
| OffendES `OFP` | — | 1 | — | Ofensivo contra persona (no dice si hay vulgaridad ni grupo) |
| OffendES `OFG` | — | — | — | Se equipararía a odio grupal solo si `OFG_COMO_GRUPO = True`, tras verificar su guía de anotación; mientras tanto no aporta etiquetas |
| Anotación humana (CSV, opcional) | columnas explícitas o taxonomía v4 | | | `ambiguo` y vacíos → ausente |

Con **solo piuba**, únicamente `odio_grupal` tendría supervisión; con OffendES, `vulgaridad` e `insulto` también.

In [ ]:
# 3.1 Particiones de trabajo + matrices Y/M. DEV se parte POR ARTÍCULO en DEV_CKPT (elegir checkpoint) y DEV_EVAL (calibrar/comparar).
def sub_grupos(df, n, col="title", seed=SEED):
    if n is None or len(df) <= n: return df
    orden = df[col].drop_duplicates().sample(frac=1, random_state=seed); cnt = df[col].value_counts(); keep, tot = [], 0
    for t in orden:
        keep.append(t); tot += cnt[t]
        if tot >= n: break
    return df[df[col].isin(keep)]

dev_ckpt_p, dev_eval_p = R.partir_dev_por_grupo(SP["dev"], "title", 0.5, SEED)      # (mitad para checkpoint, mitad para evaluar)
assert not (set(dev_ckpt_p.title) & set(dev_eval_p.title))
PIUBA = {"train": sub_grupos(SP["train"], PARAMS["n_train"]), "dev_ckpt": sub_grupos(dev_ckpt_p, PARAMS["n_dev"]),
         "dev_eval": sub_grupos(dev_eval_p, PARAMS["n_dev"]), "test": SP["test"]}

def bloque(piuba, off=None, anot=None):
    """Concatena fuentes → (DataFrame[text, ctx, grupo, fuente], Y, M). Cada fuente aporta SOLO sus etiquetas compatibles."""
    dfs, Ys, Ms = [], [], []
    if piuba is not None and len(piuba):
        Y, M = MT.etiquetas_piuba(piuba.HATEFUL)
        dfs.append(pd.DataFrame(dict(text=piuba.text.values, ctx=(piuba[COL_CTX].values if USAR_CONTEXTO else None), grupo=piuba.title.values, fuente="piuba", HATEFUL=piuba.HATEFUL.values)))
        Ys.append(Y); Ms.append(M)
    if off is not None and len(off):
        Y, M = MT.etiquetas_offendes(off.label.tolist(), OFG_COMO_GRUPO)
        dfs.append(pd.DataFrame(dict(text=off.text.values, ctx=None, grupo=[f"off_{i}" for i in range(len(off))], fuente="offendes", HATEFUL=np.nan)))
        Ys.append(Y); Ms.append(M)
    if anot is not None and len(anot):
        Y, M = MT.etiquetas_anotadas(anot)
        cc = anot["contexto"].where(anot["contexto"].notna(), None).values if "contexto" in anot else None
        dfs.append(pd.DataFrame(dict(text=anot.texto.values if "texto" in anot else anot.text.values, ctx=cc, grupo=anot["familia"].values if "familia" in anot else [f"an_{i}" for i in range(len(anot))], fuente="anotado", HATEFUL=np.nan)))
        Ys.append(Y); Ms.append(M)
    return pd.concat(dfs, ignore_index=True), np.vstack(Ys), np.vstack(Ms)

def mitad(d, k):    # OffendES dev: mitad para checkpoint, mitad para evaluar (por filas; no hay artículo)
    if d is None: return None
    i = np.random.default_rng(SEED).permutation(len(d)); h = len(d) // 2
    return d.iloc[i[:h]] if k == 0 else d.iloc[i[h:]]
A = (lambda p: ANOT[ANOT.particion == p] if ANOT is not None else None)
BL = {"train": bloque(PIUBA["train"], OFF["train"] if OFF else None, A("train")),
      "dev_ckpt": bloque(PIUBA["dev_ckpt"], mitad(OFF["dev"], 0) if OFF else None, mitad(A("dev"), 0)),
      "dev_eval": bloque(PIUBA["dev_eval"], mitad(OFF["dev"], 1) if OFF else None, mitad(A("dev"), 1)),
      "test": bloque(PIUBA["test"], OFF["test"] if OFF else None, A("test"))}
for k, (d, Y, M) in BL.items():
    assert Y.shape == M.shape == (len(d), 3) and Y.dtype == np.float32 and M.dtype == bool and set(np.unique(Y[M])) <= {0.0, 1.0}
    print(f"{k:9s} N={len(d):6d} | etiquetas por cabeza {dict(zip(MT.HEADS, M.sum(0).tolist()))}")
CHECKS["etiquetas con formato esperado (Y float32 {0,1}, M bool, N×3)"] = True

In [ ]:
# 3.2 Auditoría: qué cabezas tienen datos para entrenarse
d_tr, Y_tr, M_tr = BL["train"]
AUDIT = MT.auditoria_etiquetas(Y_tr, M_tr, d_tr.fuente.values, min_pos=50, min_neg=50)
display(AUDIT) if "display" in dir() else print(AUDIT)
ACTIVAS = MT.cabezas_activas(Y_tr, M_tr, 50, 50); PESO_POS = MT.pesos_positivos(Y_tr, M_tr, 10.0)
for h, a, w in zip(MT.HEADS, ACTIVAS, PESO_POS):
    print(f"  {h:12s} {'ENTRENABLE' if a else 'SIN DATOS SUFICIENTES — cabeza implementada, no se entrena'} | pos_weight={w:.2f}")
NECESITA = {"vulgaridad": "ejemplos con vulgaridad vs. sin ella (OffendES NOE/NO, o CSV anotado con y_vulgaridad)",
            "insulto": "ejemplos de ataque personal vs. no (OffendES OFP/NO, o CSV anotado con y_insulto)",
            "odio_grupal": "piuba HATEFUL (ya incluido)"}
for h, a in zip(MT.HEADS, ACTIVAS):
    if not a: print(f"⚠ {h}: faltan datos → {NECESITA[h]}. Mínimo 50 positivos y 50 negativos etiquetados en train.")
assert any(ACTIVAS), "ninguna cabeza tiene datos: revisa las fuentes"

## 4.2 Qué entrena cada cabeza (resultado de la auditoría)

| Cabeza | Etiquetas en train | Positivos | Negativos | `pos_weight` | Fuentes |
|---|---:|---:|---:|---:|---|
| vulgaridad | 14.444 | 1.233 | 13.211 | 10,00 (tope) | OffendES (`NOE` frente a `NO`) |
| insulto | 16.485 | 2.041 | 14.444 | 7,08 | OffendES (`OFP` frente a `NO` + `NOE`) |
| odio_grupal | 51.031 | 5.624 | 45.407 | 8,07 | piuba (36.587) + OffendES `NO`/`NOE` como negativos (14.444) |

Total de train: **53.284** filas (36.587 piuba + 16.697 OffendES). En **DEV_CKPT / DEV_EVAL** hay 4.540 / 4.530 filas, pero solo **37 / 37** etiquetas de vulgaridad y **47 / 48** de insulto (provienen de la partición `validation` de OffendES, de 99 filas). Las cabezas nuevas se validan, entonces, con muy pocos ejemplos. En TEST: 11.041 etiquetas de vulgaridad, 13.352 de insulto y 22.352 de odio grupal.

## 4.3 Preprocesamiento y aumentación de textos
Dos capas, ambas **controladas** y usadas igual en entrenamiento e inferencia:
1. **Preprocesamiento del modelo:** `preprocess_tweet(lang='es')` de pysentimiento (usuarios → `@usuario`, URLs, emojis → texto, alargamientos). El tokenizador ya pasa a minúsculas; no se limpia dos veces.
2. **Robustez ortográfica:**
   - *Aumentación en línea:* con probabilidad `p_aug = 0,30` se perturba el comentario (espaciado, *leet*, máscaras `*`, homoglifos, mayúsculas, alargamientos, errores de teclado, `hijueputa → hp` solo desde la forma canónica…) con **la misma tasa en todas las filas**, para que el modelo no aprenda “texto perturbado = odio”.
   - *Vista normalizada:* `R.normalizar_ofuscacion` revierte ofuscaciones **sin expandir abreviaturas ambiguas** (`hp` queda `hp`). Se usa como vista extra en la inferencia (TTA) y ocasionalmente (`p_vista_norm = 0,10`) en entrenamiento.
   - *Contexto opcional:* con `p_ctx_drop = 0,30` se elimina el contexto al azar para que el modelo funcione sin él.

El normalizador **no clasifica**: ninguna palabra se marca por aparecer en una lista.

In [ ]:
# 4.1 Qué hace el normalizador y las perturbaciones (los ejemplos de la tarea)
PRE, PRE_NOMBRE = MT.obtener_preprocesador(); print("preprocesamiento:", PRE_NOMBRE)
ej = ["HI J U E P U T A", "H1JUEPUTA", "hijuep*ta", "prostituTa", "prosti", "hp", "Usted sí es mucho hp, parce", "mrd qué hijueputaaaa día", "eres un h.i.j.o.d.e.p.u.t.a"]
print(pd.DataFrame(dict(original=ej, normalizado=[R.normalizar_ofuscacion(t) for t in ej], vistas=[len(R.vistas_texto(t)) for t in ej])).to_string(index=False))
rng = np.random.default_rng(0)
print("\nperturbaciones sobre «Eres una prostituta, hijueputa»:")
for tp in R.PERTURBACIONES: print(f"  {tp:11s}", R.perturbar_texto("Eres una prostituta, hijueputa", rng, 0.9, [tp])[0])
assert R.normalizar_ofuscacion("hp") == "hp"       # nunca se expande una abreviatura ambigua

In [ ]:
# 4.2 Datasets con aumentación en línea (solo train). Validación/evaluación van SIN aumentar.
AUG = dict(p_aug=0.30, p_vista_norm=0.10, p_ctx_drop=0.30 if USAR_CONTEXTO else 0.0)     # p_ctx_drop: el modelo aprende a funcionar sin contexto
d_tr, Y_tr, M_tr = BL["train"]
DS_TRAIN = MT.DatosComentarios(d_tr.text, d_tr.ctx, Y_tr, M_tr, PRE, seed=SEED, **AUG)
muestra = [DS_TRAIN[i] for i in np.random.default_rng(1).choice(len(DS_TRAIN), 4, replace=False)]
for t, c, y, m in muestra: print(repr(t)[:90], "| ctx:", (repr(c)[:40] if c else None), "| y:", y.tolist(), "| m:", m.astype(int).tolist())
VAL = lambda k: dict(textos=BL[k][0].text.tolist(), contextos=BL[k][0].ctx.tolist() if USAR_CONTEXTO else None, Y=BL[k][1], M=BL[k][2])

## 4.4 Tokenización con soporte contextual
Entrada `<s> comentario </s></s> contexto </s>` (par de segmentos) con `truncation="only_second"`: **solo se recorta el contexto**. Un comentario más largo que `max_len − 24` se acorta antes para que `only_second` siempre sea viable. Sin contexto, el segundo segmento va vacío (el mismo formato que en entrenamiento, donde `p_ctx_drop` lo elimina al azar), por lo que el modelo funciona con o sin contexto. `max_len = 128`.

In [ ]:
# 5.1 Tokenizador (RoBERTuito) o, en SIN_RED, uno diminuto entrenado localmente
from transformers import AutoTokenizer
if SIN_RED:
    TOK, CFG_ENC = MT.construir_encoder_minimo(BL["train"][0].text.tolist() + [c for c in BL["train"][0].ctx.tolist() if isinstance(c, str)], RUN / "tok_min", hidden=96, capas=2, max_pos=PARAMS["max_len"] + 2)
else:
    TOK, CFG_ENC = AutoTokenizer.from_pretrained(ENCODER), None
MAXL = PARAMS["max_len"]
e = MT.codificar(TOK, ["Usted es un hp", "Usted sí es mucho hp, parce", "x " * 500], ["Discusión acalorada por una deuda", None, "contexto " * 300], MAXL, USAR_CONTEXTO)
print("forma del lote:", tuple(e["input_ids"].shape), "(padding dinámico, ≤ max_len =", MAXL, ")")
for i in range(2): print(TOK.decode(e["input_ids"][i], skip_special_tokens=False)[:160].replace(TOK.pad_token, ""))
assert e["input_ids"].shape[1] <= MAXL
print("fragmentación subword (tokens/palabra) — texto normal vs. ofuscado:",
      round(R.fragmentacion_subword(TOK, ["me gusta el fútbol de la costa"]), 2), "vs.", round(R.fragmentacion_subword(TOK, ["H1JUEPUTA", "hijuep*ta", "HI J U E P U T A"]), 2))
CHECKS["tokenización de pares (con/sin contexto, only_second, ≤ max_len)"] = True

## 4.5 Preparación para los modelos de referencia (A1 y A2) y decisiones heredadas
El cuaderno original comparó cuatro enfoques sobre la misma partición. Para que esta versión sea autosuficiente, aquí se documenta toda la preparación de datos que los sustenta. Todas las decisiones se toman con **train** (lo que se aprende) y **DEV** (lo que se elige); el TEST no se mira.

| Decisión | A1 MLP (TF-IDF) | A2 BiLSTM | A3 / multitarea RoBERTuito |
|---|---|---|---|
| Partición | `SP` (por artículo) | igual | igual |
| Limpieza | `limpiar()`: URL y usuarios normalizados, minúsculas, puntuación separada | igual que A1 | `preprocess_tweet()` de pysentimiento (lo exige el modelo); no se limpia dos veces |
| Stopwords | se evalúan con y sin, **conservando negaciones** | no se eliminan | no se eliminan |
| Tokens | palabras y bigramas | vocabulario propio aprendido en train | tokenizador del modelo |
| Longitud | no aplica | percentil 95 de train | 128 tokens (límite del modelo) |
| Desbalance | pesos de clase de train | igual | `pos_weight` por cabeza (sección 4.2) |

### 4.5.1 Limpieza para A1/A2 y comparación antes/después
`limpiar()` solo normaliza URL y `@usuarios`, pasa a minúsculas y separa la puntuación: **no borra negaciones** (`no`, `nunca`, `ni`) ni elimina mayúsculas o emojis (la celda 3.12 mostró que casi no cambian entre clases). El multitarea usa su propio preprocesamiento. La tabla compara el texto original con las dos versiones en casos que importan: negación, mayúsculas y emojis.

In [ ]:
# 4.5.1 Limpieza para A1/A2 (el modelo Transformer usa PRE = preprocess_tweet, sin segunda limpieza)
sp = SP                       # misma partición por artículo para todos los modelos
def limpiar(t, minus=True, quitar_tildes=False, separar_puntuacion=True):
    t = re.sub(r"https?://\S+", " URL ", t)
    t = re.sub(r"@\w+", " @usuario ", t)
    if minus:
        t = t.lower()
    if quitar_tildes:
        t = "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")
    if separar_puntuacion:
        t = re.sub(r"([!?¡¿.,;:()\"])", r" \1 ", t)
    return re.sub(r"\s+", " ", t).strip()

prep_a3 = PRE
d = sp["train"]
def _muestra(mask, n, seed=0):
    s = d[mask]
    return s.sample(min(n, len(s)), random_state=seed)
casos = pd.concat([_muestra(d.text.str.contains(r"\b(?:no|nunca|ni)\b", case=False), 3),
                   _muestra(d.text.map(prop_mayus) > 0.5, 2),
                   _muestra(d.text.str.contains(EMOJI), 2)])
pd.set_option("display.max_colwidth", 160)
display(pd.DataFrame({"original": casos.text, "A1/A2: limpiar()": casos.text.map(limpiar),
                      "A3: preprocess_tweet()": casos.text.map(prep_a3), "HATEFUL": casos.HATEFUL}))
assert {"no", "nunca"} <= set(limpiar("No lo haría nunca").split())     # las negaciones se conservan

**Resultado:** `limpiar()` conserva las negaciones; solo normaliza URL/`@usuario` y minúsculas. El preprocesamiento del Transformer es el de `pysentimiento`, sin una segunda limpieza.

### 4.5.2 Stopwords: la lista estándar incluye negaciones
Quitar stopwords es habitual en modelos bolsa-de-palabras, pero la lista de NLTK para español **incluye negaciones**, que cambian el sentido («no es un ladrón» frente a «es un ladrón»). La celda lo comprueba y construye la lista alternativa sin negaciones que se usó en el experimento de A1.

In [ ]:
# 4.5.2 Stopwords: ¿la lista estándar de español incluye negaciones?
try:
    import nltk; nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    sw = set(stopwords.words("spanish"))
    negaciones = ["no", "ni", "nunca", "jamás", "sin", "nada", "nadie", "tampoco"]
    print("negaciones dentro de la lista NLTK:", [w for w in negaciones if w in sw])
    sw_sin_neg = sorted(sw - set(negaciones))      # lista a usar en el experimento de A1
    print("stopwords NLTK:", len(sw), "| sin negaciones:", len(sw_sin_neg))
except Exception as ex:
    sw_sin_neg = None
    print("NLTK o la descarga no están disponibles aquí; se omite este experimento:", repr(ex)[:120])

**Resultado:** NLTK incluye `no`, `ni`, `sin` y `nada`. En A1 se probó con y sin stopwords **conservando negaciones**; la mejor configuración en DEV fue **sin eliminar stopwords**.

### 4.5.3 Tokenizador del Transformer y truncamiento con título
Se comprueba (a) que el tokenizador de RoBERTuito ya pasa a minúsculas (no hace falta un `.lower()` extra) y (b) cuánto se trunca con 128 tokens, con el comentario solo y con comentario + título de la noticia. Con `truncation="only_second"` solo se recorta el título, nunca el comentario.

In [ ]:
# 4.5.3 Lowercasing del tokenizador y truncamiento (comentario solo vs. comentario + título)
print("¿el tokenizador pasa a minúsculas solo?", TOK.tokenize("HOLA MUNDO") == TOK.tokenize("hola mundo"))
if SIN_RED:
    print("(SIN_RED: el tokenizador es de verificación, sin preentrenar; las cifras siguientes no significan nada)")
m = d.sample(min(5000, len(d)), random_state=0)
_t = [PRE(x) for x in m.text]; _c = [PRE(x) for x in m[COL_CTX]]
l_txt = np.array([len(x) for x in TOK(_t)["input_ids"]])
l_par = np.array([len(x) for x in TOK(_t, _c)["input_ids"]])
print("solo comentario   p95=%d  truncados a %d tokens=%.2f%%" % (np.percentile(l_txt, 95), MAXL, (l_txt > MAXL).mean() * 100))
print("comentario+título p95=%d  truncados a %d tokens=%.2f%%" % (np.percentile(l_par, 95), MAXL, (l_par > MAXL).mean() * 100))

**Resultado (corrida de referencia):** el tokenizador **sí** hace *lowercasing* (`True`). Solo comentario: p95 = 54 tokens y 0,05 % truncados; comentario + título: p95 = 77 y 0,08 % truncados. El título cabe casi siempre en 128 tokens.

### 4.5.4 A1: TF-IDF ajustado solo con train
Representación de A1: palabras y bigramas, `min_df = 3`, hasta 50.000 rasgos, `sublinear_tf`, y conservando `!` y `?`. **El vectorizador se ajusta solo con train** y se aplica a dev y test, para que ninguna estadística del vocabulario provenga de las particiones de evaluación.

In [ ]:
# 4.5.4 A1: TF-IDF ajustado SOLO con train
from sklearn.feature_extraction.text import TfidfVectorizer
def hacer_tfidf(stop=None):
    return TfidfVectorizer(preprocessor=limpiar, token_pattern=r"(?u)\b\w+\b|[!?¡¿]",
                           ngram_range=(1, 2), min_df=3, max_features=50_000,
                           sublinear_tf=True, stop_words=stop)
tfidf = hacer_tfidf()
X1 = {"train": tfidf.fit_transform(sp["train"].text)}
X1.update({n: tfidf.transform(sp[n].text) for n in ["dev", "test"]})
print({n: X.shape for n, X in X1.items()})
# Experimento en DEV: hacer_tfidf(sw_sin_neg) frente a hacer_tfidf(None).

**Resultado (corrida de referencia):** matrices de forma (36.587, 38.367) para train, (8.971, 38.367) para dev y (11.311, 38.367) para test.

### 4.5.5 A2: vocabulario propio y longitud de secuencia
Para el BiLSTM se aprende un vocabulario de 30.000 palabras **solo con train** y se fija la longitud de secuencia en el percentil 95 de train (menos relleno, poco truncamiento). Se mide también el porcentaje de palabras de dev que quedan fuera del vocabulario (OOV). Requiere TensorFlow; si no está instalado, la celda se omite.

In [ ]:
# 4.5.5 A2: vocabulario propio aprendido SOLO con train (requiere TensorFlow; opcional)
try:
    import tensorflow as tf
    limpio = {n: sp[n].text.map(limpiar).values for n in sp}
    n_tok = np.array([len(t.split()) for t in limpio["train"]])
    SEQ_LEN = int(np.percentile(n_tok, 95))
    vec = tf.keras.layers.TextVectorization(max_tokens=30_000, output_sequence_length=SEQ_LEN,
                                            standardize=None, split="whitespace")
    vec.adapt(limpio["train"])
    X2 = {n: vec(limpio[n]).numpy() for n in sp}
    vocab = set(vec.get_vocabulary())
    oov = np.mean([w not in vocab for t in limpio["dev"] for w in t.split()])
    print("SEQ_LEN:", SEQ_LEN, "| vocabulario:", len(vocab), "| % de palabras de dev fuera del vocabulario: %.1f" % (oov * 100))
    print("% de comentarios de train truncados:", round((n_tok > SEQ_LEN).mean() * 100, 1))
except ImportError:
    SEQ_LEN = None
    print("TensorFlow no está instalado: se omite la preparación de A2 (no afecta al modelo multitarea).")

**Resultado (corrida de referencia):** `SEQ_LEN` = 47 (percentil 95), vocabulario de 30.000 palabras, OOV en dev ≈ 4,3 % y 5,0 % de los comentarios de train truncados.

### 4.5.6 Desbalance de clases y configuración guardada
`HATEFUL` está desbalanceado (≈ 1 a 5,6), por lo que los modelos de referencia usan pesos de clase calculados **solo con train**. El multitarea usa `pos_weight` por cabeza (sección 4.2). Se guarda `prep_config.json` en la carpeta de la corrida para que la app y futuras ejecuciones lean exactamente la misma configuración.

In [ ]:
# 4.5.6 Pesos de clase (modelos de referencia), pos_weight por cabeza (multitarea) y configuración guardada
from sklearn.utils.class_weight import compute_class_weight
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=sp["train"].HATEFUL.values)
print("pesos de clase (train, modelos de referencia):", pesos.round(3))
print("pos_weight por cabeza (multitarea):", dict(zip(MT.HEADS, np.round(PESO_POS, 2).tolist())))
prep_config = {"split": "particion por articulo (seed 42)", "encoder": ENCODER, "max_len": MAXL,
               "preprocesamiento_transformer": PRE_NOMBRE, "a2_seq_len": SEQ_LEN,
               "limpiar": {"minus": True, "quitar_tildes": False, "separar_puntuacion": True},
               "class_weights": pesos.round(4).tolist(), "pos_weight_por_cabeza": np.round(PESO_POS, 3).tolist()}
json.dump(prep_config, open(RUN / "prep_config.json", "w"), indent=2, ensure_ascii=False)
print(prep_config)
CHECKS["preparación de A1/A2 y configuración guardada"] = True

**Resultado (corrida de referencia):** pesos de clase ≈ `[0,59 ; 3,25]` para el desbalance de `HATEFUL`.

**Conclusiones de la preparación para los modelos de referencia:**
- `limpiar()` no elimina negaciones; las mayúsculas y los emojis casi no difieren entre clases, por lo que no se agregaron rasgos extra.
- La lista de stopwords de NLTK contiene negaciones: en A1 la mejor configuración fue **sin stopwords**.
- El tokenizador de RoBERTuito ya pasa a minúsculas y el truncamiento a 128 tokens es ≈ 0 % (con o sin título).
- El vocabulario de A2 y el TF-IDF de A1 se ajustan **solo con train**; los pesos de clase también.

**Conclusiones de la preparación** (evidencia de las celdas 4.1–4.4):
- **Etiquetas:** `Y` es `float32` en {0, 1} y `M` es `bool`, ambas de forma N×3. Con OffendES, las tres cabezas superan el mínimo (50 positivos y 50 negativos) y se entrenan; sin OffendES, solo `odio_grupal`.
- **Normalizador:** `HI J U E P U T A`, `H1JUEPUTA` y `hijuep*ta` → `hijueputa`; `prostituTa` → `prostituta`; `prosti` y `hp` **se dejan intactos** (no se expanden). `eres un h.i.j.o.d.e.p.u.t.a` → `eres un hijodeputa`.
- **Tokenización:** el par comentario + contexto se codifica con padding dinámico (≤ 128 tokens). El texto ofuscado se fragmenta más (≈ 1,6 tokens por palabra frente a 1,0 del texto normal), señal de que el vocabulario no reconoce esas formas y de que la aumentación y la vista normalizada tienen sentido.
- **Sesgo de contexto a vigilar:** el contexto solo existe en las filas de piuba, que solo supervisan `odio_grupal`; las filas de OffendES no traen contexto. Por tanto, **las cabezas `vulgaridad` e `insulto` nunca recibieron contexto con etiqueta**: cualquier efecto del contexto sobre ellas no fue entrenado de forma explícita (ver 5.6).

# 5. Modelado y evaluación

Semilla fija **42**. Mismas particiones por artículo para el modelo nuevo y para A3. El **TEST no se usa** para elegir hiperparámetros, arquitectura ni umbral.

## 5.1 Arquitectura
```
 comentario [+ contexto] ──► encoder compartido (RoBERTuito) ──► vector <s> ─┬─► cabeza vulgaridad   ─► logit ─► σ
                                                                            ├─► cabeza insulto      ─► logit ─► σ
                                                                            └─► cabeza odio_grupal  ─► logit ─► σ
```
| Componente | Decisión |
|---|---|
| Encoder | `pysentimiento/robertuito-base-uncased` (108,2 M parámetros), el mismo de A3: preentrenado en tuits en español (jerga, ortografía informal, emojis); no se cambia |
| Cabezas | Tres, independientes (`Dropout → Linear`, 2.307 parámetros en total); salidas **no excluyentes** (sigmoide por cabeza, no softmax) |
| Pérdida | BCE con logits × máscara; cada cabeza se normaliza por *sus* etiquetas presentes y se promedian las cabezas con datos en el lote; `pos_weight` por cabeza (neg/pos de sus propias etiquetas, tope 10) |
| Cabeza sin datos suficientes | No se optimiza y devuelve `NaN` en inferencia |

**Solapamiento a declarar:** RoBERTuito y piuba provienen del mismo grupo (tuits); posible solapamiento con el preentrenamiento (arXiv 2111.09453).

In [ ]:
# 6.1 Instanciar y comprobaciones previas: forward, pérdida válida, backward y gradientes (con un lote REAL del train)
MT.fijar_semilla(SEED)
MODEL = MT.MultitareaToxicidad(encoder_name=None if SIN_RED else ENCODER, config=CFG_ENC, dropout=0.1, head_hidden=0, pooling="cls").to(DEVICE)
n_enc = sum(p.numel() for p in MODEL.encoder.parameters()); n_head = sum(p.numel() for p in MODEL.cabezas.parameters())
print(f"parámetros: encoder {n_enc/1e6:.1f} M | cabezas {n_head:,}")
lote = MT.Colador(TOK, MAXL, USAR_CONTEXTO)([DS_TRAIN[i] for i in range(8)])
st = MT.verificar_gradientes(MODEL, lote, torch.tensor(PESO_POS)); print(st)
assert st["loss_finita"] and st["forma_logits"] == (8, 3) and st["grad_encoder"] > 0
# Una cabeza sin etiquetas en el lote no recibe gradiente (máscara correcta); las que sí tienen etiquetas, sí.
for j, h in enumerate(MT.HEADS):
    assert (st["grad_cabezas"][h] > 0) == bool(lote["mask"][:, j].any()), h
MODEL.zero_grad(set_to_none=True)
CHECKS["forward, pérdida finita, backward y gradientes (encoder + cabezas con etiquetas)"] = True

**Verificación previa al entrenamiento** (lote real de train): la pérdida es finita, los logits tienen forma (8, 3), el encoder recibe gradiente y **una cabeza sin etiquetas en el lote no recibe gradiente** (la máscara funciona). En la corrida, 199 de 203 tensores entrenables recibieron gradiente.

## 5.2 Entrenamiento
AdamW (sin *weight decay* en bias / LayerNorm), *warmup* lineal y decaimiento, recorte de gradiente, **fp16** (`autocast` + `GradScaler`), padding dinámico y acumulación de gradiente. Parámetros de la corrida completa: 4 épocas, lote 16 × 2 (efectivo 32), `lr = 2e-5`, `max_len = 128`, validación cada media época en **DEV_CKPT**, paciencia 4. Se guarda `mejor.pt` cuando mejora el F1 macro (mejor umbral de la propia validación) de las cabezas activas.

In [ ]:
# 7.1 Entrenar (modo prueba: unos pasos para verificar; modo completo: ver PARAMS)
CFG = MT.CfgEntr(epochs=PARAMS["epochs"], bs=PARAMS["bs"], acum=PARAMS["acum"], lr=PARAMS["lr"], max_len=MAXL, eval_cada=PARAMS["eval_cada"],
                 paciencia=PARAMS["paciencia"], max_pasos=PARAMS["max_pasos"], amp=True, usar_contexto=USAR_CONTEXTO, seed=SEED, **AUG)
json.dump(dict(modo=MODO, sin_red=SIN_RED, params=PARAMS, cfg=CFG.__dict__, activas=ACTIVAS, pos_weight=PESO_POS.tolist(), origen=ORIGEN,
               n={k: len(v[0]) for k, v in BL.items()}), open(RUN / "config_corrida.json", "w"), indent=2, default=str)
t0 = time.time()
HIST = MT.entrenar(MODEL, TOK, DS_TRAIN, VAL("dev_ckpt"), CFG, PRE, RUN / "entrenamiento", PESO_POS, ACTIVAS, DEVICE)
print(f"entrenamiento: {time.time()-t0:.0f} s | validaciones: {len(HIST)} | épocas recorridas: {HIST.epoca.max():.2f}")
HIST

**Resultado del entrenamiento (corrida completa):** 778 s (≈ 13 min) en una GPU de Colab, 8 validaciones, 4 épocas. El mejor checkpoint fue el del **paso 4.998 (época 3,0)**, con F1 macro de validación = **0,857**; las dos validaciones posteriores no mejoraron y la paciencia (4) no se agotó, por lo que el entrenamiento terminó las 4 épocas.

| Época | loss_train | loss_val | F1 macro val | F1 odio grupal val |
|---:|---:|---:|---:|---:|
| 0,5 | 0,858 | 0,742 | 0,812 | 0,594 |
| 1,0 | 0,594 | 0,806 | 0,842 | 0,638 |
| 2,0 | 0,471 | 0,585 | 0,850 | 0,649 |
| 3,0 | 0,363 | 0,739 | **0,857** | **0,683** |
| 4,0 | 0,286 | 0,927 | 0,857 | 0,681 |

Lectura: el F1 de odio grupal sube hasta la época 3 y se estabiliza; la pérdida de validación crece después de la época 2 (probable sobreajuste de calibración), por lo que el criterio de selección es el F1 y no la pérdida. **El F1 de vulgaridad vale 1,0 en todas las validaciones y el de insulto ≈ 0,89**: con 37 y 47 etiquetas de validación esas cifras casi no discriminan entre checkpoints y la selección queda determinada, en la práctica, por `odio_grupal`.

In [ ]:
# 7.2 Curvas (si hay matplotlib) y comprobación de que el entrenamiento produjo valores válidos
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(9, 3)); HIST.plot(x="epoca", y=["loss_train", "loss_val"], ax=ax[0]); HIST.plot(x="epoca", y=[c for c in HIST if c.startswith("f1_")], ax=ax[1]); plt.tight_layout(); plt.show()
except Exception as ex: print("sin gráfico:", ex)
assert len(HIST) >= 1 and np.isfinite(HIST.loss_train).all() and (RUN / "entrenamiento" / "mejor.pt").exists()
CHECKS["entrenamiento completó ≥1 época/validación y guardó checkpoint"] = True

## 5.3 Validación, selección de checkpoint y calibración
El checkpoint ya es el de mejor F1 macro en **DEV_CKPT** (se recarga al terminar el bucle). Con ese modelo se ajustan, **solo en DEV_EVAL**, la *temperatura* por cabeza (*temperature scaling*) y el *umbral* que maximiza F1. Después se guarda el `Predictor` y se vuelve a cargar para comprobar que reproduce las mismas probabilidades.

In [ ]:
# 8.1 Calibración + Predictor + guardado y recarga (verifica que el checkpoint reproduce las mismas probabilidades)
dv = BL["dev_eval"]; Z_dv = MT.logits_de(MODEL, TOK, dv[0].text.tolist(), dv[0].ctx.tolist() if USAR_CONTEXTO else None, PRE, MAXL, USAR_CONTEXTO, device=DEVICE)
CAL = MT.calibrar(Z_dv, dv[1], dv[2], ACTIVAS); print(CAL)
META = dict(run=RUN_ID, modo=MODO, sin_red=SIN_RED, encoder=ENCODER if not SIN_RED else "aleatorio-diminuto", origen_datos=ORIGEN, calibradas=CAL["calibradas"],
            fuentes=sorted(BL["train"][0].fuente.unique().tolist()), col_ctx=COL_CTX, nota="umbrales y temperaturas ajustados solo en DEV_EVAL")
PRED = MT.Predictor(MODEL, TOK, PRE, PRE_NOMBRE, MAXL, USAR_CONTEXTO, ACTIVAS, CAL["temperaturas"], CAL["umbrales"], device=DEVICE, meta=META)
PRED.guardar(RUN / "modelo_v5")
PRED2 = MT.Predictor.cargar(RUN / "modelo_v5", DEVICE)
pr = ["Usted es un hp", "HI J U E P U T A", "Me encanta la cultura costeña"]
P1, P2 = PRED.probs(pr, ["Discusión", None, None]), PRED2.probs(pr, ["Discusión", None, None])
assert np.allclose(P1, P2, atol=1e-4, equal_nan=True), "el modelo recargado no reproduce las probabilidades"
ok_rango = np.all((P1[~np.isnan(P1)] >= 0) & (P1[~np.isnan(P1)] <= 1))
assert ok_rango and np.isnan(P1[:, [not a for a in ACTIVAS]]).all()
print("guardar → cargar: probabilidades idénticas ✔ | cabezas sin entrenar devuelven NaN ✔")
CHECKS["checkpoint guarda y recarga con probabilidades idénticas, en [0,1]"] = True

**Resultado:** temperaturas = vulgaridad 0,90 · insulto 1,43 · odio grupal 1,35; umbrales = **0,302 · 0,351 · 0,824**. El modelo recargado desde disco entrega probabilidades idénticas (tolerancia 1e-4), todas en [0, 1].

El umbral de `odio_grupal` (0,824) es alto: se ajustó con piuba, cuyo odio es explícito; esto pesa en la batería colombiana (5.5). Las temperaturas de insulto y odio grupal > 1 indican que el modelo estaba sobreconfiado y se suaviza.

## 5.4 Métricas por cabeza (DEV_EVAL)
Todas las cifras de la sección 5 son de **DEV_EVAL**. Las métricas son **por cabeza y solo sobre filas etiquetadas**; las tareas nunca se mezclan en una sola cifra.

In [ ]:
# 9.1 Métricas multietiqueta por cabeza (P/R/F1 de la clase positiva, F1 macro de la cabeza, AUC, matriz de confusión) + insultos
dv_df, dv_Y, dv_M = BL["dev_eval"]
P_dv = PRED.probs(dv_df.text.tolist(), dv_df.ctx.tolist() if USAR_CONTEXTO else None)
MET_DEV = MT.metricas_multietiqueta(dv_Y, dv_M, P_dv, PRED.thr)
display(MET_DEV) if "display" in dir() else print(MET_DEV)
j_ins = MT.HEADS.index("insulto"); j_g = MT.HEADS.index("odio_grupal")
if not ACTIVAS[j_ins]: print("⚠ Insulto: SIN evaluación en datos reales — la cabeza no tiene etiquetas. Necesita:", NECESITA["insulto"])
CHECKS["métricas por cabeza calculadas sin error"] = True

| Cabeza | n etiquetado | Positivos | Precision | Recall | F1 (positiva) | F1 macro de la cabeza | AUC | Umbral |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| vulgaridad | 37 | 7 | 1,000 | 1,000 | 1,000 | 1,000 | 1,000 | 0,302 |
| insulto | 48 | 11 | 0,900 | 0,818 | 0,857 | 0,909 | 0,961 | 0,351 |
| **odio_grupal** | **4.517** | **548** | **0,754** | **0,664** | **0,706** | **0,834** | **0,932** | **0,824** |

Matrices de confusión [TN FP / FN TP]: vulgaridad [30 0 / 0 7] · insulto [36 1 / 2 9] · odio grupal [3.850 119 / 184 364].

**Lectura:** solo `odio_grupal` tiene una evaluación con tamaño suficiente (548 positivos). Las cifras de vulgaridad (7 positivos) e insulto (11 positivos) **no permiten concluir nada**: un solo error cambiaría el F1 en varios puntos. La fila “MACRO” de la tabla impresa promedia tareas distintas y solo sirve como resumen, no como métrica de comparación.

## 5.5 Robustez ortográfica y batería contrastiva
**Variantes ortográficas (DEV_EVAL, primeras 3.000 filas, una familia de ofuscación por vez).** Retención = F1 con el comentario perturbado ÷ F1 con el original, para `odio_grupal`. Las filas de OffendES quedan fuera de esta submuestra, por lo que esta prueba **solo cubre `odio_grupal`**; para `insulto` se usa la batería.

In [ ]:
# 9.2 Variantes ortográficas: F1 por cabeza con el comentario perturbado vs. original (mismas filas; retención = F1_pert / F1_orig)
sub = dv_df.index[: (600 if MODO == "prueba" or SIN_RED else 3000)]
ORT = MT.evaluar_ortografia(PRED, dv_df.loc[sub].reset_index(drop=True), dv_Y[sub], dv_M[sub], col_ctx="ctx" if USAR_CONTEXTO else None)
display(ORT.pivot(index="perturbacion", columns="cabeza", values="retencion").round(3)) if "display" in dir() else print(ORT)

| Perturbación | Retención | Perturbación | Retención |
|---|---:|---|---:|
| caso (mayúsculas) | 1,000 | borrar | 0,862 |
| alargar | 0,963 | teclado | 0,855 |
| leet | 0,918 | mezcla | 0,840 |
| homoglifo | 0,912 | máscara `*` | 0,826 |
| sin_vocales | 0,818 | sustituir | 0,816 |
| espaciado | 0,794 | | |

Las perturbaciones más dañinas son el **espaciado** (0,79), la **sustitución de letras** (0,82), la eliminación de vocales (0,82) y las **máscaras** con `*` (0,83); el cambio de mayúsculas y el alargamiento casi no afectan. La retención mínima es 0,79: el modelo conserva la mayor parte de su F1 con texto alterado, sin llegar a ser invariante.

**Batería contrastiva (DEV, 77 casos).** Las etiquetas de la batería son propuestas sin validar; la tabla describe patrones, no verdad de referencia.

In [ ]:
# 9.3 Batería contrastiva DEV (ofuscaciones del enunciado, estereotipo vs. contradiscurso, hp amistoso/ofensivo, etc.)
# Etiquetas = propuestas de los autores (validado=False): se usan para ver patrones, NO como verdad de referencia.
bd = BAT["dev"].copy(); Yb, Mb = MT.etiquetas_bateria(bd); Pb = PRED.probs(bd.texto.tolist(), None)
for j, h in enumerate(MT.HEADS): bd[f"y_{h}"] = np.where(Mb[:, j], Yb[:, j], np.nan)
display(MT.metricas_multietiqueta(Yb, Mb, Pb, PRED.thr)) if "display" in dir() else None
for j, h in enumerate(MT.HEADS):
    if not ACTIVAS[j]: print(f"[{h}] sin entrenar: no se evalúa"); continue
    r = R.evaluar_bateria(bd[bd[f"y_{h}"].notna()], Pb[bd[f"y_{h}"].notna(), j], PRED.thr[j], objetivo=f"y_{h}")
    print(f"\n[{h}] acierto por familia = {r['acierto_familia']:.2f} (n={r['n_familias']}); peores fenómenos:"); print(r["por_fenomeno"].head(5).round(2).to_string())
# Consistencia ante ofuscación de la cabeza de insulto (variantes de «hijueputa»/«prostituta» vs. su forma base)
if ACTIVAS[j_ins]:
    bd2 = bd.assign(y_i=bd["y_insulto"]); print("\nconsistencia ofuscación (insulto):", {k: v for k, v in R.consistencia_ofuscacion(bd2, Pb[:, j_ins], PRED.thr[j_ins], "y_i").items() if k != "por_perturbacion"})

| Cabeza | n | Positivos | Precision | Recall | F1 | AUC | Acierto por familia |
|---|---:|---:|---:|---:|---:|---:|---:|
| vulgaridad | 42 | 3 | 0,273 | 1,000 | 0,429 | 0,923 | 0,64 |
| insulto | 63 | 21 | 0,750 | 1,000 | 0,857 | 0,998 | 0,75 |
| odio_grupal | 71 | 8 | 1,000 | 0,125 | 0,222 | 0,960 | 0,75 |

- **Insulto:** detecta los 21 casos positivos (recall 1,0) con 7 falsos positivos. En las **variantes ofuscadas** (23) de “hijueputa” y “prostituta” la retención de recall es 1,0, sin falsos positivos y con un cambio medio de probabilidad de apenas 0,0075 respecto de la forma base.
- **Vulgaridad:** recall 1,0 pero precisión baja (8 falsos positivos): marca como vulgares usos educativos, periodísticos o negados de la palabra. Parte de esto depende de la definición de la etiqueta (mención frente a uso) en la batería.
- **Odio grupal:** precisión 1,0 pero **recall 0,125 (1 de 8)**; los estereotipos regionales colombianos (“Todos los costeños son perezosos”) obtienen probabilidades ≈ 0,42, por debajo del umbral 0,824. Con tan pocos positivos la cifra es inestable, pero indica una debilidad real (ver 5.6 y 7).

## 5.6 Evaluación contextual
Mismas frases **sin** y **con** contexto. Son pruebas funcionales: no hay etiqueta, se inspeccionan y se comparan.

In [ ]:
# 9.4 Evaluación contextual: mismas frases SIN y CON contexto (pruebas funcionales; no hay etiquetas, se inspeccionan)
CTX = MT.evaluar_contexto(PRED)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 55)
display(CTX.round(3)) if "display" in dir() else print(CTX.round(3))

| Caso | Cabeza | Sin contexto | Con contexto |
|---|---|---:|---:|
| hp amistoso (“Usted sí es mucho hp, parce”) | vulgaridad | 0,941 | **0,300** |
| | insulto | 0,978 | **0,403** |
| hp en discusión (“Usted es un hp”) | vulgaridad | 0,990 | 0,977 |
| | insulto | 0,993 | 0,979 |
| Resto de casos (sin contexto en ambos) | — | igual | igual |

- El contexto amistoso **baja** de forma marcada las señales de vulgaridad e insulto del “hp”, mientras que el contexto de discusión las deja altas: el comportamiento deseado.
- **Cautela:** como se señaló en 4.4, las cabezas `vulgaridad` e `insulto` nunca vieron contexto con etiqueta durante el entrenamiento. Este efecto es un hallazgo **a confirmar**, no una capacidad demostrada: podría deberse a cómo el encoder reacciona a cualquier segundo segmento. Hace falta un conjunto contextual anotado (protocolo de la v4, §20) con contextos de control (otro texto neutro) para descartarlo.
- Con una redacción distinta del contexto amistoso (sección 6.1), el insulto del “hp” queda en 0,283, bajo el umbral, lo que muestra sensibilidad a la formulación del contexto.
- Ofuscaciones sin contexto: `H1JUEPUTA` (insulto 0,987), `hijuep*ta` (0,977) y `HI J U E P U T A` (0,749) superan el umbral de insulto; `prostituTa` (insulto 0,995, odio grupal 0,756) y `prosti` (insulto 0,942, vulgaridad 0,521) también se detectan como insulto.

## 5.7 Comparación con el modelo original (A3)
**Solo se compara lo comparable:** A3 es un clasificador binario de *odio grupal*, por lo que se compara con la cabeza `odio_grupal` sobre las **mismas filas** (piuba, DEV_EVAL, partición por artículo, N = 4.480; 548 positivos). El umbral de ambos se elige en DEV_EVAL (trato idéntico). A3 **no tiene** vulgaridad ni insulto: esas cabezas no se comparan con él. La diferencia de F1 lleva IC 95 % por *bootstrap* por artículo; una mejora solo se afirma si el IC excluye 0. A3 se carga desde el Hub y se usa solo en inferencia (no se modifica).

In [ ]:
# 10.1 Cargar A3 (solo inferencia). En SIN_RED se usa un sustituto aleatorio SOLO para ejercitar el código.
class _A3Falso:
    """NO es A3: clasificador diminuto aleatorio para verificar el flujo de comparación sin red."""
    def __init__(self):
        from transformers import RobertaForSequenceClassification
        self.m = RobertaForSequenceClassification(CFG_ENC).eval(); self.pre = PRE
    @torch.no_grad()
    def probs(self, textos, bs=64):
        out = []
        for i in range(0, len(textos), bs):
            enc = TOK([self.pre(t) for t in textos[i:i+bs]], truncation=True, max_length=MAXL, padding=True, return_tensors="pt"); enc.pop("token_type_ids", None)
            out.append(torch.softmax(self.m(**enc).logits, -1)[:, 1].numpy())
        return np.concatenate(out)
A3, A3_NOMBRE = None, None
if SIN_RED: A3, A3_NOMBRE = _A3Falso(), "SUSTITUTO ALEATORIO (no es A3)"
else:
    for cand in [Path("a3_model"), Path("artifacts/a3_model"), Path("artifacts_v2/backup_a3_original")]:     # copia local primero; el original no se toca
        if (cand / "config.json").exists(): A3, A3_NOMBRE = MT.ProbA3(str(cand)), f"local:{cand}"; break
    else:
        try: A3, A3_NOMBRE = MT.ProbA3(A3_HUB), f"hub:{A3_HUB}"
        except Exception as ex: print("⚠ No se pudo cargar A3:", repr(ex)[:200], "→ la comparación se omite (no se inventa).")
print("A3:", A3_NOMBRE)

In [ ]:
# 10.2 A3 vs. cabeza odio_grupal — mismas filas, mismo criterio de umbral (F1 máx. en DEV_EVAL), IC por artículo
if A3 is not None:
    s = (dv_df.fuente == "piuba").values; d_ = dv_df[s].reset_index(drop=True); y = d_.HATEFUL.astype(int).values
    ctx = d_.ctx.tolist() if USAR_CONTEXTO else None
    cand = {"A3 original": A3.probs(d_.text.tolist()),
            "v5 sin contexto, sin TTA": PRED.probs(d_.text.tolist(), None, tta=False)[:, j_g],
            "v5 con contexto, sin TTA": PRED.probs(d_.text.tolist(), ctx, tta=False)[:, j_g],
            "v5 con contexto + TTA": PRED.probs(d_.text.tolist(), ctx, tta=True)[:, j_g]}
    from sklearn.metrics import roc_auc_score
    thr = {k: R.elegir_umbral_f1(y, v) for k, v in cand.items()}
    filas = []
    for k, v in cand.items():
        m = R.metricas_binarias(y, v >= thr[k]); b = MT.bootstrap_delta_f1(y, cand["A3 original"], thr["A3 original"], v, thr[k], d_.grupo.values, B=300) if k != "A3 original" else {}
        filas.append(dict(modelo=k, umbral=round(thr[k], 3), precision=m["precision"], recall=m["recall"], f1=m["f1"], f1_macro=m["f1_macro"], auc=roc_auc_score(y, v),
                          dF1_vs_A3=b.get("delta_medio"), IC95=tuple(round(x, 3) for x in b["ic95"]) if b else None))
    COMP = pd.DataFrame(filas).set_index("modelo"); display(COMP.round(3)) if "display" in dir() else print(COMP.round(3))
    COMP.to_csv(RUN / "comparacion_A3_dev_eval.csv")
    print(f"\nA3 = {A3_NOMBRE} | N={len(y)} | positivos={int(y.sum())}")
    if SIN_RED: print("⚠ SIN_RED: A3 es un sustituto aleatorio y los datos son sintéticos → estas cifras NO tienen interpretación.")
    for k in list(cand)[1:]:
        ic = COMP.loc[k, "IC95"]
        print(f"  {k}: ", "IC95 de ΔF1 excluye 0 a favor de v5" if ic[0] > 0 else ("IC95 excluye 0 a favor de A3" if ic[1] < 0 else "sin diferencia detectable (IC95 incluye 0)"))
    CHECKS["comparación A3 vs. cabeza odio_grupal calculada (mismas filas)"] = True

| Modelo | Umbral | Precision | Recall | F1 | F1 macro | AUC | ΔF1 frente a A3 [IC 95 %] |
|---|---:|---:|---:|---:|---:|---:|---|
| A3 original | 0,487 | 0,784 | 0,615 | 0,689 | 0,826 | 0,919 | — |
| v5 sin contexto, sin TTA | 0,863 | 0,813 | 0,588 | 0,682 | 0,822 | 0,913 | −0,007 [−0,034; 0,018] |
| v5 con contexto, sin TTA | 0,824 | 0,749 | 0,664 | 0,704 | 0,833 | 0,932 | +0,014 [−0,016; 0,043] |
| **v5 con contexto + TTA** | 0,824 | 0,754 | 0,664 | **0,706** | 0,834 | 0,932 | **+0,016 [−0,012; 0,045]** |

**Lectura:** en la tarea común **no hay diferencia detectable** (todos los IC incluyen 0). La versión con contexto gana recall (0,664 frente a 0,615) a costa de precisión (0,754 frente a 0,784), y su AUC es algo mayor (0,932 frente a 0,919), pero **no se declara mejora**. Sin contexto, la cabeza equivale a A3, como era de esperar con el mismo encoder y los mismos datos. Esto cumple H2 en el sentido de “sin pérdida”.

In [ ]:
# 10.3 Batería DEV: A3 (umbral DEV_EVAL) vs. cabeza odio_grupal, objetivo y_grupo. Etiquetas = propuestas (validado=False).
if A3 is not None:
    b = BAT["dev"]; k = b.y_grupo.isin([0, 1]).values
    pa = A3.probs(b.texto.tolist()); pv = PRED.probs(b.texto.tolist(), None)[:, j_g]
    for nombre, p, t in [("A3", pa, thr["A3 original"]), ("v5 odio_grupal", pv, PRED.thr[j_g])]:
        r = R.evaluar_bateria(b, p, t, "y_grupo"); print(f"{nombre:15s} F1={r['global_']['f1']:.3f} P={r['global_']['precision']:.3f} R={r['global_']['recall']:.3f} | acierto por familia={r['acierto_familia']:.2f}")
    # Insultos: A3 no se entrenó para ellos; solo se muestra qué hace (descriptivo, NO comparable)
    ins = b.categoria == "insulto"; print(f"\n[descriptivo, no comparable] filas 'insulto' de la batería marcadas por A3 como odio grupal: {(pa[ins.values] >= thr['A3 original']).mean():.2f} | por la cabeza odio_grupal v5: {(pv[ins.values] >= PRED.thr[j_g]).mean():.2f}")
    if ACTIVAS[j_ins]: print("cabeza insulto v5 sobre esas filas:", round(float((PRED.probs(b.texto[ins].tolist(), None)[:, j_ins] >= PRED.thr[j_ins]).mean()), 2))

**Batería DEV (objetivo `y_grupo`, etiquetas propuestas, 8 positivos):** A3 obtiene F1 = 0,500 (P = 0,75; R = 0,375) y la cabeza `odio_grupal` de v5 F1 = 0,222 (P = 1,0; R = 0,125); ambos con acierto por familia = 0,75. **En estereotipos regionales colombianos la cabeza nueva rinde peor que A3**, con la salvedad de que son 8 positivos y de que las etiquetas no están validadas.

**Insultos (descriptivo, no comparable):** A3 marca como “odio grupal” el 5 % de las filas de insulto de la batería y la cabeza `odio_grupal` de v5 el 0 %; la cabeza `insulto` de v5 las marca todas (100 %). Es consistente con que el insulto personal **no** es odio grupal, pero A3 no se entrenó para insultos, por lo que no se compara.

**TEST final:** no se evaluó (`EVALUAR_TEST_FINAL = False`). Debe ejecutarse una sola vez, con el modelo congelado.

In [ ]:
# 10.4 TEST FINAL — se ejecuta UNA vez, con todo congelado (EVALUAR_TEST_FINAL=True). Nada se elige con esto.
if EVALUAR_TEST_FINAL:
    te_df, te_Y, te_M = BL["test"]; P_te = PRED.probs(te_df.text.tolist(), te_df.ctx.tolist() if USAR_CONTEXTO else None)
    MET_TEST = MT.metricas_multietiqueta(te_Y, te_M, P_te, PRED.thr); display(MET_TEST) if "display" in dir() else print(MET_TEST)
    if A3 is not None:
        s = (te_df.fuente == "piuba").values; yt = te_df.HATEFUL[s].astype(int).values; pa = A3.probs(te_df.text[s].tolist())
        pv = P_te[s, j_g]; print("TEST A3 F1:", round(R.metricas_binarias(yt, pa >= thr["A3 original"])["f1"], 3), "| v5 odio_grupal F1:", round(R.metricas_binarias(yt, pv >= PRED.thr[j_g])["f1"], 3),
                                 "| ΔF1 IC95:", MT.bootstrap_delta_f1(yt, pa, thr["A3 original"], pv, PRED.thr[j_g], te_df.grupo[s].values, B=300)["ic95"])
    bt = BAT["test"]; Yt_, Mt_ = MT.etiquetas_bateria(bt); display(MT.metricas_multietiqueta(Yt_, Mt_, PRED.probs(bt.texto.tolist(), None), PRED.thr)) if "display" in dir() else None
else: print("TEST reservado (EVALUAR_TEST_FINAL=False). Actívalo solo cuando el modelo esté congelado.")

## 5.8 Análisis de errores e iteración
Falsos positivos y falsos negativos más seguros de la cabeza `odio_grupal` en DEV_EVAL (filas de piuba).

In [ ]:
# 9.5 Falsos positivos y falsos negativos (cabeza odio grupal sobre DEV_EVAL, filas piuba)
j_g = MT.HEADS.index("odio_grupal"); sel = (dv_df.fuente == "piuba").values
fp, fn = MT.errores_fp_fn(dv_df[sel].reset_index(drop=True), dv_Y[sel, j_g], P_dv[sel, j_g], PRED.thr[j_g], k=8)
print("FALSOS POSITIVOS (más seguros):"); print(fp.to_string()); print("\nFALSOS NEGATIVOS (más seguros):"); print(fn.to_string())

## 5.9 HateCheck-es: pruebas funcionales
`Paul/hatecheck-spanish` agrupa casos de prueba por **funcionalidad** (ortografía alterada, negaciones, contra-discurso, uso no ofensivo de insultos…). El cuaderno original mostró aquí las debilidades de A3: ortografía alterada, odio implícito, negaciones y contra-discurso (muchos falsos positivos). La celda repite la prueba con **A3** y con la cabeza **`odio_grupal`** de v5 (sin contexto, con TTA), cada uno con su umbral elegido en DEV_EVAL (no se reutiliza el umbral 0,28 de la v4), y muestra además cuánto marcan **vulgaridad** e **insulto** en las funcionalidades de groserías, insultos y contra-discurso. Es una evaluación **solo de lectura**: no se ajusta nada con ella.

**Qué mirar:** las funcionalidades `spell_*` (hipótesis H3), `negate_*`, `counter_*` / `ref_*` (contra-discurso y referencias) y las de groserías / insultos no ofensivos.

In [ ]:
# 5.9 HateCheck-es: A3 frente a la cabeza odio_grupal (mismos casos) + tasas de las tres cabezas por funcionalidad
if SIN_RED or A3 is None:
    print("Omitido (SIN_RED o A3 no disponible).")
else:
    from datasets import load_dataset
    hc = load_dataset("Paul/hatecheck-spanish", split="test").to_pandas()
    y_hc = (hc.label_gold.str.lower() == "hateful").astype(int).values
    P_hc = PRED.probs(hc.test_case.tolist(), None, tta=True); p_a3 = A3.probs(hc.test_case.tolist())
    filas = []
    for nombre, p, t in [("A3 original", p_a3, thr["A3 original"]), ("v5 odio_grupal", P_hc[:, j_g], PRED.thr[j_g])]:
        r = R.metricas_binarias(y_hc, p >= t); filas.append(dict(modelo=nombre, umbral=round(float(t), 3), precision=r["precision"], recall=r["recall"], f1=r["f1"], n=r["n"]))
    display(pd.DataFrame(filas).set_index("modelo").round(3))
    hc = hc.assign(y=y_hc, ok_a3=((p_a3 >= thr["A3 original"]).astype(int) == y_hc), ok_v5=((P_hc[:, j_g] >= PRED.thr[j_g]).astype(int) == y_hc),
                   marca_vulg=(P_hc[:, 0] >= PRED.thr[0]), marca_ins=(P_hc[:, 1] >= PRED.thr[1]), marca_grp=(P_hc[:, 2] >= PRED.thr[2]))
    acc = hc.groupby("functionality").agg(n=("y", "size"), acc_A3=("ok_a3", "mean"), acc_v5=("ok_v5", "mean"))
    acc["delta_v5_menos_A3"] = acc.acc_v5 - acc.acc_A3
    print("\nFuncionalidades con menor acierto de v5:"); display(acc.sort_values("acc_v5").head(10).round(3))
    print("Donde v5 empeora más frente a A3:"); display(acc.sort_values("delta_v5_menos_A3").head(8).round(3))
    print("Donde v5 mejora más frente a A3:"); display(acc.sort_values("delta_v5_menos_A3").tail(8).round(3))
    sel = hc.functionality.str.contains("profanity|slur|counter|ident|ref_", regex=True)
    print("Tasa de casos marcados por cada cabeza (funcionalidades de groserías, insultos, identidad y contra-discurso):")
    display(hc[sel].groupby("functionality")[["marca_vulg", "marca_ins", "marca_grp"]].mean().round(2))
    hc.to_csv(RUN / "hatecheck_v5_vs_A3.csv", index=False)

**Lectura de HateCheck-es:** la comparación global y por funcionalidad se lee en la salida de la celda anterior; no se resumen aquí cifras que dependen de la corrida. Precauciones: HateCheck-es tiene etiquetas binarias de **odio**, por lo que solo la cabeza `odio_grupal` es comparable con A3; las tasas de `vulgaridad` e `insulto` son descriptivas. El cuaderno original no replicó con código la evaluación fuera de dominio con ALIA (YouTube / TikTok, etiquetas de LLM), donde reportó para A3 F1 ≈ 0,28 y recall ≈ 0,17; **esa evaluación no se repitió para v5** y queda pendiente (sección 7).

- **Falsos positivos (p ≈ 0,98–0,99):** agresiones políticas o personales sin grupo protegido (“Viejo terrorista, igual que su fundación”, “Esta vieja es peor que el Tuerto”), expresiones coloquiales (“Uno menos”) e insultos que mezclan una alusión a un colectivo (“los bolivianos se vienen a operar acá gratis”). El modelo confunde **ataque** con **odio a un grupo**: justamente la frontera que la cabeza `insulto` debería absorber, pero que `HATEFUL` de piuba no etiqueta de forma explícita.
- **Falsos negativos (p < 0,03):** términos discriminatorios coloquiales o regionales (“mogólico”), burlas por apariencia (“Lali gera”) y odio político sin léxico explícito (“el VOTANTE primate de Cambiemos…”). Es el límite conocido: el odio implícito y la jerga fuera del vocabulario aprendido.
- **Decisión de iteración:** no se ajustó nada con esta evidencia (sería usar DEV_EVAL para decidir y luego reportar sobre él). Se registra como entrada para la siguiente versión: una cabeza `insulto` con datos de ataque a grupos y anotación en español colombiano.

## 5.10 Recursos de cómputo
- **Entrenamiento:** 778 s (≈ 13 min) en una GPU de Colab, fp16, lote efectivo 32, 4 épocas. Para comparar, A3 se entrenó en ≈ 55 min en Apple Silicon (MPS) sobre solo piuba.
- **Tamaño:** encoder de 108,2 M parámetros + 2.307 de cabezas; el checkpoint guardado (`pesos.pt`) ocupa lo mismo que A3 (misma arquitectura base).
- **Inferencia:** la TTA duplica el costo cuando el normalizador cambia el texto (dos vistas). La celda siguiente mide la latencia en la máquina donde se ejecute el cuaderno y estima las horas de cómputo para 100.000 comentarios al mes, con la misma referencia de costo por API del cuaderno original (sin F1 medido para el LLM).
- **Decisión económica:** igual que en el cuaderno original, el modelo local no tiene costo por solicitud y mantiene el texto en el servidor propio; la inversión es el entrenamiento (≈ 13 min en GPU de Colab) y el alojamiento de la app.

In [ ]:
# 5.10 Latencia de inferencia por comentario (lote = 1) en el equipo actual
import platform
print(platform.platform(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")
_tx = ["HI J U E P U T A"]          # el normalizador cambia este texto → 2 vistas con TTA
_lat = {}
for _tta in (False, True):
    PRED.probs(_tx, [None], tta=_tta)                       # calentamiento
    _t0 = time.time(); _n = 30
    for _ in range(_n): PRED.probs(_tx, [None], tta=_tta)
    _lat[_tta] = (time.time() - _t0) / _n * 1000
    print(f"latencia por comentario (lote=1, {'con' if _tta else 'sin'} TTA): {_lat[_tta]:.1f} ms")
print(f"Cómputo por 100.000 comentarios/mes: {100000 * _lat[True] / 3.6e6:.2f} h (con TTA) | {100000 * _lat[False] / 3.6e6:.2f} h (sin TTA)")
print("Referencia API (supuesto del cuaderno original; tarifa consultada el 2026-11-06 y NO reverificada): gpt-4o-mini $0,15 / $0,60 por 1 M de tokens "
      "de entrada / salida, ≈ 200 + 20 tokens por comentario → ≈ $4,2/mes a 100.000 comentarios. No se midió el F1 de ningún LLM.")

## 5.11 Estado de las hipótesis
| Hipótesis | Estado | Evidencia |
|---|---|---|
| H1 — etiquetas compatibles por cabeza | **Cumplida por construcción** | Máscara + auditoría (4.2); `HATEFUL = 0` no se usa como negativo de las otras cabezas |
| H2 — sin pérdida frente a A3 en la tarea común | **No refutada** | ΔF1 = +0,016, IC 95 % [−0,012; 0,045] (5.7). No se declara mejora |
| H3 — robustez ortográfica | **Parcialmente respaldada** | Retención ≥ 0,79 en `odio_grupal`; insulto estable en 23 variantes de la batería. Vulgaridad e insulto sin evaluación ortográfica en DEV_EVAL con tamaño suficiente |
| H4 — el contexto cambia la salida como se espera | **No concluyente** | Efecto visible en “hp amistoso” (5.6), pero las cabezas afectadas no se entrenaron con contexto etiquetado |

# 6. Aplicación web
La app es el gesto concreto del pitch: **alguien escribe, alguien más podría leerlo; el sistema muestra tres señales y deja la decisión a una persona.**

1. **Inferencia desde el modelo guardado (6.1):** el `Predictor` se carga desde disco —el mismo objeto que usa la app— y se prueba con ejemplos con y sin contexto.
2. **Exportación (6.2):** `modelo_v5/` (pesos, configuración del encoder, tokenizador y `multitarea_config.json` con umbrales, temperaturas y cabezas activas), los dos módulos, `app.py` (Gradio) y `requirements.txt`, listos para un Space de Hugging Face. Se comprueba que la carpeta exportada reproduce las probabilidades.
3. **Laboratorio Streamlit (6.3):** interfaz de prueba con el comentario, el contexto opcional, las tres probabilidades con su umbral y la diferencia que produce el contexto.

## 6.1 Inferencia con ejemplos
Probabilidades calibradas por cabeza, con y sin contexto. Una señal alta no es un veredicto: la decisión final requiere revisión humana.

In [ ]:
# 11.1 Inferencia con el modelo CARGADO desde disco (el mismo que usará la app)
PRED_APP = MT.Predictor.cargar(RUN / "modelo_v5", DEVICE)
EJ = [("Me encanta la cultura costeña", None), ("Todos los costeños son perezosos", None),
      ("No está bien decir que todos los costeños son perezosos", None),
      ("Usted sí es mucho hp, parce", "Conversación entre amigos planeando el asado, todo en buena onda."),
      ("Usted es un hp", "Discusión acalorada: le reclama a otro usuario por una deuda."),
      ("HI J U E P U T A", None), ("H1JUEPUTA", None), ("hijuep*ta", None), ("prostituTa", None), ("prosti", None),
      ("Qué parche tan bacano, mijo", None), ("no manches güey, qué chido", None)]
RES = PRED_APP.predecir([e[0] for e in EJ], [e[1] for e in EJ]).assign(contexto=[e[1] or "—" for e in EJ])
cols = ["texto", "contexto"] + [c for h in MT.HEADS for c in (f"p_{h}", h)]
display(RES[cols].round(3)) if "display" in dir() else print(RES[cols].round(3))
CHECKS["inferencia produce probabilidades válidas por categoría (con y sin contexto)"] = bool(np.isfinite(RES[[f"p_{h}" for h, a in zip(MT.HEADS, ACTIVAS) if a]].values).all())

**Resultados de la corrida (señal = probabilidad ≥ umbral de la cabeza):**

| Comentario (contexto) | Vulgaridad | Insulto | Odio grupal |
|---|---:|---:|---:|
| Me encanta la cultura costeña | 0,047 | 0,041 | 0,056 |
| Todos los costeños son perezosos | 0,426 ✔ | 0,358 ✔ | 0,420 |
| No está bien decir que todos los costeños son perezosos | 0,101 | 0,121 | 0,235 |
| Usted sí es mucho hp, parce (conversación entre amigos) | 0,209 | 0,283 | 0,061 |
| Usted es un hp (discusión acalorada) | 0,979 ✔ | 0,978 ✔ | 0,039 |
| `HI J U E P U T A` | 0,851 ✔ | 0,749 ✔ | 0,248 |
| `H1JUEPUTA` | 0,994 ✔ | 0,987 ✔ | 0,357 |
| `hijuep*ta` | 0,995 ✔ | 0,977 ✔ | 0,284 |
| `prostituTa` | 0,615 ✔ | 0,995 ✔ | 0,756 |
| `prosti` | 0,521 ✔ | 0,942 ✔ | 0,098 |
| Qué parche tan bacano, mijo | 0,087 | 0,012 | 0,018 |
| no manches güey, qué chido | 0,266 | 0,050 | 0,008 |

Lectura: las expresiones coloquiales amistosas quedan por debajo de los umbrales y el `hp` en discusión por encima, y las cinco formas ofuscadas del enunciado se detectan como vulgaridad e insulto. **Punto débil:** “Todos los costeños son perezosos” activa vulgaridad e insulto (0,426 y 0,358, apenas sobre sus umbrales) pero **no** `odio_grupal` (0,420 < 0,824): el estereotipo regional, que es exactamente el caso de odio grupal, no se reconoce como tal (ver 7).

## 6.2 Exportación para Gradio / Hugging Face Spaces
La app usa **la misma clase `Predictor`** que este cuaderno, así que preprocesamiento, normalización / TTA y calibración son idénticos. Las cabezas sin entrenar aparecen en la app como “sin entrenar (faltan datos)”.

In [ ]:
EXP = MT.exportar_app(PRED, RUN / "export_v5")
print(sorted(p.name for p in EXP.iterdir()), "| modelo_v5:", sorted(p.name for p in (EXP / "modelo_v5").iterdir()))
# La carpeta exportada debe cargar y reproducir las probabilidades
P3 = MT.Predictor.cargar(EXP / "modelo_v5", DEVICE).probs(pr, ["Discusión", None, None]); assert np.allclose(P1, P3, atol=1e-4, equal_nan=True)
import shutil; shutil.make_archive(str(RUN / "export_v5"), "zip", EXP); print("zip:", RUN / "export_v5.zip")
CHECKS["exportación carga y reproduce las probabilidades"] = True
# Para la app: subir el contenido de export_v5/ a un Space de Gradio (app.py, requirements.txt, multitarea_es.py, robustez_es.py, modelo_v5/).
# Las cabezas sin entrenar aparecen en la app como «sin entrenar (faltan datos)».

## 6.3 Laboratorio Streamlit
La app de Streamlit es la interfaz de prueba del modelo. Permite:
- elegir un **ejemplo** (los casos de la tarea: cultura costeña, estereotipo, contradiscurso, `hp` amistoso y en discusión, ofuscaciones) o escribir un comentario propio;
- añadir un **contexto opcional** (título de la noticia o mensaje previo);
- activar o no la **vista normalizada (TTA)**;
- ver, por cada categoría, la **probabilidad**, el **umbral** y la señal (sí / no), y cuánto cambia la probabilidad **con y sin contexto**.

**Cómo se ejecuta.** *En Colab* (celdas siguientes): se escribe `app_streamlit.py`, se lanza Streamlit y se abre un túnel público temporal de Cloudflare cuyo enlace se imprime; el enlace deja de funcionar al detener la celda. *En local:* se descarga `export_v5.zip`, se descomprime y se ejecuta `streamlit run app_streamlit.py` desde esa carpeta (la app busca `modelo_v5` en la carpeta actual). En ambos casos se requieren las mismas versiones de `torch` y `transformers` con las que se entrenó.

**Estado de verificación:** en la corrida de referencia, la app se lanzó y el túnel entregó el enlace; la revisión manual de la interfaz queda a cargo del equipo. Se probó, además, la lógica de la app con el marco de pruebas de Streamlit sobre un modelo diminuto de verificación (no con el modelo entrenado).

In [ ]:
%%writefile app_streamlit.py
# -*- coding: utf-8 -*-
import os, glob
from pathlib import Path
import numpy as np, pandas as pd, streamlit as st
import robustez_es as R
from multitarea_es import Predictor, HEADS

st.set_page_config(page_title="Menos odio a la vista · v5", page_icon="🛡️", layout="centered")

def _ruta_modelo():
    for p in [os.environ.get("MODELO_DIR"), "modelo_v5"]:
        if p and Path(p, "multitarea_config.json").exists(): return p
    c = sorted(glob.glob("artifacts_v5/*/modelo_v5/multitarea_config.json"))
    return str(Path(c[-1]).parent) if c else None

@st.cache_resource(show_spinner="Cargando modelo…")
def cargar(ruta): return Predictor.cargar(ruta)

ruta = _ruta_modelo()
if ruta is None:
    st.error("No se encontró `modelo_v5`. Defina MODELO_DIR o ejecute el entrenamiento primero."); st.stop()
P = cargar(ruta)

NOMBRES = {"vulgaridad": "Vulgaridad", "insulto": "Insulto / ataque personal", "odio_grupal": "Odio grupal"}
AYUDA = {"vulgaridad": "Lenguaje grosero, aunque no agreda a nadie.",
         "insulto": "Expresión dirigida a agredir a otra persona.",
         "odio_grupal": "Ataque, discriminación o estereotipo despectivo contra un grupo."}
EJEMPLOS = {
    "— elegir ejemplo —": ("", ""),
    "Cultura costeña": ("Me encanta la cultura costeña", ""),
    "Estereotipo": ("Todos los costeños son perezosos", ""),
    "Contradiscurso": ("No está bien decir que todos los costeños son perezosos", ""),
    "hp amistoso": ("Usted sí es mucho hp, parce", "Conversación entre amigos planeando el asado, todo en buena onda."),
    "hp en discusión": ("Usted es un hp", "Discusión acalorada: le reclama a otro usuario por una deuda."),
    "Ofuscado: espaciado": ("HI J U E P U T A", ""), "Ofuscado: leet": ("H1JUEPUTA", ""),
    "Ofuscado: máscara": ("hijuep*ta", ""), "Ofuscado: mayúscula": ("prostituTa", ""), "Abreviado": ("prosti", ""),
}

st.title("🛡️ Menos odio a la vista · v5")
st.caption("Un único modelo entrega tres señales independientes sobre el comentario y, si existe, su contexto. "
           "Son probabilidades calibradas, no un veredicto: la decisión final corresponde a una persona.")

ej = st.selectbox("Ejemplos", list(EJEMPLOS))
if ej != st.session_state.get("_ej"):
    st.session_state["_ej"] = ej
    st.session_state["txt"], st.session_state["ctx"] = EJEMPLOS[ej]
texto = st.text_area("Comentario", key="txt", height=90)
ctx = st.text_input("Contexto (opcional: título de la noticia o mensaje previo)", key="ctx")
c1, c2 = st.columns(2)
tta = c1.checkbox("Usar vista normalizada (TTA)", value=True, help="Promedia el texto original y su versión sin ofuscación (espaciado, leet, máscaras…).")
comparar = c2.checkbox("Comparar con y sin contexto", value=True)

if st.button("Analizar", type="primary") and texto.strip():
    ctxs = [ctx.strip() or None]
    pc = P.probs([texto], ctxs, tta=tta)[0]
    ps = P.probs([texto], None, tta=tta)[0] if (comparar and ctxs[0]) else None
    norm = R.normalizar_ofuscacion(texto, minusculas=False)
    if norm != texto: st.info(f"Vista normalizada del comentario: **{norm}**")
    for j, h in enumerate(HEADS):
        st.subheader(NOMBRES[h]); st.caption(AYUDA[h])
        if not P.activas[j]:
            st.warning("Cabeza sin entrenar: faltan datos etiquetados para esta categoría."); continue
        p, t = float(pc[j]), float(P.thr[j])
        st.progress(min(max(p, 0.0), 1.0), text=f"probabilidad {p:.3f}  ·  umbral {t:.3f}  →  {'🔴 señal' if p >= t else '🟢 sin señal'}")
        if ps is not None: st.caption(f"Sin contexto: {float(ps[j]):.3f} (cambio al añadir contexto: {p - float(ps[j]):+.3f})")
    with st.expander("Detalles del modelo"):
        st.json({k: v for k, v in P.meta.items()})
        st.write("Umbrales:", dict(zip(HEADS, np.round(P.thr, 3))), "· Temperaturas:", dict(zip(HEADS, np.round(P.T, 3))))
elif not texto.strip():
    st.write("Escriba un comentario o elija un ejemplo.")

with st.expander("Cómo leer el resultado y límites"):
    st.markdown("""
- Las tres categorías **no son excluyentes**: un comentario puede ser vulgar e insultante a la vez.
- El umbral de cada categoría se eligió en una partición de desarrollo y **no es universal**; cambiarlo mueve el equilibrio entre falsos positivos y falsos negativos.
- El **contexto** puede cambiar la señal, pero las categorías de vulgaridad e insulto no se entrenaron con contexto etiquetado: el efecto es una observación, no una garantía.
- Los estereotipos regionales (por ejemplo, contra costeños) **no siempre se reconocen** como odio grupal: el modelo se entrenó con español rioplatense y comentarios de redes de influencers.
- Con textos muy ofuscados el desempeño baja; la vista normalizada ayuda, pero no resuelve todos los casos.
- La herramienta no sustituye la moderación humana ni es una evaluación clínica.
""")

In [ ]:
# 6.3 Lanzar Streamlit en Colab y abrir un enlace público temporal (Cloudflare Tunnel)
import os, subprocess, sys, time, re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"])
try: os.environ["MODELO_DIR"] = str((RUN / "modelo_v5").resolve())   # el modelo recién entrenado
except NameError: pass                                               # si se reinició la sesión: usa el último artifacts_v5/*/modelo_v5

subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
subprocess.Popen(["streamlit", "run", "app_streamlit.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    time.sleep(1.5)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: url = m.group(0); break
print("Abre:", url or "no apareció el enlace; revisa tunnel.log y streamlit.log")

## 6.4 Estado de verificación de ESTA ejecución
La lista sale de las comprobaciones (`assert`) que se ejecutaron realmente arriba. Si un paso no corrió, no aparece como verificado.

In [ ]:
print(f"modo={MODO} | SIN_RED={SIN_RED} | datos={ORIGEN}\n")
for k, v in CHECKS.items(): print("✔" if v else "✘", k)
print("\ncabezas entrenadas:", [h for h, a in zip(MT.HEADS, ACTIVAS) if a], "| sin datos (no entrenadas):", [h for h, a in zip(MT.HEADS, ACTIVAS) if not a])
print("TEST evaluado:", EVALUAR_TEST_FINAL)
if SIN_RED or MODO == "prueba": print("\n⚠ Esta corrida es de PRUEBA/verificación técnica: sus cifras no sirven para decidir. Ejecuta MODO='completo' en Colab T4 para resultados.")
json.dump(CHECKS, open(RUN / "verificaciones.json", "w"), indent=2, ensure_ascii=False)

# 7. Limitaciones, condiciones de uso y mejoras futuras
- **No es terapia ni prevención clínica:** solo ofrece señales para reducir la exposición a ataques.
- **No es censura absoluta:** tres probabilidades + umbrales explícitos + revisión humana.
- **Validación diminuta de las cabezas nuevas:** `vulgaridad` e `insulto` se validaron con 37 y 47–48 etiquetas (la partición `validation` de OffendES tiene 99 filas). Un F1 de 1,0 en vulgaridad con 7 positivos **no es evidencia**. *Mejora:* reservar una validación mayor tomada de `train` de OffendES (por influencer, para evitar fuga) y reentrenar; evaluar en el `test` oficial de OffendES (≈ 11.000 y 13.000 etiquetas).
- **TEST sin evaluar:** `EVALUAR_TEST_FINAL = False`. Debe ejecutarse una vez, con el modelo congelado.
- **Estereotipos regionales:** la cabeza `odio_grupal` no reconoce “Todos los costeños son perezosos” (0,42 frente a un umbral de 0,82) y rinde peor que A3 en la batería (F1 0,22 frente a 0,50, con 8 positivos). *Mejora:* anotar ejemplos colombianos (protocolo de la v4, §20) y revisar el umbral.
- **Evaluación fuera de dominio pendiente:** ALIA (YouTube / TikTok) no se replicó en v5; el cuaderno original reportó para A3 F1 ≈ 0,28 y recall ≈ 0,17.
- **Dominio:** piuba es español rioplatense / pandemia; OffendES son comentarios de influencers; la batería es colombiana. No hay datos de entrenamiento colombianos.
- **Contexto no entrenado para las cabezas nuevas:** el contexto solo acompaña a las filas de piuba. El efecto observado en “hp amistoso” requiere un conjunto contextual anotado y contextos de control.
- **Definiciones:** `OFG` de OffendES no se equipara a `HATEFUL` mientras no se verifique su guía de anotación (212 filas de entrenamiento quedan fuera de `odio_grupal`).
- **Etiquetas de la batería:** son propuestas de los autores (`validado = False`); las cifras sobre la batería son descriptivas.
- **Duplicados entre particiones:** hay textos exactos repetidos entre train, dev y test (134 / 167 / 67); no se eliminaron para conservar las mismas filas que la v4.
- **Licencia:** la ficha de OffendES declara licencias contradictorias; verificar antes de redistribuir.
- **Privacidad y contenido sensible:** el texto sale del navegador hacia el servidor que ejecute la app; el corpus y las demos contienen lenguaje ofensivo.
- **Posible solapamiento** del preentrenamiento de RoBERTuito con tuits del grupo UBA: declarado.
- **Mejoras futuras:** banda de “revisar” (`R.elegir_banda`) en lugar de umbral único; umbral por tipo de comunidad; contexto conversacional real; más dialectos y plataformas; modelo en el navegador.

# 8. Equipo
**Integrantes:** Esteban Salazar, Samuel Pico, David Vargas y Tatiana Vivas.

El desarrollo técnico de esta versión (arquitectura multitarea, entrenamiento, evaluación y app de prueba) se implementó con apoyo de un asistente de IA y fue ejecutado y verificado en Colab por el equipo, en el marco del proyecto grupal.

# 9. Registro de revisión con IA
Cada observación de la IA se verificó con las celdas indicadas antes de presentarla como hallazgo. La IA solo recibió texto (no tono ni contexto audiovisual).

| Etapa | Prompt (resumen) | Evidencia | Crítica (síntesis) | Comprobación | Decisión |
|---|---|---|---|---|---|
| Fase 0 | ¿Se puede usar `HATEFUL = 0` como “sin insulto ni vulgaridad”? | Definición de piuba + hallazgo D1 de la v4 | No: los insultos personales aparecen como no-hateful | Auditoría de etiquetas (4.2) | Pérdida enmascarada; cada fuente aporta solo lo que define |
| Datos | ¿Qué fuente respalda vulgaridad e insulto? | Esquema de OffendES (`NO`/`NOE`/`OFP`/`OFG`) | `OFG` no equivale a `HATEFUL` sin verificar | Conteos por cabeza (4.2) | `OFG_COMO_GRUPO = False`; `NOE` como vulgaridad, `OFP` como insulto |
| Evaluación | ¿Cómo evitar que la partición que decide sea la que reporta? | DEV partido por artículo | Separar checkpoint y calibración | 3.6, 4 y 5.3 | `DEV_CKPT` elige; `DEV_EVAL` calibra y compara; TEST reservado |
| Tokenización | Error en Colab: “Sequence to truncate too short” | Traza de la celda de entrenamiento | `only_second` falla si el comentario ya excede `max_len` | Prueba con comentarios de 600 palabras y `max_len` pequeño (5.2) | Acortar el comentario con margen creciente; integrado en el módulo |
| Comparación | ¿La cabeza nueva mejora a A3? | Tabla 5.7 | La diferencia de F1 tiene IC que incluye 0 | *Bootstrap* por artículo | No declarar mejora |
| Validación | ¿El F1 = 1,0 de vulgaridad es fiable? | 37 etiquetas, 7 positivos | No: muestra diminuta | Conteos de 4.2 y 5.4 | Declararlo como limitación; pedir validación mayor |
| Contexto | ¿El efecto del `hp` amistoso es una capacidad entrenada? | 5.6 y sesgo de 4.4 | Las cabezas nuevas no vieron contexto etiquetado | Revisión de `bloque()` y de los *datasets* | Presentarlo como observación a confirmar |

# 10. Conclusiones del proyecto
1. **Problema:** separar en español tres señales —vulgaridad, insulto y odio grupal— con un único modelo que acepte contexto opcional, para que quien modere pueda aplicar reglas distintas con humano en el loop.
2. **Datos:** piuba (`HATEFUL`, 36.587 filas de train, partición por artículo con seed 42) para `odio_grupal` y OffendES (16.697 filas) para `vulgaridad` e `insulto`, combinados con etiquetas parciales y máscara; ninguna etiqueta se inventó.
3. **Arquitectura:** RoBERTuito compartido + tres cabezas independientes, BCE enmascarada, calibración por temperatura y umbrales por cabeza en DEV_EVAL (0,302 · 0,351 · 0,824). Entrenamiento de 778 s en una GPU de Colab; mejor checkpoint en la época 3.
4. **Tarea común con A3:** F1 de `odio_grupal` con contexto + TTA = **0,706** frente a **0,689** de A3 (ΔF1 = +0,016; IC 95 % [−0,012; 0,045]). **Sin diferencia detectable**; no se declara mejora.
5. **Robustez ortográfica:** retención de F1 ≥ 0,79 en `odio_grupal`; `HI J U E P U T A`, `H1JUEPUTA`, `hijuep*ta`, `prostituTa` y `prosti` se detectan como insulto, y el normalizador conserva `hp` sin expandirlo. Las perturbaciones más dañinas son el espaciado, la sustitución de letras y las máscaras.
6. **Contexto:** en la prueba funcional, un contexto amistoso reduce las señales del `hp` (insulto 0,978 → 0,403; vulgaridad 0,941 → 0,300) y uno de discusión las mantiene. Es una **observación a confirmar**, porque las cabezas nuevas no se entrenaron con contexto etiquetado.
7. **Errores conocidos:** estereotipos regionales colombianos (recall 0,125 en la batería, peor que A3), confusión entre ataque personal y odio grupal, jerga discriminatoria fuera de vocabulario y vulgaridad con usos educativos o negados.
8. **EDA:** el corpus es corto (mediana ≈ 11 palabras), con ≈ 15 % de odio, sin diferencias de longitud entre clases y con una lista de groserías que solo alcanza F1 ≈ 0,20 (cifras del EDA original, recalculadas en 3.8–3.13); aparecen además textos repetidos entre particiones.
9. **Pendiente principal:** evaluar en TEST (una sola vez), ampliar la validación de las cabezas nuevas con datos de OffendES, anotar un conjunto contextual y colombiano y verificar la definición de `OFG`; leer la salida de HateCheck-es (celda 5.9) y repetir la evaluación fuera de dominio.
10. **Producto:** modelo guardado como un solo objeto (`Predictor`), paquete de exportación para Gradio / Spaces y laboratorio en Streamlit (túnel temporal en Colab o ejecución local).
11. **Ética de uso:** no sustituye la moderación humana; las tres probabilidades son una ayuda para decidir, y los límites de dominio deben comunicarse a quien use la herramienta.